# Agentic SOAP-Note Generation from Hindi-English Doctor–Patient Conversations

This notebook is the **agentic version** of the straight Audio → SOAP pipeline. Each box on the design
whiteboard is an **agent** (a LangGraph node). The agents share one state, use tools, and decide
for themselves when a step has to be **redone**:

- **VerifiNER** can send the K-Span agent back to re-extract.
- **Reflection** can send the SOAP generator back to revise.

The schema, prompts, knowledge tables and metrics from the straight implementation are reused. The
control flow is now a graph of agents instead of one long function with overrides.

```
                  (text input)
      START ───────────────────────────┐
        │                              ▼
        └──▶ asr_agent ──▶ diarization_agent ──▶ kspan_agent ──▶ verifiner_agent ──┐
                                                  ▲                                │ too many spans rejected
                                                  └────────────────────────────────┤ (feedback, ≤ 1 retry)
                                                                                   ▼
                          ┌──────────────▶ soap_generator_agent (fine-tuned) ──▶ reflection_agent ──┐
                          │                                                                         │
                          └──────────── issues found: hallucination / S,O,A,P check (≤ 2 rounds) ◀──┤
                                                                                                    ▼
                                 addition_agent (codes + AI suggestion) ──▶ evaluation_agent ──▶ report_agent ──▶ END
```

## Whiteboard → agents

| Whiteboard box | Agent (LangGraph node) | What it does |
|---|---|---|
| **Transcript** | `asr_agent`, `diarization_agent` | Whisper large-v3-turbo on Hindi-English audio (8 kHz or 16 kHz), then Doctor/Patient/Attendant labelling. A **word-preservation check** retries if the LLM translated or dropped words. |
| **K-Spans + info** (10 entities, 5 assertion classes) | `kspan_agent` | Extracts spans **exactly as spoken** (Hindi, Devanagari or English) with type (10), existence (5), and info: English normalisation, severity, temporality, experiencer, speaker and cue word. Output format: `(Disease, Existence: Present)`. |
| **K-Spans refinement — VerifiNER** (span, entity) | `verifiner_agent` | ① **span** verification tool (exact, token or fuzzy alignment, with boundary correction) → ② **knowledge** retrieval (SNOMED CT + ICD-10) → ③ LLM verification of **entity type and existence**, sampled 3× → ④ **consistency voting** together with a Hindi-English cue-rule voter. Sends the K-Span agent back if too much is rejected. |
| **Fine-tuning** | `soap_generator_agent` | The **fine-tuned gpt-4o-mini** writes S/O/A/P from the transcript plus the verified K-spans (the checklist). On later rounds it revises using the reflection feedback. |
| **Reflection** ① hallucination check | `reflection_agent` | Tools (plan grounding, speculation stripping, drug check, **assertion conflict**: a denied symptom written as present) plus an LLM critic. |
| **Reflection** ② S, O, A, P check | `reflection_agent` | Sections present, every verified K-span covered, content placed in the right section (vitals → O, orders → P). Loops back to the generator (≤ 2 rounds) and **keeps the best draft**. |
| **Reflection** ③ Addition → standard codes | `addition_agent` | SNOMED CT, ICD-10-CM and CPT codes, plus a clearly labelled **AI Suggestion** that is not scored. |
| (evaluation) | `evaluation_agent` | Independent LLM judge (4 axes), K-span coverage, hallucinations, cross-lingual grounding, and ROUGE/BLEU/BERTScore when a reference is given. |

## How to run on Kaggle
1. **Settings → Accelerator: GPU T4 ×1**, **Internet: on**.
2. **Add-ons → Secrets**: add `OPENAI_API_KEY` and tick it. `HF_TOKEN` is only needed for the local LLaMA fallback.
3. **Run All.** The **DEMO** cell runs a built-in Hindi-English transcript, so no audio is needed. The next cells run on
   one audio file and on a whole dataset folder (`run_batch`), and the last cell opens a Gradio UI.
4. The first run builds the ICD-10 index (~2–4 min on a T4). It is cached in `/kaggle/working` for re-runs.

> The schema (10 entity types, 5 existence classes) lives in the **CONFIG** cell. Every agent reads it
> from there, so changing that one cell changes the whole pipeline.
## VeriSOAP: one base model + one LoRA adapter per agent (CONFIG → `STAGE`)
As in *Agentic AI Architectures for SOAP Note Generation* (Hanken, BioNLP 2026), every agent runs on **one shared
base model** (Llama 3.2 3B Instruct, 4-bit NF4) and hot-swaps its **own LoRA adapter** for its calls:
`diarization`, `kspan`, `verifier`, `generator`, `critic`, `reviser`. The judge stays on OpenAI (gpt-4.1) so the
scores do not come from the writers' own model.

| Step | Where | Setting | What happens |
|---|---|---|---|
| 1 | this notebook | `STAGE = "export"` | OpenAI teacher + BeTraC **train** split → `adapter_data/*.jsonl` (nothing else runs) |
| 2 | `lora_adapter_training.ipynb` | attach step 1's output | QLoRA trains the 6 adapters → `soap_adapters/<agent>/` |
| 3 | this notebook | `STAGE = "lora"` (default) | attach step 2's output; all agents run on the base model + their adapters |
| (baseline) | this notebook | `STAGE = "openai"` | API-only run (fine-tuned gpt-4o-mini + gpt-4.1-mini), for comparison |

In `STAGE = "lora"` the notebook **stops with an error** if any adapter is missing, so a run can never fall back to
the plain base model without you noticing.

In [ ]:
# ============================ 1. INSTALL ============================
# Kaggle already ships torch, transformers, librosa, pandas and scikit-learn.
import subprocess, sys
NEED_MINISTRAL = True   # model sweep with Ministral 3 needs transformers >= 5 + mistral-common (False = keep Kaggle's)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "langgraph", "openai>=1.40", "sentence-transformers", "simple-icd-10-cm", "gradio", "tabulate"],
               check=False)
if NEED_MINISTRAL:
    from importlib.metadata import version
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mistral-common>=1.8.6", "bitsandbytes", "accelerate"],
                   check=False)
    if int(version("transformers").split(".")[0]) < 5:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=5"], check=False)
    print("transformers", version("transformers"))
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)

In [ ]:
# ============================ 2. CONFIG (edit here) ============================
import os, re, sys, json, time, math, textwrap, warnings, operator, difflib, subprocess
from collections import Counter
from datetime import datetime
warnings.filterwarnings("ignore")

def _secret(name):
    """Env var -> Kaggle secret -> ''. Keys are never written into the notebook."""
    val = os.environ.get(name, "")
    if val:
        return val
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name) or ""
    except Exception:
        return ""

OPENAI_API_KEY = _secret("OPENAI_API_KEY")
HF_TOKEN       = _secret("HF_TOKEN")          # only for the local LLaMA fallback

# ---- which model plays which role ----
OPENAI_FT_MODEL    = "ft:gpt-4o-mini-2024-07-18:personal:soap-noise-aug:DypalTtg"  # SOAP generator (fine-tuned)
OPENAI_AGENT_MODEL = "gpt-4.1-mini"   # K-Span, VerifiNER, Diarization, Reflection, Addition agents
OPENAI_JUDGE_MODEL = "gpt-4.1"        # evaluation judge: must differ from BOTH writers (fine-tuned draft and
                                      # gpt-4.1-mini revisions), otherwise it can favour its own writing
LOCAL_LLM_ID       = "meta-llama/Meta-Llama-3.1-8B-Instruct"   # used only when there is no OpenAI key

# ---- STAGE: which step of the VeriSOAP workflow this run is ----
#   "export": step 1 - the OpenAI teacher builds the LoRA training data from the BeTraC train split (only that runs)
#   "lora"  : step 3 - VeriSOAP: ONE local base model shared by all agents, each agent hot-swaps its own LoRA adapter
#             (adapters from lora_adapter_training.ipynb). The judge stays on OpenAI when a key exists.
#   "openai": API-only baseline (fine-tuned gpt-4o-mini drafts, gpt-4.1-mini agents), for comparison
#   "sweep" : run OUR code as single-agent and multi-agent on each model in SWEEP_MODELS (only that runs)
STAGE            = "lora"
LLM_BACKEND      = "openai" if STAGE in ("export", "openai", "sweep") else "lora"
LORA_BASE_MODEL  = "meta-llama/Llama-3.2-3B-Instruct"   # must equal BASE_MODEL in lora_adapter_training.ipynb
LORA_ADAPTER_DIR = None      # folder with one sub-folder per adapter; None = search /kaggle/input for adapter_config.json
LORA_4BIT        = True      # 4-bit NF4 base model (fits a Kaggle T4 with Whisper and the embedders)
LORA_REQUIRE_ALL = True      # stop if any adapter is missing (otherwise those agents would silently use the plain base)
LORA_ADAPTERS    = {"diarization": "diarization", "kspan": "kspan", "verifier": "verifier",
                    "generator": "generator", "reviser": "reviser", "critic": "critic"}   # agent role -> adapter name


WHISPER_MODEL_ID = "openai/whisper-large-v3-turbo"
ASR_LANGUAGE     = None   # None = auto-detect; "hi" = force Hindi (Devanagari output); "en" = force English
EMBEDDER_ID      = "pritamdeka/S-PubMedBert-MS-MARCO"       # medical English embeddings (coding tools)
CROSS_ENCODER_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"   # re-ranker (coding tools)
ML_EMBEDDER_ID   = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"  # Hindi-English grounding

# ---- K-span schema: 10 entity types + 5 assertion (existence) classes ----
# EDIT these if the final list differs; every agent, prompt and check reads them from here.
ENTITY_TYPES = {
    "Symptom":   "a complaint or sign the patient has or denies (fever, bukhar, sir dard, khansi, chest pain)",
    "Disease":   "a diagnosis, condition or disorder (diabetes, sugar, viral fever, hypertension)",
    "Drug":      "a medicine name, brand or class (Paracetamol, Crocin, antibiotic, insulin)",
    "Dosage":    "dose, strength, frequency, route or duration of a drug (650 mg, din mein teen baar, 5 din tak)",
    "Procedure": "a surgery, therapy, referral or intervention (appendectomy, physiotherapy, referral to cardiology)",
    "Test":      "a lab test, imaging or investigation (CBC, X-ray, dengue NS1, ECG, sugar test)",
    "Vital":     "a vital-sign measurement with its value (BP 118 by 76, pulse 96, temperature 101.4)",
    "Anatomy":   "a body part or site that locates a finding (left knee, pet, seena, gala)",
    "Allergy":   "an allergy or adverse reaction and its trigger (penicillin se allergy)",
    "Lifestyle": "social / lifestyle history (smoking, alcohol, diet, occupation, exercise)",
}
EXISTENCE_CLASSES = {
    "Present":       "stated as present / confirmed / currently true for the experiencer",
    "Absent":        "explicitly denied or negated (nahi, nahin, no, denies, without)",
    "Uncertain":     "possible, suspected or doubtful (shayad, ho sakta hai, lagta hai, maybe, rule out)",
    "Hypothetical":  "conditional or future (agar ... toh, if it happens, in case)",
    "Not specified": "mentioned but whether it exists is never stated (e.g. a question left unanswered)",
}
SEVERITY_VALUES    = ["mild", "moderate", "severe", "unspecified"]
TEMPORALITY_VALUES = ["current", "historical", "unspecified"]
EXPERIENCER_VALUES = ["patient", "family", "other"]
COVERAGE_TYPES     = {"Symptom", "Disease", "Drug", "Vital", "Test", "Procedure", "Allergy"}  # must reach the note

# ---- agent behaviour ----
KSPAN_MAX_ATTEMPTS    = 2      # VerifiNER may send the K-Span agent back once
KSPAN_CHUNK_CHARS     = 3000   # long transcripts are extracted chunk by chunk
VERIFY_VOTES          = 3      # VerifiNER consistency voting: verifier samples per span (1 = cheapest)
VERIFY_BATCH          = 20     # spans verified per LLM call
VERIFY_MIN_SURVIVAL   = 0.30   # below this fraction of surviving spans -> K-Span agent re-extracts
FUZZY_SPAN_THRESHOLD  = 0.85   # span alignment similarity accepted for ASR spelling variants
MAX_REFLECTION_ROUNDS = 2      # Reflection -> Generator revision loops (bounded)
REVISION_ROLE = "reviser"      # who writes revision rounds: "reviser" (gpt-4.1-mini / reviser adapter) or "generator"
REFLECTION_ACCEPT_SCORE = 2   # accept a draft with no hallucination and <= this many minor (missing/misplaced) issues
TRUE_BERTSCORE = True          # real BERTScore (bert-score, roberta-large, baseline-rescaled) vs reference SOAP
ICD10_COSINE_THRESHOLD = 0.65
RERANK_THRESHOLD       = 0.8
SOAP_MAX_TOKENS        = 3000

PIPELINE_VERSION = "v5"        # bump when the pipeline changes; result files are named with it
AUDIO_EXTS =(".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac", ".opus", ".webm")
OUTPUT_DIR = "/kaggle/working/agentic_soap_outputs" if os.path.isdir("/kaggle/working") else "./agentic_soap_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
VERBOSE = True

# ---- what "Run All" does (each step is skipped when False) ----
RUN_DEMO       = True    # built-in Hindi-English demo transcript
RUN_BETRAC     = True    # BeTraC dev benchmark: agentic notes + scores
RUN_COMPARISON = True    # friend's straight pipeline vs agentic, same dialogues (needs the friend's notebook as input)
RUN_LATENCY    = False   # latency benchmark (straight / agentic / agentic-fast)
RUN_EXPORT     = False   # build LoRA training data from the BeTraC train split (OpenAI teacher)
RUN_GRADIO     = True    # launch the Gradio UI at the end
RUN_SWEEP      = False   # model sweep (set by STAGE = "sweep")

# ---- model sweep (STAGE = "sweep"): every agent role runs on one of these models; the judge stays on OPENAI_JUDGE_MODEL ----
SWEEP_MODELS = {
    "Llama-3.2-3B":   ("hf",           "meta-llama/Llama-3.2-3B-Instruct"),
    "Llama-3.1-8B":   ("hf",           "meta-llama/Llama-3.1-8B-Instruct"),
    "Ministral-3-3B": ("hf-ministral", "mistralai/Ministral-3-3B-Instruct-2512"),
    "Ministral-3-8B": ("hf-ministral", "mistralai/Ministral-3-8B-Instruct-2512"),
    "GPT-5-mini":     ("openai",       "gpt-5-mini"),
}
SWEEP_RUN   = list(SWEEP_MODELS)        # models to run in THIS session (results of earlier sessions are kept)
SWEEP_ARCHS = ["single", "multi"]       # single = one call writes the K-SOAP note; multi = the full VeriSOAP graph
SWEEP_LIMIT = 10                        # same first-N BeTraC dev dialogues as the Nishat comparison
SWEEP_INPUT = "diarized"                # "diarized": every model gets the reference run's speaker-labelled transcript
                                        #   (fast, identical input); "asr": raw Whisper text, diarization runs per model (slow)
SWEEP_VERIFY_VOTES = 3                  # VerifiNER votes in the sweep (3 = same as the VeriSOAP row; 1 = ~2x faster)
if STAGE == "export":    # step 1 only builds the training data; every other step is switched off
    RUN_DEMO = RUN_BETRAC = RUN_COMPARISON = RUN_LATENCY = RUN_GRADIO = False
    RUN_EXPORT = True
if STAGE == "sweep":     # only the model sweep runs
    RUN_DEMO = RUN_BETRAC = RUN_COMPARISON = RUN_LATENCY = RUN_EXPORT = RUN_GRADIO = False
    RUN_SWEEP = True
assert STAGE in ("export", "lora", "openai", "sweep"), f"unknown STAGE {STAGE!r}"

# ---- Kaggle: "Save Version -> Save & Run All" runs the notebook in the background (run type "Batch") ----
KAGGLE_BATCH = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "") == "Batch"
if KAGGLE_BATCH and RUN_GRADIO:
    RUN_GRADIO = False   # nobody can click a UI in a background run
    print("[config] Kaggle background run (Save & Run All): Gradio UI switched off")
if LLM_BACKEND == "lora" and LORA_BASE_MODEL.startswith("meta-llama/") and not HF_TOKEN:
    raise RuntimeError("STAGE = 'lora' downloads the gated Llama base model: add the HF_TOKEN secret "
                       "(Add-ons -> Secrets, tick it for this notebook) after accepting the Llama 3.2 licence on Hugging Face.")

print(f"[config] STAGE = {STAGE!r} -> LLM backend {LLM_BACKEND!r}")
print(f"[config] OpenAI key: {'found' if OPENAI_API_KEY else 'NOT found'}"
      + (" -> local LLaMA fallback" if not OPENAI_API_KEY and STAGE == "openai" else "")
      + (" -> WARNING: the judge will be the writers' own base model" if not OPENAI_API_KEY and LLM_BACKEND == "lora" else ""))
if STAGE in ("export", "sweep") and not OPENAI_API_KEY:
    raise RuntimeError(f"STAGE = {STAGE!r} needs OPENAI_API_KEY (teacher / judge). Add it under Add-ons -> Secrets.")
print(f"[config] schema: {len(ENTITY_TYPES)} entity types, {len(EXISTENCE_CLASSES)} existence classes")
print(f"[config] outputs -> {OUTPUT_DIR}")

In [ ]:
# ============================ 3. LLM ROUTER (one door to every language model) ============================
# Robust JSON parser carried over from the straight implementation:
def _parse_json(raw, expect="list"):
    for fence in ("```json", "```"):
        if fence in raw:
            raw = raw.split(fence)[1].split("```")[0].strip()
            break
    if expect == "dict":
        search_order = [("{", "}"), ("[", "]")]
    else:
        search_order = [("[", "]"), ("{", "}")]
    for opener, closer in search_order:
        start = raw.find(opener)
        if start == -1:
            continue
        depth, end = 0, -1
        for i, ch in enumerate(raw[start:], start):
            if ch == opener:
                depth += 1
            elif ch == closer:
                depth -= 1
            if depth == 0:
                end = i + 1
                break
        if end != -1:
            raw = raw[start:end]
            break
        if opener == "[" and start != -1:
            last_brace = raw.rfind("}")
            if last_brace > start:
                truncated = raw[start:last_brace + 1] + "]"
                try:
                    test = json.loads(truncated, strict=False)
                    if isinstance(test, list):
                        raw = truncated
                        break
                except Exception:
                    last_comma = truncated.rfind(",")
                    if last_comma > start:
                        truncated2 = truncated[:last_comma] + "]"
                        try:
                            test2 = json.loads(truncated2, strict=False)
                            if isinstance(test2, list):
                                raw = truncated2
                                break
                        except Exception:
                            pass
    raw = re.sub(r",\s*([}\]])", r"\1", raw)
    _ESC = {"\n": "\\n", "\r": "\\r", "\t": "\\t"}
    result, in_str, esc_next = [], False, False
    for ch in raw:
        if esc_next:
            result.append(ch)
            esc_next = False
            continue
        if ch == "\\":
            result.append(ch)
            esc_next = in_str
            continue
        if ch == '"':
            in_str = not in_str
        if in_str and ch in _ESC:
            result.append(_ESC[ch])
            continue
        result.append(ch)
    cleaned = "".join(result)
    try:
        parsed = json.loads(cleaned, strict=False)
    except Exception:
        for m in re.finditer(r'[\[{]', cleaned):
            try:
                parsed = json.loads(cleaned[m.start():], strict=False)
                if expect == "list" and isinstance(parsed, list):
                    return parsed
                if expect == "dict" and isinstance(parsed, dict):
                    return parsed
            except Exception:
                continue
        return [] if expect == "list" else {}
    if expect == "list":
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, dict):
            for v in parsed.values():
                if isinstance(v, list) and v and isinstance(v[0], dict):
                    return v
        return []
    else:
        if isinstance(parsed, dict):
            return parsed
        return {}


AGENT_ROLES = ["diarization", "kspan", "verifier", "generator", "reviser", "critic", "agent", "judge"]


class LLMRouter:
    """Agents ask for a ROLE, never a model name:
         diarization / kspan / verifier / critic / reviser / agent -> extraction, checking and revision work
         generator -> writes the first SOAP draft
         judge     -> independent model used only for scoring (never one of the writers)
    Backends:
         openai : generator = fine-tuned gpt-4o-mini, every other role = gpt-4.1-mini, judge = gpt-4.1
         lora   : ONE local base model; each role hot-swaps its own LoRA adapter (roles without one use the
                  plain base model); the judge stays on OpenAI when a key exists
         local  : no OpenAI key -> one local model for every role, no adapters"""

    def __init__(self):
        self.calls = Counter()
        self.client, self._model, self._tok, self._eos, self._peft = None, None, None, None, False
        self.adapter_for, self.call_hook = {}, None          # call_hook(dict) is used by the latency benchmark
        if OPENAI_API_KEY:
            from openai import OpenAI
            self.client = OpenAI(api_key=OPENAI_API_KEY)
        if LLM_BACKEND == "lora":
            self.backend = "lora"
            self._load_hf(LORA_BASE_MODEL, adapters=True)
            judge = self._check_openai(OPENAI_JUDGE_MODEL) if self.client else None
            self.models = {r: LORA_BASE_MODEL + (f" + {self.adapter_for[r]}" if r in self.adapter_for else " (no adapter)")
                           for r in AGENT_ROLES if r != "judge"}
            self.models["judge"] = judge or LORA_BASE_MODEL + " (no adapter; WARNING: judge = writer's base model)"
            self._judge_api = bool(judge)
        elif self.client:
            self.backend = "openai"
            gen = self._check_openai(OPENAI_FT_MODEL) if OPENAI_FT_MODEL else None
            if OPENAI_FT_MODEL and not gen:
                print("[llm] -> the SOAP generator falls back to the base model")
            judge = self._check_openai(OPENAI_JUDGE_MODEL)
            if not judge:
                print("[llm] WARNING: judge = reviser model, so judge scores may favour revised notes")
            self.models = {r: OPENAI_AGENT_MODEL for r in AGENT_ROLES}
            self.models.update(generator=gen or OPENAI_AGENT_MODEL, judge=judge or OPENAI_AGENT_MODEL)
        else:
            self.backend = "local"
            self._load_hf(LOCAL_LLM_ID, adapters=False)
            self.models = {r: LOCAL_LLM_ID for r in AGENT_ROLES}
        for role in AGENT_ROLES:
            print(f"[llm] {role:<11} -> {self.models[role]}")

    def _check_openai(self, model):
        try:
            self.client.chat.completions.create(model=model, messages=[{"role": "user", "content": "ping"}], max_tokens=1)
            return model
        except Exception as e:
            print(f"[llm] {model} not available on this account ({str(e)[:70]})")
            return None

    def _find_adapters(self):
        """{adapter name: folder}: LORA_ADAPTER_DIR/<name>, or any folder under /kaggle/input with adapter_config.json."""
        roots = [LORA_ADAPTER_DIR] if LORA_ADAPTER_DIR else ["/kaggle/input"]
        found = {}
        for root in roots:
            for dirpath, _, files in os.walk(root or "."):
                if "adapter_config.json" in files:
                    found.setdefault(os.path.basename(dirpath.rstrip("/")), dirpath)
        return found

    def _load_hf(self, model_id, adapters):
        import torch
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate", "peft"])
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        if HF_TOKEN:
            from huggingface_hub import login
            login(token=HF_TOKEN)
        else:
            print("[llm] no HF_TOKEN secret: a gated Llama download will fail unless the image is logged in")
        kw = dict(device_map="auto", torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
        if LORA_4BIT and torch.cuda.is_available():
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                           bnb_4bit_compute_dtype=torch.float16,
                                                           bnb_4bit_use_double_quant=True)
        model = AutoModelForCausalLM.from_pretrained(model_id, **kw)
        tok = AutoTokenizer.from_pretrained(model_id)
        tok.pad_token = tok.pad_token or tok.eos_token
        # end-of-turn ids: tokenizer + generation_config + known chat end tokens (works for Llama, Mistral, Qwen, ...)
        eos = {tok.eos_token_id}
        g_eos = getattr(getattr(model, "generation_config", None), "eos_token_id", None)
        eos |= set(g_eos if isinstance(g_eos, (list, tuple)) else [g_eos])
        for t in ("<|eot_id|>", "<|im_end|>", "<|end|>"):
            i = tok.convert_tokens_to_ids(t)
            if isinstance(i, int) and i >= 0 and i != tok.unk_token_id:
                eos.add(i)
        self._eos = sorted(t for t in eos if isinstance(t, int) and t >= 0)
        if adapters:
            from peft import PeftModel
            avail = self._find_adapters()
            wanted = {role: name for role, name in LORA_ADAPTERS.items() if name in avail}
            missing = sorted(set(LORA_ADAPTERS.values()) - set(avail))
            if missing:
                msg = (f"[llm] LoRA adapters not found: {missing} (searched {LORA_ADAPTER_DIR or '/kaggle/input'}). "
                       "Attach the output of lora_adapter_training.ipynb (Add Input -> Notebook output) or set "
                       "LORA_ADAPTER_DIR.")
                if LORA_REQUIRE_ALL:
                    raise RuntimeError(msg + " (set LORA_REQUIRE_ALL = False to run those agents on the plain base)")
                print(msg + " Those roles use the plain base model.")
            for name in dict.fromkeys(wanted.values()):          # adapters must be trained on THIS base model
                try:
                    trained_on = json.load(open(os.path.join(avail[name], "adapter_config.json"))).get("base_model_name_or_path")
                except Exception:
                    trained_on = None
                if trained_on and trained_on.rstrip("/").split("/")[-1] != model_id.rstrip("/").split("/")[-1]:
                    raise RuntimeError(f"[llm] adapter '{name}' was trained on {trained_on}, but the base model is "
                                       f"{model_id}. Set LORA_BASE_MODEL to the training notebook's BASE_MODEL.")
            for name in dict.fromkeys(wanted.values()):          # load each adapter once, keep order
                if not self._peft:
                    model = PeftModel.from_pretrained(model, avail[name], adapter_name=name)
                    self._peft = True
                else:
                    model.load_adapter(avail[name], adapter_name=name)
                print(f"[llm] loaded LoRA adapter '{name}' from {avail[name]}")
            self.adapter_for = wanted
            print(f"[llm] one base model ({model_id}) + {len(set(wanted.values()))} LoRA adapters, hot-swapped per agent")
        model.eval()
        self._model, self._tok = model, tok

    def _hf_chat(self, messages, role, max_tokens, temperature):
        import torch, contextlib
        tok, model = self._tok, self._model
        enc = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
        enc = {k: v.to(model.device) for k, v in enc.items()}
        gen = dict(max_new_tokens=max_tokens, eos_token_id=self._eos, pad_token_id=tok.pad_token_id)
        if temperature and temperature > 0:
            gen.update(do_sample=True, temperature=temperature, top_p=0.9)
        else:
            gen.update(do_sample=False)
        adapter = self.adapter_for.get(role)
        if self._peft and adapter:
            model.set_adapter(adapter)                           # hot-swap: same base, this agent's adapter
            ctx = contextlib.nullcontext()
        elif self._peft:
            ctx = model.disable_adapter()                        # role without adapter -> plain base model
        else:
            ctx = contextlib.nullcontext()
        t0 = time.perf_counter()
        with torch.no_grad(), ctx:
            out = model.generate(**enc, **gen)
        new = out[0, enc["input_ids"].shape[1]:]
        text = tok.decode(new, skip_special_tokens=True)
        if self.call_hook:
            self.call_hook({"role": role, "model": self.models.get(role), "latency_s": round(time.perf_counter() - t0, 3),
                            "prompt_tokens": int(enc["input_ids"].shape[1]), "completion_tokens": int(new.shape[0])})
        return text

    def chat(self, messages, role="agent", max_tokens=1500, temperature=0.2, json_mode=False):
        self.calls[role] += 1
        use_api = self.backend == "openai" or (self.backend == "lora" and role == "judge" and self._judge_api)
        if not use_api:
            return self._hf_chat(messages, role, max_tokens, temperature)
        kw = dict(model=self.models.get(role, self.models["agent"]), messages=messages,
                  max_tokens=max_tokens, temperature=temperature)
        if json_mode:
            kw["response_format"] = {"type": "json_object"}
        last = None
        for attempt in range(4):
            try:
                return self.client.chat.completions.create(**kw).choices[0].message.content or ""
            except Exception as e:
                last = e
                if "response_format" in kw and "response_format" in str(e):
                    kw.pop("response_format")          # model without JSON mode -> plain call
                    continue
                time.sleep(2 ** attempt)               # rate limit / transient error
        raise RuntimeError(f"LLM call failed ({role}): {last}")

    def json(self, messages, role="agent", max_tokens=1500, temperature=0.0, retries=1):
        """Ask for a JSON object; if the reply cannot be parsed, show it back and ask again."""
        msgs = list(messages)
        for _ in range(retries + 1):
            raw = self.chat(msgs, role, max_tokens, temperature, json_mode=True)
            d = _parse_json(raw, expect="dict")
            if d:
                return d
            msgs = msgs + [{"role": "assistant", "content": raw[:1500]},
                           {"role": "user", "content": "That was not valid JSON. Reply again with ONLY the JSON object."}]
        return {}


LLM = LLMRouter()

In [ ]:
# ============================ 4. PERCEPTION + KNOWLEDGE MODELS (agent tools, loaded once) ============================
import torch, numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import simple_icd_10_cm as icd
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline as hf_pipeline

t_boot = time.time()
device      = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print("[1/4] Whisper ASR ...")
_asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    WHISPER_MODEL_ID, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True).to(device)
_asr_proc = AutoProcessor.from_pretrained(WHISPER_MODEL_ID)
asr_pipe = hf_pipeline("automatic-speech-recognition", model=_asr_model,
                       tokenizer=_asr_proc.tokenizer, feature_extractor=_asr_proc.feature_extractor,
                       torch_dtype=torch_dtype, device=device, chunk_length_s=30)

print("[2/4] Embedders + re-ranker ...")
embedder      = SentenceTransformer(EMBEDDER_ID, device=device)      # medical English (coding)
ml_embedder   = SentenceTransformer(ML_EMBEDDER_ID, device=device)   # multilingual (Hindi-English grounding)
cross_encoder = CrossEncoder(CROSS_ENCODER_ID, device=device)

print("[3/4] ICD-10-CM index ...")
icd10_db = pd.DataFrame([{"code": c, "description": icd.get_description(c)}
                         for c in icd.get_all_codes() if icd.is_leaf(c)])
_icd_cache = os.path.join(OUTPUT_DIR, "icd10_embeddings.pt")
icd10_embeddings = None
if os.path.exists(_icd_cache):
    icd10_embeddings = torch.load(_icd_cache, map_location=device)
    if icd10_embeddings.shape[0] != len(icd10_db):
        icd10_embeddings = None                                   # stale cache -> rebuild
if icd10_embeddings is None:
    icd10_embeddings = embedder.encode(icd10_db["description"].tolist(), convert_to_tensor=True,
                                       show_progress_bar=True, batch_size=256)
    torch.save(icd10_embeddings, _icd_cache)
print(f"[4/4] models ready in {time.time() - t_boot:.0f}s (device={device}, {len(icd10_db)} ICD-10 codes)")

In [ ]:
# ============================ 5. CLINICAL KNOWLEDGE BASE (VerifiNER knowledge + coding tools) ============================
# SNOMED / normalisation / CPT / ICD chapter routing tables are carried over unchanged from the
# straight implementation (including its later SNOMED fixes). Hindi-English resources are new.

SNOMED_TABLE = [
    {"id":"422587007","t":"Nausea","c":"R11.0","kw":["nausea","nauseated","nauseous","feeling sick"]},
    {"id":"422400008","t":"Vomiting","c":"R11.10","kw":["vomiting","vomit","throw up","throwing up","emesis"]},
    {"id":"73879007","t":"Nausea and vomiting","c":"R11.2","kw":["nausea and vomiting","nausea with vomiting"]},
    {"id":"62315008","t":"Diarrhea","c":"R19.7","kw":["diarrhea","diarrhoea","loose stools","watery stools","diadea"]},
    {"id":"14760008","t":"Constipation","c":"K59.00","kw":["constipation","constipated"]},
    {"id":"21522001","t":"Abdominal pain","c":"R10.9","kw":["abdominal pain","belly pain","stomach pain","stomachache","abdominal tenderness"]},
    {"id":"73063007","t":"Abdominal cramps","c":"R10.84","kw":["cramps","crampy","cramping","abdominal cramps","stomach cramps"]},
    {"id":"698065002","t":"Heartburn","c":"R12","kw":["heartburn","acid reflux","pyrosis"]},
    {"id":"235365009","t":"GERD","c":"K21.0","kw":["gerd","gastroesophageal reflux","reflux disease"]},
    {"id":"235856003","t":"Gastroenteritis","c":"K52.9","kw":["gastroenteritis","stomach flu","stomach bug","food poisoning"]},
    {"id":"4556007","t":"Gastritis","c":"K29.70","kw":["gastritis"]},
    {"id":"64226004","t":"Dysphagia","c":"R13.10","kw":["dysphagia","difficulty swallowing","trouble swallowing"]},
    {"id":"271681002","t":"Loss of appetite","c":"R63.0","kw":["loss of appetite","not eating","not hungry","decreased appetite","anorexia"]},
    {"id":"249497008","t":"Bloating","c":"R14.0","kw":["bloating","bloated","distended","abdominal distension"]},
    {"id":"162607003","t":"Blood in stool","c":"K92.1","kw":["blood in stool","blood in stools","blood in your stools","rectal bleeding","bloody stool","melena","hematochezia"]},
    {"id":"422823003","t":"Hematemesis","c":"K92.0","kw":["blood in vomit","hematemesis","vomiting blood"]},
    {"id":"197456007","t":"IBS","c":"K58.9","kw":["irritable bowel","ibs"]},
    {"id":"396332003","t":"Appendicitis","c":"K35.80","kw":["appendicitis"]},
    {"id":"34000006","t":"Crohn disease","c":"K50.90","kw":["crohn","crohn's"]},
    {"id":"64766004","t":"Ulcerative colitis","c":"K51.90","kw":["ulcerative colitis","colitis"]},
    {"id":"44054006","t":"Bowel obstruction","c":"K56.60","kw":["bowel obstruction","intestinal obstruction"]},
    {"id":"60728008","t":"Jaundice","c":"R17","kw":["jaundice","yellow skin","icterus","yellow eyes"]},

    {"id":"29857009","t":"Chest pain","c":"R07.9","kw":["chest pain","chest tightness","tight feeling in my chest","pressure on my chest","chest pressure"]},
    {"id":"80313002","t":"Palpitations","c":"R00.2","kw":["palpitations","heart racing","racing of the heart","heart pounding","beating faster","heart fluttering","thumping"]},
    {"id":"3424008","t":"Tachycardia","c":"R00.0","kw":["tachycardia","fast heart rate","rapid heart rate"]},
    {"id":"48867003","t":"Bradycardia","c":"R00.1","kw":["bradycardia","slow heart rate"]},
    {"id":"42343007","t":"Heart failure","c":"I50.9","kw":["heart failure","chf","congestive heart failure","fluid overload"]},
    {"id":"22298006","t":"Myocardial infarction","c":"I21.9","kw":["heart attack","myocardial infarction","mi","stemi","nstemi"]},
    {"id":"194828000","t":"Angina","c":"I20.9","kw":["angina","angina pectoris"]},
    {"id":"38341003","t":"Hypertension","c":"I10","kw":["high blood pressure","hypertension","elevated bp","elevated blood pressure","blood pressure high"]},
    {"id":"45007003","t":"Hypotension","c":"I95.9","kw":["low blood pressure","hypotension"]},
    {"id":"49436004","t":"Atrial fibrillation","c":"I48.91","kw":["atrial fibrillation","afib","a-fib","irregular heartbeat"]},
    {"id":"233604007","t":"Pericarditis","c":"I30.9","kw":["pericarditis","pericardial"]},
    {"id":"253273006","t":"Heart murmur","c":"R01.1","kw":["heart murmur","murmur"]},
    {"id":"59282003","t":"Pulmonary edema","c":"J81.0","kw":["fluid in lungs","pulmonary edema","fluid in the lungs"]},

    {"id":"49727002","t":"Cough","c":"R05.9","kw":["cough","coughing","persistent cough","dry cough","productive cough"]},
    {"id":"267036007","t":"Dyspnea","c":"R06.00","kw":["shortness of breath","difficulty breathing","trouble breathing","short of breath","breathless","dyspnea","winded","gasping for air"]},
    {"id":"56018004","t":"Wheezing","c":"R06.2","kw":["wheeze","wheezing"]},
    {"id":"48409008","t":"Orthopnea","c":"R06.01","kw":["orthopnea","can't breathe lying flat","breathe lying flat","prop up with pillows","pillows at night"]},
    {"id":"70572006","t":"Stridor","c":"R06.1","kw":["stridor","noisy breathing"]},
    {"id":"195967001","t":"Asthma","c":"J45.909","kw":["asthma","asthmatic","reactive airway"]},
    {"id":"13645005","t":"COPD","c":"J44.1","kw":["copd","chronic obstructive","emphysema"]},
    {"id":"233604007","t":"Pneumonia","c":"J18.9","kw":["pneumonia"]},
    {"id":"233703007","t":"Pulmonary embolism","c":"I26.99","kw":["pulmonary embolism","pe","blood clot in lung"]},
    {"id":"18165001","t":"Pleural effusion","c":"J90","kw":["pleural effusion","fluid around lungs"]},
    {"id":"65124004","t":"Crackles","c":"R09.89","kw":["crackles","rales","crepitations","bibasilar crackles"]},
    {"id":"267038008","t":"Edema","c":"R60.0","kw":["swelling","edema","oedema","swollen","pitting edema","ankle swelling","swelling in my ankles","swelling in ankles","leg swelling"]},
    {"id":"87433001","t":"Hemoptysis","c":"R04.2","kw":["coughing blood","hemoptysis","blood in sputum","bringing up blood"]},
    {"id":"68235000","t":"Nasal congestion","c":"R09.81","kw":["congestion","stuffy nose","blocked nose","runny nose","nasal congestion","rhinorrhea"]},
    {"id":"162397003","t":"Sore throat","c":"R07.0","kw":["sore throat","throat pain","pharyngitis","scratchy throat"]},
    {"id":"301354004","t":"Mucus production","c":"R09.3","kw":["mucus","phlegm","sputum","clear mucus"]},

    {"id":"25064002","t":"Headache","c":"R51.9","kw":["headache","headaches","head pain","cephalgia"]},
    {"id":"37796009","t":"Migraine","c":"G43.909","kw":["migraine","migraines"]},
    {"id":"404640003","t":"Dizziness","c":"R42","kw":["dizziness","dizzy","lightheaded","lightheadedness","vertigo","room spinning"]},
    {"id":"271594007","t":"Syncope","c":"R55","kw":["syncope","fainted","fainting","faint","passed out","loss of consciousness","lost consciousness","blacked out"]},
    {"id":"62106007","t":"Numbness","c":"R20.0","kw":["numbness","numb","tingling","paresthesia","pins and needles"]},
    {"id":"26079004","t":"Tremor","c":"R25.1","kw":["tremor","tremors","shaking","shaky"]},
    {"id":"91175000","t":"Seizure","c":"R56.9","kw":["seizure","seizures","convulsion","convulsions","fit","fits"]},
    {"id":"230690007","t":"Stroke","c":"I63.9","kw":["stroke","cva","cerebrovascular"]},
    {"id":"52448006","t":"Dementia","c":"F03.90","kw":["dementia","alzheimer","memory loss","cognitive decline"]},
    {"id":"23056005","t":"Sciatica","c":"M54.30","kw":["sciatica","sciatic","shooting pain down leg"]},
    {"id":"128196005","t":"Confusion","c":"R41.0","kw":["confusion","confused","disoriented","altered mental status"]},
    {"id":"40917007","t":"Visual disturbance","c":"H53.9","kw":["blurry vision","blurred vision","double vision","diplopia","visual changes"]},

    {"id":"161891005","t":"Back pain","c":"M54.9","kw":["back pain","backache","lower back pain","back is killing","lumbago"]},
    {"id":"57676002","t":"Joint pain","c":"M25.50","kw":["joint pain","arthralgia","knee pain","hip pain","shoulder pain","elbow pain","wrist pain"]},
    {"id":"68962001","t":"Muscle pain","c":"M79.10","kw":["muscle aches","muscle pain","myalgia","body aches","sore muscles"]},
    {"id":"55300003","t":"Muscle cramp","c":"R25.2","kw":["muscle cramp","muscle spasm","charley horse"]},
    {"id":"263171000","t":"Stiffness","c":"M25.60","kw":["stiffness","stiff","joint stiffness"]},
    {"id":"125605004","t":"Fracture","c":"T14.8XXA","kw":["fracture","broken bone","broken"]},
    {"id":"44465007","t":"Sprain","c":"S93.409A","kw":["sprain","sprained","rolled ankle","twisted","ankle injury","ankle sprain"]},
    {"id":"299308007","t":"Neck pain","c":"M54.2","kw":["neck pain","stiff neck","neck stiffness"]},
    {"id":"239872002","t":"Limited ROM","c":"M25.60","kw":["limited range of motion","can't move","difficulty moving","limited mobility"]},

    {"id":"73211009","t":"Diabetes mellitus","c":"E11.9","kw":["diabetes","diabetic","type 2 diabetes","type 1 diabetes","sugar","blood sugar"]},
    {"id":"40930008","t":"Hypothyroidism","c":"E03.9","kw":["hypothyroidism","underactive thyroid","low thyroid"]},
    {"id":"34095006","t":"Dehydration","c":"E86.0","kw":["dehydration","dehydrated"]},
    {"id":"190372001","t":"Polydipsia","c":"R63.1","kw":["polydipsia","excessive thirst","very thirsty","thirsty","increased thirst"]},
    {"id":"55822004","t":"Hyperlipidemia","c":"E78.5","kw":["high cholesterol","cholesterol","cholesterol problems","hyperlipidemia","dyslipidemia","triglycerides"]},
    {"id":"5291005","t":"Hypoglycemia","c":"E16.2","kw":["hypoglycemia","low blood sugar","low sugar"]},
    {"id":"14304000","t":"Thyroid disorder","c":"E07.9","kw":["thyroid","thyroid disorder","thyroid problem","goiter"]},
    {"id":"190268003","t":"Obesity","c":"E66.9","kw":["obesity","obese","overweight","bmi"]},
    {"id":"238131007","t":"Weight loss","c":"R63.4","kw":["weight loss","losing weight","lost weight","unintentional weight loss"]},
    {"id":"8943002","t":"Weight gain","c":"R63.5","kw":["weight gain","gaining weight","gained weight"]},

    {"id":"162116003","t":"Urinary frequency","c":"R35.0","kw":["urinary frequency","frequent urination","peeing a lot","pee a lot","going to bathroom often"]},
    {"id":"49650001","t":"Dysuria","c":"R30.0","kw":["dysuria","pain when peeing","painful urination","burning urination","pain when urinating"]},
    {"id":"75088002","t":"Urinary urgency","c":"R39.15","kw":["urinary urgency","urgent urination","need to go urgently"]},
    {"id":"165232002","t":"Urinary incontinence","c":"R32","kw":["incontinence","urine leakage","leaking urine","can't hold it"]},
    {"id":"68566005","t":"UTI","c":"N39.0","kw":["uti","urinary tract infection","bladder infection"]},
    {"id":"34436003","t":"Hematuria","c":"R31.9","kw":["hematuria","blood in urine","bloody urine"]},
    {"id":"236578006","t":"Kidney stones","c":"N20.0","kw":["kidney stone","kidney stones","renal calculus","nephrolithiasis"]},
    {"id":"197927001","t":"Flank pain","c":"R10.9","kw":["flank pain","kidney pain","side pain"]},

    {"id":"77386006","t":"Pregnancy","c":"Z33.1","kw":["pregnancy","pregnant","expecting"]},
    {"id":"14094001","t":"Amenorrhea","c":"N91.2","kw":["amenorrhea","missed period","late period","no period","absent menstruation","period missed","period late","missed periods"]},
    {"id":"289903006","t":"Menstrual finding","c":"N94.89","kw":["period","periods","menstrual","menstruation","menses"]},
    {"id":"266897007","t":"Menstrual irregularity","c":"N92.6","kw":["irregular period","irregular periods","irregular menstruation"]},
    {"id":"418290006","t":"Dysmenorrhea","c":"N94.6","kw":["painful period","period pain","menstrual cramps","menstrual pain","dysmenorrhea"]},
    {"id":"237079002","t":"Morning sickness","c":"O21.0","kw":["morning sickness"]},
    {"id":"289908002","t":"Pregnancy test positive","c":"Z32.01","kw":["pregnancy test positive","positive pregnancy test"]},
    {"id":"169553002","t":"Contraception","c":"Z30.9","kw":["birth control","oral contraceptive","contraception","contraceptive","condom","condoms","iud"]},

    {"id":"386661006","t":"Fever","c":"R50.9","kw":["fever","fevers","febrile","temperature elevated","hot today","feel hot","feeling hot","warm at night"]},
    {"id":"43724002","t":"Chills","c":"R68.83","kw":["chills","shivering","rigors"]},
    {"id":"84229001","t":"Fatigue","c":"R53.83","kw":["fatigue","tired","exhausted","exhaustion","malaise","lethargy","low energy","feeling tired"]},
    {"id":"42984000","t":"Night sweats","c":"R61","kw":["night sweats","sweating at night","sweats"]},
    {"id":"271807003","t":"Rash","c":"R21","kw":["rash","rashes","skin rash","eruption","hives","urticaria"]},
    {"id":"64779008","t":"Blood","c":"R58","kw":["blood","bleeding","hemorrhage"]},

    {"id":"197480006","t":"Anxiety","c":"F41.9","kw":["anxiety","anxious","worried","worry","panic","panic attack","nervousness"]},
    {"id":"35489007","t":"Depression","c":"F32.9","kw":["depression","depressed","feeling down","low mood","sadness"]},
    {"id":"193462001","t":"Insomnia","c":"G47.00","kw":["insomnia","can't sleep","difficulty sleeping","trouble sleeping","sleep problems"]},
    {"id":"3415004","t":"Substance use","c":"F19.10","kw":["substance use","drug use","recreational drugs","illicit drugs"]},

    {"id":"6142004","t":"Influenza","c":"J11.1","kw":["flu","influenza","flu-like symptoms","flu-like"]},
    {"id":"186747009","t":"COVID-19","c":"U07.1","kw":["covid","covid-19","coronavirus","sars-cov-2"]},
    {"id":"36971009","t":"Sinusitis","c":"J32.9","kw":["sinusitis","sinus infection","sinus"]},
    {"id":"55735004","t":"Bronchitis","c":"J20.9","kw":["bronchitis"]},
    {"id":"302911003","t":"Pharyngitis","c":"J02.9","kw":["pharyngitis","strep throat"]},

    {"id":"294431004","t":"Nickel allergy","c":"L23.0","kw":["nickel","nickel allergy","jewelry allergy","jewelry","contact dermatitis"]},
    {"id":"43116000","t":"Eczema","c":"L30.9","kw":["eczema","atopic dermatitis"]},
    {"id":"9014002","t":"Psoriasis","c":"L40.9","kw":["psoriasis"]},
    {"id":"126485001","t":"Cellulitis","c":"L03.90","kw":["cellulitis"]},
    {"id":"271756007","t":"Pruritus","c":"L29.9","kw":["itching","itchy","pruritus"]},
    {"id":"95324001","t":"Bruising","c":"R23.3","kw":["bruise","bruising","ecchymosis"]},
    {"id":"95328003","t":"Skin lesion","c":"L98.9","kw":["lesion","sore","wound","ulcer","skin lesion"]},

    {"id":"419199007","t":"Allergy","c":"T78.40XA","kw":["allergy","allergic","allergies"]},
    {"id":"91936005","t":"Penicillin allergy","c":"Z88.0","kw":["penicillin allergy","allergic to penicillin"]},
    {"id":"300916003","t":"Latex allergy","c":"L23.4","kw":["latex allergy","allergic to latex"]},
    {"id":"414285001","t":"Food allergy","c":"T78.1XXA","kw":["food allergy","allergic to food"]},
    {"id":"213020009","t":"Drug allergy","c":"T88.7XXA","kw":["drug allergy","medication allergy"]},

    {"id":"271737000","t":"Anemia","c":"D64.9","kw":["anemia","anaemia","low hemoglobin","low iron"]},
    {"id":"363346000","t":"Cancer","c":"C80.1","kw":["cancer","malignancy","carcinoma","tumor","tumour","neoplasm"]},
    {"id":"93143009","t":"Leukemia","c":"C95.90","kw":["leukemia","leukaemia"]},
    {"id":"118600007","t":"Lymphoma","c":"C85.90","kw":["lymphoma"]},

    {"id":"60862001","t":"Tinnitus","c":"H93.19","kw":["tinnitus","ringing in ears","ringing ears"]},
    {"id":"1023001","t":"Hearing loss","c":"H91.90","kw":["hearing loss","deaf","hard of hearing","can't hear"]},
    {"id":"162298006","t":"Ear pain","c":"H92.09","kw":["ear pain","earache","otalgia"]},
    {"id":"65363002","t":"Nosebleed","c":"R04.0","kw":["nosebleed","epistaxis","nose bleed"]},

    {"id":"246636008","t":"Blurred vision","c":"H53.8","kw":["blurry vision","blurred vision","blurry","vision problems","visual changes"]},
    {"id":"225560005","t":"Eye pain","c":"H57.10","kw":["eye pain","painful eye"]},

    {"id":"412244003","t":"Ginger preparation","c":None,"kw":["ginger","ginger things","ginger supplement","ginger chews","ginger candy"]},
    {"id":"372614000","t":"Lisinopril","c":None,"kw":["lisinopril"]},
    {"id":"386864001","t":"Amlodipine","c":None,"kw":["amlodipine","norvasc"]},
    {"id":"109081006","t":"Metformin","c":None,"kw":["metformin","glucophage"]},
    {"id":"372826007","t":"Metoprolol","c":None,"kw":["metoprolol","lopressor","toprol"]},
    {"id":"387475002","t":"Furosemide","c":None,"kw":["furosemide","lasix"]},
    {"id":"387207008","t":"Naproxen","c":None,"kw":["naproxen","aleve","naprosyn"]},
    {"id":"386845007","t":"Gabapentin","c":None,"kw":["gabapentin","neurontin"]},
    {"id":"412588001","t":"Rosuvastatin","c":None,"kw":["rosuvastatin","rosvastetin","crestor"]},
    {"id":"373462001","t":"Atorvastatin","c":None,"kw":["atorvastatin","lipitor"]},
    {"id":"387207008","t":"Ibuprofen","c":None,"kw":["ibuprofen","advil","motrin"]},
    {"id":"387517004","t":"Paracetamol","c":None,"kw":["acetaminophen","tylenol","paracetamol"]},
    {"id":"387458008","t":"Aspirin","c":None,"kw":["aspirin","asa"]},
    {"id":"764146007","t":"Penicillin","c":None,"kw":["penicillin"]},
    {"id":"387501005","t":"Amoxicillin","c":None,"kw":["amoxicillin","amoxil"]},
    {"id":"387322009","t":"Multivitamin","c":None,"kw":["multivitamin","vitamin","vitamins"]},
    {"id":"108774000","t":"Losartan","c":None,"kw":["losartan","cozaar"]},
    {"id":"387207008","t":"Albuterol","c":None,"kw":["albuterol","ventolin","salbutamol","inhaler","puffer","puffers","nebulizer"]},
    {"id":"126218002","t":"Omeprazole","c":None,"kw":["omeprazole","prilosec"]},
    {"id":"387168006","t":"Levothyroxine","c":None,"kw":["levothyroxine","synthroid"]},
    {"id":"387286002","t":"Prednisone","c":None,"kw":["prednisone","prednisolone","steroid","steroids"]},
    {"id":"372756006","t":"Warfarin","c":None,"kw":["warfarin","coumadin","blood thinner"]},
    {"id":"442031002","t":"Recreational methamphetamine","c":None,"kw":["crystal meth","methamphetamine","meth"]},
    {"id":"229880002","t":"Cannabis use","c":None,"kw":["marijuana","cannabis","weed","pot"]},
    {"id":"228388006","t":"Cocaine use","c":None,"kw":["cocaine","crack"]},

    {"id":"77176002","t":"Smoking","c":"F17.210","kw":["smoking","smoker","cigarette","cigarettes","tobacco","pack a day","half a pack"]},
    {"id":"228273003","t":"Alcohol use","c":"F10.10","kw":["alcohol","drinking","drinks","wine","beer","liquor"]},

    {"id":"399208008","t":"Chest X-ray","c":None,"kw":["chest x-ray","cxr","chest xray","x-ray"]},
    {"id":"241441007","t":"Ankle X-ray","c":None,"kw":["ankle x-ray","ankle xray","left ankle x-ray","right ankle x-ray"]},
    {"id":"113091000","t":"MRI","c":None,"kw":["mri","magnetic resonance","lumbar mri","cervical mri","brain mri"]},
    {"id":"77477000","t":"CT scan","c":None,"kw":["ct scan","ct","cat scan","computed tomography"]},
    {"id":"16310003","t":"Ultrasound","c":None,"kw":["ultrasound","ultrasonography","sonogram","echo","echocardiogram"]},
    {"id":"29303009","t":"ECG","c":None,"kw":["ecg","ekg","electrocardiogram"]},
    {"id":"26604007","t":"CBC","c":None,"kw":["complete blood count","cbc","blood count","blood work"]},
    {"id":"20109005","t":"BMP","c":None,"kw":["basic metabolic panel","bmp","metabolic panel"]},
    {"id":"390840006","t":"BNP measurement","c":None,"kw":["bnp","bnp level","natriuretic peptide"]},
    {"id":"167252002","t":"Pregnancy test","c":None,"kw":["pregnancy test","urine pregnancy","hcg","beta hcg"]},
    {"id":"27171005","t":"Urinalysis","c":None,"kw":["urinalysis","urine test","urine analysis","ua"]},
    {"id":"371907003","t":"Oxygen therapy","c":None,"kw":["supplemental oxygen","oxygen therapy","oxygen","nasal cannula"]},
    {"id":"183524004","t":"Referral","c":None,"kw":["referral","referring","consultation","cardiology referral","orthopedic referral","specialist","cardiology","neurology","orthopedic"]},
    {"id":"183519002","t":"ED referral","c":None,"kw":["emergency department","ed referral","emergency room","er"]},
    {"id":"80146002","t":"Appendectomy","c":None,"kw":["appendectomy"]},
    {"id":"236886002","t":"Hysterectomy","c":None,"kw":["hysterectomy"]},
    {"id":"118949005","t":"Wisdom teeth extraction","c":None,"kw":["wisdom teeth","wisdom tooth","wisdom teeth removal","wisdom teeth removed"]},
    {"id":"71388002","t":"Endoscopy","c":None,"kw":["endoscopy","upper endoscopy","egd"]},
    {"id":"73761001","t":"Colonoscopy","c":None,"kw":["colonoscopy"]},
    {"id":"387713003","t":"Stool test","c":None,"kw":["stool test","stool culture","stool sample"]},

    {"id":"75367002","t":"Blood pressure","c":"R03.0","kw":["blood pressure","bp","systolic","diastolic"]},
    {"id":"364075005","t":"Heart rate","c":None,"kw":["heart rate","pulse","bpm"]},
    {"id":"86290005","t":"Respiratory rate","c":None,"kw":["respiratory rate","breaths per minute"]},
    {"id":"386725007","t":"Body temperature","c":"R50.9","kw":["temperature","temp","38 degrees","39 degrees","100.4","101","102","103","104"]},
    {"id":"431314004","t":"Oxygen saturation","c":None,"kw":["oxygen saturation","spo2","o2 sat","91 percent","92 percent","88 percent"]},
    {"id":"12063002","t":"Pitting edema","c":"R60.0","kw":["pitting edema","2+ edema","3+ edema","pitting"]},
    {"id":"442551007","t":"JVD","c":None,"kw":["jvd","jugular venous distension","neck swelling","neck swollen","neck seems to be a little swollen"]},

    {"id":"160303001","t":"Family history of MI","c":"Z82.49","kw":["father had a heart attack","mother had a heart attack","family heart attack","family history of heart attack"]},
    {"id":"160357008","t":"Family history of cancer","c":"Z80.9","kw":["family cancer","cancer in family","family history of cancer"]},
    {"id":"160377001","t":"Family history of diabetes","c":"Z83.3","kw":["family diabetes","family history of diabetes"]},
    {"id":"160303001","t":"Family history of hypertension","c":"Z82.49","kw":["family high blood pressure","father has high blood pressure","mother has high blood pressure","dad has high blood pressure"]},
    {"id":"160364003","t":"Family history of stroke","c":"Z82.3","kw":["family stroke","family history of stroke"]},
]

SNOMED_TABLE.extend([   # concepts added by the straight implementation's coding-accuracy fix
    {"id": "13200003",  "t": "Peptic ulcer", "c": "K27.9",
     "kw": ["peptic ulcer", "gastric ulcer", "duodenal ulcer", "stomach ulcer"]},
    {"id": "431314004", "t": "Oxygen saturation measurement", "c": None,
     "kw": ["oxygen saturation", "o2 saturation", "o2 sat", "spo2"]},
    {"id": "4556007",   "t": "Gastritis", "c": "K29.70", "kw": ["gastritis"]},
    {"id": "38362002",  "t": "Dengue", "c": "A90", "kw": ["dengue", "dengue fever"]},
    {"id": "34014006",  "t": "Viral disease", "c": "B34.9", "kw": ["viral infection", "viral fever", "viral illness"]},
    {"id": "13791008",  "t": "Asthenia", "c": "R53.1", "kw": ["weakness", "generalized weakness", "asthenia"]},
])

_FAST_NORMALIZE = {
    "nauseated": "nausea", "nauseous": "nausea",
    "throw up": "vomiting", "throwing up": "vomiting", "threw up": "vomiting",
    "crampy": "abdominal cramps", "belly pain": "abdominal pain",
    "stomach pain": "abdominal pain", "tummy pain": "abdominal pain",
    "missed period": "amenorrhea", "late period": "amenorrhea",
    "high blood pressure": "hypertension", "high bp": "hypertension",
    "shortness of breath": "dyspnea", "difficulty breathing": "dyspnea",
    "trouble breathing": "dyspnea", "short of breath": "dyspnea",
    "can't breathe": "dyspnea", "winded": "dyspnea", "gasping for air": "dyspnea",
    "chest tightness": "chest pressure", "tight feeling in my chest": "chest pressure",
    "pressure on my chest": "chest pressure",
    "heart racing": "palpitations", "racing of the heart": "palpitations",
    "beating faster": "tachycardia", "heart pounding": "palpitations",
    "lightheaded": "dizziness", "dizzy": "dizziness", "room spinning": "vertigo",
    "high cholesterol": "hyperlipidemia", "cholesterol problems": "hyperlipidemia",
    "loose stools": "diarrhea", "watery stools": "diarrhea",
    "blood in stool": "rectal bleeding", "blood in stools": "rectal bleeding",
    "blood in your stools": "rectal bleeding",
    "burning urination": "dysuria", "pain when peeing": "dysuria",
    "swelling": "edema", "swollen": "edema", "pitting edema": "edema",
    "muscle aches": "myalgia", "body aches": "myalgia",
    "runny nose": "rhinorrhea", "stuffy nose": "nasal congestion",
    "sore throat": "pharyngitis",
    "thirsty": "polydipsia", "excessive thirst": "polydipsia",
    "ginger things": "ginger supplement", "ginger chews": "ginger supplement",
    "birth control": "oral contraceptive",
    "flu-like symptoms": "influenza-like illness", "flu-like": "influenza-like illness",
    "crystal meth": "methamphetamine", "crystal": "methamphetamine",
    "pack a day": "tobacco use", "half a pack": "tobacco use",
    "sweaty": "diaphoresis", "noisy breathing": "stridor",
    "warm at night": "night sweats",
    "neck swollen": "cervical lymphadenopathy",
    "neck seems to be a little swollen": "cervical swelling",
}

CPT_TABLE = [
    {"code": "99213", "description": "Office visit, established patient, low complexity",
     "keywords": ["office visit", "follow-up", "routine visit"]},
    {"code": "99214", "description": "Office visit, established patient, moderate complexity",
     "keywords": ["office visit moderate", "established patient moderate"]},
    {"code": "99215", "description": "Office visit, established patient, high complexity",
     "keywords": ["complex visit", "multiple problems"]},
    {"code": "99202", "description": "Office visit, new patient, low complexity",
     "keywords": ["new patient visit", "initial visit"]},
    {"code": "99281", "description": "Emergency department visit, low severity",
     "keywords": ["emergency department", "ED visit"]},
    {"code": "81025", "description": "Urine pregnancy test",
     "keywords": ["pregnancy test", "urine pregnancy", "HCG", "hcg test"]},
    {"code": "59400", "description": "Routine obstetric care",
     "keywords": ["obstetric care", "prenatal care", "pregnancy care"]},
    {"code": "76801", "description": "Ultrasound, pregnant uterus, first trimester",
     "keywords": ["obstetric ultrasound", "pregnancy ultrasound", "dating ultrasound"]},
    {"code": "81001", "description": "Urinalysis, automated with microscopy",
     "keywords": ["urinalysis", "urine test", "urine analysis", "UA"]},
    {"code": "80048", "description": "Basic metabolic panel (BMP)",
     "keywords": ["basic metabolic panel", "BMP", "metabolic panel"]},
    {"code": "85025", "description": "Complete blood count with differential (CBC)",
     "keywords": ["complete blood count", "CBC", "blood count"]},
    {"code": "80061", "description": "Lipid panel",
     "keywords": ["lipid panel", "cholesterol test"]},
    {"code": "84443", "description": "Thyroid stimulating hormone (TSH)",
     "keywords": ["TSH", "thyroid test", "thyroid function"]},
    {"code": "82947", "description": "Glucose, quantitative, blood",
     "keywords": ["blood glucose", "glucose test", "blood sugar"]},
    {"code": "71046", "description": "Chest X-ray, 2 views",
     "keywords": ["chest x-ray", "chest xray", "CXR"]},
    {"code": "74018", "description": "Abdominal X-ray, 1 view",
     "keywords": ["abdominal x-ray", "KUB", "abdominal xray"]},
    {"code": "73610", "description": "X-ray of ankle, 3 views",
     "keywords": ["ankle x-ray", "ankle xray", "left ankle x-ray", "right ankle x-ray"]},
    {"code": "72148", "description": "MRI of lumbar spine without contrast",
     "keywords": ["MRI lumbar", "MRI spine", "lumbar MRI", "lumbar spine"]},
    {"code": "72141", "description": "MRI of cervical spine without contrast",
     "keywords": ["MRI cervical", "cervical MRI", "cervical spine"]},
    {"code": "83880", "description": "B-type natriuretic peptide (BNP)",
     "keywords": ["BNP", "natriuretic peptide", "BNP level"]},
    {"code": "80053", "description": "Comprehensive metabolic panel (CMP)",
     "keywords": ["comprehensive metabolic panel", "CMP"]},
    {"code": "93000", "description": "Electrocardiogram (ECG), 12-lead",
     "keywords": ["ECG", "EKG", "electrocardiogram"]},
    {"code": "94760", "description": "Pulse oximetry, single reading",
     "keywords": ["oxygen saturation", "pulse oximetry", "SpO2"]},
    {"code": "94002", "description": "Ventilation assist and management",
     "keywords": ["supplemental oxygen", "oxygen therapy", "oxygen"]},
    {"code": "99242", "description": "Consultation, moderate complexity",
     "keywords": ["referral", "referring", "consultation", "follow-up",
                  "cardiology", "orthopedic", "spine specialist"]},
]

CPT_TABLE.extend([
    {"code": "87449", "description": "Infectious agent antigen detection by immunoassay (e.g. dengue NS1)",
     "keywords": ["dengue ns1", "ns1 antigen", "ns1", "dengue test"]},
    {"code": "86790", "description": "Antibody test, virus, not elsewhere specified (e.g. dengue IgM)",
     "keywords": ["dengue igm", "dengue serology"]},
    {"code": "85049", "description": "Platelet count, automated", "keywords": ["platelet count", "platelets test"]},
])
_cpt_embs = embedder.encode([c["description"] for c in CPT_TABLE], convert_to_tensor=True)

def _determine_likely_chapters(entity_text, entity_label):
    text_lower = entity_text.lower()
    chapters = set()
    rules = {
        "K,R": ["nausea", "vomit", "abdomin", "cramp", "stomach", "belly",
                "diarrhea", "constipat", "heartburn", "bowel", "stool", "gastri"],
        "N,O,R,Z": ["pregnan", "amenorrh", "period", "menstr", "morning sickness",
                    "gravid", "obstet", "contracept"],
        "N,R": ["urin", "bladder", "kidney", "micturi", "dysuria", "polydipsia"],
        "I,R": ["hypertens", "blood pressure", "heart", "chest pain", "palpitat",
                "cardiac", "coronary", "angina", "tachycard", "infarct"],
        "J,R": ["cough", "breath", "dyspnea", "wheez", "pulmon", "lung",
                "asthma", "copd", "pneumon", "bronch", "stridor"],
        "M,R": ["muscle", "joint", "back pain", "arthral", "myalgia",
                "sciatica", "sprain", "fracture", "stiff"],
        "E,R": ["diabet", "thyroid", "dehydrat", "cholesterol", "lipid",
                "polydipsia", "obesity", "hypoglyc"],
        "L,T": ["allerg", "nickel", "rash", "dermatit", "eczema", "hives",
                "urticaria", "pruritus"],
        "G,R": ["headache", "migraine", "dizz", "seizure", "numb", "tingling",
                "stroke", "tremor", "vertigo", "syncope"],
        "F": ["anxiety", "depress", "insomnia", "panic", "substance"],
    }
    for chapter_str, terms in rules.items():
        if any(t in text_lower for t in terms):
            chapters.update(chapter_str.split(","))
    if entity_label == "Symptom":
        chapters.add("R")
    if not chapters:
        chapters = set("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
    return list(chapters)

_KNOWN_DRUGS = re.compile(
    r'\b(?:metformin|lisinopril|amlodipine|aspirin|ibuprofen|acetaminophen|'
    r'amoxicillin|azithromycin|ciprofloxacin|prednisone|omeprazole|sumatriptan|'
    r'gabapentin|metoprolol|furosemide|atorvastatin|rosuvastatin|warfarin|'
    r'levothyroxine|losartan|albuterol|naproxen|hydrochlorothiazide|'
    r'clopidogrel|pantoprazole|sertraline|fluoxetine|diazepam|lorazepam)\b',
    re.I
)

# ---- Hindi-English -> English clinical terms (fallback knowledge when the LLM gives no normalisation) ----
HINGLISH_NORMALIZE = {
    "bukhar": "fever", "bukhaar": "fever", "tez bukhar": "high fever", "बुखार": "fever",
    "khansi": "cough", "khaansi": "cough", "खांसी": "cough", "खाँसी": "cough",
    "sir dard": "headache", "sar dard": "headache", "sir mein dard": "headache", "सिर दर्द": "headache",
    "pet dard": "abdominal pain", "pet mein dard": "abdominal pain", "पेट दर्द": "abdominal pain",
    "seene mein dard": "chest pain", "chhati mein dard": "chest pain", "सीने में दर्द": "chest pain",
    "kamar dard": "back pain", "ghutne mein dard": "knee pain", "jodon mein dard": "joint pain",
    "badan dard": "myalgia", "body pain": "myalgia", "बदन दर्द": "myalgia",
    "ulti": "vomiting", "ultiyan": "vomiting", "उल्टी": "vomiting", "ji machlana": "nausea", "matli": "nausea",
    "dast": "diarrhea", "loose motion": "diarrhea", "loose motions": "diarrhea", "दस्त": "diarrhea",
    "kabz": "constipation", "chakkar": "dizziness", "चक्कर": "dizziness",
    "saans phoolna": "dyspnea", "saans lene mein takleef": "dyspnea", "saans ki takleef": "dyspnea",
    "kamzori": "weakness", "कमजोरी": "weakness", "thakan": "fatigue", "थकान": "fatigue",
    "zukam": "common cold", "jukam": "common cold", "nazla": "rhinorrhea", "naak behna": "rhinorrhea",
    "gala kharab": "sore throat", "gale mein dard": "sore throat", "गले में दर्द": "sore throat",
    "sujan": "swelling", "soojan": "swelling", "सूजन": "swelling", "khujli": "pruritus", "खुजली": "pruritus",
    "neend nahi": "insomnia", "bhookh nahi": "loss of appetite", "bhook nahi lagti": "loss of appetite",
    "peshab mein jalan": "dysuria", "baar baar peshab": "urinary frequency",
    "sugar": "diabetes mellitus", "madhumeh": "diabetes mellitus", "शुगर": "diabetes mellitus",
    "bp": "blood pressure", "high bp": "hypertension", "khoon ki kami": "anemia",
    "khoon ki jaanch": "blood test", "khoon jaanch": "blood test", "peshab ki jaanch": "urinalysis",
    "dawai": "medication", "dawa": "medication", "goli": "tablet", "davai": "medication",
}

# ---- assertion cue lexicon (romanised Hindi + Devanagari + English) used by the rule voter ----
NEG_CUES = ["nahi", "nahin", "nhi", "nahii", "nai", "mat", "bilkul nahi", "koi nahi", "kabhi nahi",
            "नहीं", "नही", "मत", "no", "not", "denies", "denied", "without", "never", "negative",
            "none", "nope", "don't", "doesn't", "didn't", "haven't", "hasn't", "isn't", "wasn't"]
UNCERTAIN_CUES = ["shayad", "shaayad", "ho sakta", "ho sakti", "ho sakte", "lagta hai", "lag raha",
                  "sambhav", "शायद", "हो सकता", "हो सकती", "लगता है", "maybe", "possibly", "possible",
                  "probable", "probably", "likely", "suspect", "suspected", "rule out", "could be", "might be"]
HYPOTHETICAL_CUES = ["agar", "if", "in case", "jab bhi", "ho jaye to", "ho jaye toh", "hone par",
                     "अगर", "यदि", "should you", "whenever"]
FAMILY_CUES = ["papa", "pita", "pitaji", "mummy", "mammi", "maa", "mata", "mataji", "bhai", "behen", "behan",
               "dada", "dadi", "nana", "nani", "chacha", "chachi", "mama", "mami", "bua", "family", "father",
               "mother", "brother", "sister", "parents", "grandfather", "grandmother", "uncle", "aunt",
               "पापा", "पिता", "माँ", "मां", "भाई", "बहन", "दादा", "दादी", "नाना", "नानी"]
HISTORICAL_CUES = ["pehle", "pahle", "purana", "purani", "saal pehle", "pichle saal", "bachpan mein",
                   "पहले", "पुराना", "years ago", "previously", "in the past", "history of", "used to"]
AFFIRM_CUES = ["haan", "han", "haa", "ji haan", "ji", "yes", "yeah", "hmm haan", "bilkul", "हाँ", "हां", "जी"]

_WB = r"[\wऀ-ॿ]"   # a word character, including Devanagari vowel signs (matras)
def _cue_regex(words):
    alts = sorted({re.escape(w) for w in words}, key=len, reverse=True)
    return re.compile(rf"(?<!{_WB})(?:{'|'.join(alts)})(?!{_WB})", re.I)

NEG_RE, UNC_RE, HYP_RE = _cue_regex(NEG_CUES), _cue_regex(UNCERTAIN_CUES), _cue_regex(HYPOTHETICAL_CUES)
FAM_RE, HIST_RE, AFF_RE = _cue_regex(FAMILY_CUES), _cue_regex(HISTORICAL_CUES), _cue_regex(AFFIRM_CUES)
print(f"[kb] SNOMED concepts: {len(SNOMED_TABLE)} | CPT: {len(CPT_TABLE)} | Hindi-English terms: {len(HINGLISH_NORMALIZE)}")

In [ ]:
# ============================ 6. AGENT TOOLS (deterministic, reusable) ============================
_PUNCT = ".,!?;:\"'()[]{}<>-–—…।॥|/\\*"

def _norm_tok(tok):
    return tok.strip(_PUNCT).lower()

def _tok_offsets(text):
    """[(normalised token, start, end)] with offsets that exclude surrounding punctuation."""
    out = []
    for m in re.finditer(r"\S+", text):
        raw = m.group()
        norm = _norm_tok(raw)
        if norm:
            lead = len(raw) - len(raw.lstrip(_PUNCT))
            trail = len(raw) - len(raw.rstrip(_PUNCT))
            out.append((norm, m.start() + lead, m.end() - trail))
    return out

# ---------------------------------------------------------------- language ----
_HI_MARKERS = {"hai", "hain", "nahi", "nahin", "kya", "aap", "aapko", "mujhe", "mera", "meri", "mere", "mein",
               "ko", "se", "ka", "ki", "ke", "ho", "raha", "rahi", "tha", "thi", "bhi", "aur", "kab", "kitne",
               "din", "dard", "haan", "ji", "kuch", "bahut", "theek", "dawai", "abhi", "toh", "wala", "wali"}

def language_profile(text):
    """Tool: how Hindi / English / code-mixed is this transcript (script + romanised-Hindi markers)."""
    dev = sum(1 for ch in text if "ऀ" <= ch <= "ॿ")      # includes vowel signs (not isalpha)
    letters = sum(1 for ch in text if ch.isalpha() or "ऀ" <= ch <= "ॿ") or 1
    words = re.findall(r"[a-zA-Z']+", text.lower())
    hi_rom = sum(w in _HI_MARKERS for w in words)
    dev_ratio, rom_ratio = dev / letters, hi_rom / max(1, len(words))
    if dev_ratio > 0.8:
        label = "Hindi (Devanagari)"
    elif dev_ratio > 0.15:
        label = "Hindi-English code-mixed (mixed script)"
    elif rom_ratio > 0.08:
        label = "Hindi-English code-mixed (romanised)"
    else:
        label = "English"
    return {"label": label, "devanagari_ratio": round(dev_ratio, 3), "romanised_hindi_ratio": round(rom_ratio, 3)}

def word_preservation(src, out):
    """Tool: fraction of source words that survive in the output (catches translation / dropped words)."""
    a = Counter(t for t in (_norm_tok(x) for x in src.split()) if t)
    b = Counter(t for t in (_norm_tok(x) for x in re.sub(r"(?im)^\s*(doctor|patient|attendant)\s*:", " ", out).split()) if t)
    return sum((a & b).values()) / max(1, sum(a.values()))

# ---------------------------------------------------------------- spans ----
def align_span(span, text):
    """Tool (VerifiNER step 1): find a span in the transcript.
    exact -> case-insensitive -> token sequence (punctuation-insensitive) -> fuzzy (ASR spelling variants).
    Returns {start, end, matched, method, score} or None when the span is not in the transcript."""
    s = (span or "").strip()
    if len(s) < 2 or not text:
        return None
    i = text.find(s)
    if i >= 0:
        return {"start": i, "end": i + len(s), "matched": s, "method": "exact", "score": 1.0}
    i = text.lower().find(s.lower())
    if i >= 0:
        return {"start": i, "end": i + len(s), "matched": text[i:i + len(s)], "method": "case_insensitive", "score": 1.0}
    stoks = [t for t in (_norm_tok(x) for x in s.split()) if t]
    if not stoks:
        return None
    ttoks, n = _tok_offsets(text), len(stoks)
    for j in range(len(ttoks) - n + 1):
        if [t[0] for t in ttoks[j:j + n]] == stoks:
            st, en = ttoks[j][1], ttoks[j + n - 1][2]
            return {"start": st, "end": en, "matched": text[st:en], "method": "token", "score": 1.0}
    target = " ".join(stoks)
    if len(target) < 4:                      # too short to fuzzy-match safely
        return None
    best = (0.0, None)
    for w in sorted({max(1, n - 1), n, n + 1}):
        for j in range(len(ttoks) - w + 1):
            cand = " ".join(t[0] for t in ttoks[j:j + w])
            sm = difflib.SequenceMatcher(None, target, cand)
            if sm.real_quick_ratio() < FUZZY_SPAN_THRESHOLD or sm.quick_ratio() < FUZZY_SPAN_THRESHOLD:
                continue
            r = sm.ratio()
            if r > best[0]:
                best = (r, (ttoks[j][1], ttoks[j + w - 1][2]))
    if best[1] and best[0] >= FUZZY_SPAN_THRESHOLD:
        st, en = best[1]
        return {"start": st, "end": en, "matched": text[st:en], "method": "fuzzy", "score": round(best[0], 3)}
    return None

def context_window(text, start, end, width=120):
    a, b = max(0, start - width), min(len(text), end + width)
    return ("..." if a else "") + text[a:start] + "[[" + text[start:end] + "]]" + text[end:b] + ("..." if b < len(text) else "")

_TURN_RE = re.compile(r"(?im)^\s*(doctor|patient|attendant)\s*:")

def turn_at(text, pos):
    """Tool: the speaker turn containing character `pos`, plus the next turn."""
    marks = list(_TURN_RE.finditer(text))
    for k, m in enumerate(marks):
        end = marks[k + 1].start() if k + 1 < len(marks) else len(text)
        if m.start() <= pos < end:
            nxt = None
            if k + 1 < len(marks):
                nend = marks[k + 2].start() if k + 2 < len(marks) else len(text)
                nxt = {"speaker": marks[k + 1].group(1).lower(), "text": text[marks[k + 1].end():nend]}
            return {"speaker": m.group(1).lower(), "start": m.end(), "end": end, "next": nxt}
    return None

def _clause(text, start, end, lo, hi, n_pre, n_post, splitter=r"[,.!?;।]"):
    """Words just before / after a span, cut at clause punctuation and at the turn boundary."""
    pre = re.split(splitter, text[max(lo, start - 80):start])[-1].split()[-n_pre:] if n_pre else []
    post = re.split(splitter, text[end:min(hi, end + 80)])[0].split()[:n_post] if n_post else []
    return " ".join(pre), " ".join(post)

def rule_assertion(text, start, end):
    """Tool (VerifiNER voter): existence class from Hindi-English cue words, or None (abstain)."""
    turn = turn_at(text, start)
    lo, hi = (turn["start"], turn["end"]) if turn else (0, len(text))
    pre_h, _ = _clause(text, start, end, lo, hi, 8, 0, splitter=r"[.!?;।]")
    if HYP_RE.search(pre_h):
        return "Hypothetical"
    pre, post = _clause(text, start, end, lo, hi, 4, 3)
    if UNC_RE.search(pre) or UNC_RE.search(post):
        return "Uncertain"
    if NEG_RE.search(pre) or NEG_RE.search(post):
        return "Absent"
    # doctor asks "Khansi?" -> the patient's next turn answers it
    if turn and turn["speaker"] == "doctor" and turn["next"] and turn["next"]["speaker"] != "doctor":
        rest = text[end:hi]
        if "?" in rest[:40]:
            ans = " ".join(turn["next"]["text"].split()[:3])
            if NEG_RE.search(ans):
                return "Absent"
            if AFF_RE.search(ans):
                return "Present"
    return None

def rule_experiencer(text, start, end):
    turn = turn_at(text, start)
    lo, hi = (turn["start"], turn["end"]) if turn else (0, len(text))
    pre, post = _clause(text, start, end, lo, hi, 5, 3)
    return "family" if FAM_RE.search(pre) or FAM_RE.search(post) else None

def rule_temporality(text, start, end):
    turn = turn_at(text, start)
    lo, hi = (turn["start"], turn["end"]) if turn else (0, len(text))
    pre, post = _clause(text, start, end, lo, hi, 6, 5, splitter=r"[.!?;।]")
    return "historical" if HIST_RE.search(pre) or HIST_RE.search(post) else None

# ---------------------------------------------------------------- knowledge + coding ----
def snomed_keyword(term, non_diagnosis=False):
    """Most specific (longest) SNOMED keyword contained in the term.
    non_diagnosis=True -> only drug / procedure / test concepts (the table's entries without an ICD crosswalk),
    so "dengue NS1 test" maps to a test, not to the disease Dengue."""
    tl = (term or "").lower().strip()
    best, best_len = None, 0
    for entry in SNOMED_TABLE:
        if non_diagnosis and entry.get("c"):
            continue
        for kw in entry["kw"]:
            hit = re.search(r"\b" + re.escape(kw) + r"\b", tl) if len(kw) < 4 else (kw in tl)
            if hit and len(kw) > best_len:
                best, best_len = entry, len(kw)
    return best

def kb_lookup_batch(terms, k=2):
    """Tool (VerifiNER step 2): knowledge for each term = SNOMED concept + nearest ICD-10 descriptions."""
    if not terms:
        return []
    embs = embedder.encode(list(terms), convert_to_tensor=True, batch_size=64)
    hits = util.semantic_search(embs, icd10_embeddings, top_k=k)
    out = []
    for term, hs in zip(terms, hits):
        kn = {}
        sn = snomed_keyword(term)
        if sn:
            kn["snomed"] = f'{sn["t"]} (SNOMED {sn["id"]})'
        kn["icd10_nearest"] = [f'{icd10_db.iloc[h["corpus_id"]]["code"]}: '
                               f'{icd10_db.iloc[h["corpus_id"]]["description"]} (sim {h["score"]:.2f})' for h in hs]
        out.append(kn)
    return out

def get_snomed(term, entity_type=None):
    e = snomed_keyword(term, non_diagnosis=entity_type in ("Drug", "Test", "Procedure"))
    if not e:
        return None
    return {"concept_id": e["id"], "term": e["t"], "entity_text": term, "method": "keyword",
            "crosswalk_icd10": e.get("c"), "crosswalk_icd10_desc": e["t"] if e.get("c") else None}

def _retrieve_rerank(search_text, entity_label):
    likely = _determine_likely_chapters(search_text, entity_label)
    q = embedder.encode(search_text, convert_to_tensor=True)
    hits = util.semantic_search(q, icd10_embeddings, top_k=25)[0]
    seen = {}
    for h in hits:
        if h["score"] >= ICD10_COSINE_THRESHOLD:
            code = icd10_db.iloc[h["corpus_id"]]["code"]
            cand = {"code": code, "description": icd10_db.iloc[h["corpus_id"]]["description"],
                    "cosine_score": round(float(h["score"]) + (0.05 if code[0] in likely else 0.0), 4),
                    "chapter": code[0]}
            if code not in seen or cand["cosine_score"] > seen[code]["cosine_score"]:
                seen[code] = cand
    cands = list(seen.values())
    if not cands:
        return {"code": None, "description": None, "method": "no_match"}
    scores = cross_encoder.predict([(search_text, c["description"]) for c in cands])
    for c, s in zip(cands, scores):
        c["rerank_score"] = round(float(s), 4)
    cands.sort(key=lambda x: x["rerank_score"] + (0.5 if x["chapter"] in likely else 0.0), reverse=True)
    best = cands[0]
    if best["rerank_score"] >= RERANK_THRESHOLD:
        best["method"] = "retrieve_rerank"
        return best
    return {"code": None, "description": None, "method": "below_threshold"}

def get_icd10(term, entity_label="Symptom", snomed=None):
    """Tool: ICD-10-CM via the SNOMED crosswalk, else semantic retrieval + cross-encoder re-ranking."""
    if snomed and snomed.get("crosswalk_icd10"):
        code = snomed["crosswalk_icd10"]
        m = icd10_db[icd10_db["code"] == code]
        desc = m.iloc[0]["description"] if len(m) else snomed.get("crosswalk_icd10_desc", "")
        return {"code": code, "description": desc, "method": "SNOMED_crosswalk"}
    return _retrieve_rerank(term, entity_label)

def get_cpt(term):
    """Tool: CPT code by keyword, else semantic similarity + re-ranking."""
    tl = (term or "").lower()
    for entry in CPT_TABLE:
        for kw in entry["keywords"]:
            kwl = kw.lower()
            if (re.search(r"\b" + re.escape(kwl) + r"\b", tl) if len(kwl) < 4 else kwl in tl):
                return {"code": entry["code"], "description": entry["description"], "method": "keyword"}
    q = embedder.encode(term, convert_to_tensor=True)
    hits = [h for h in util.semantic_search(q, _cpt_embs, top_k=5)[0] if h["score"] >= 0.5]
    if not hits:
        return None
    scores = cross_encoder.predict([(term, CPT_TABLE[h["corpus_id"]]["description"]) for h in hits])
    k = int(np.argmax(scores))
    if float(scores[k]) >= 0.5:
        e = CPT_TABLE[hits[k]["corpus_id"]]
        return {"code": e["code"], "description": e["description"], "method": "semantic_reranked"}
    return None

_VITALS_RE = re.compile(r"\b(blood pressure|bp|pulse|heart rate|respiratory rate|oxygen saturation|spo2|"
                        r"temperature|temp|bpm|\d+\s*(?:over|by|/)\s*\d+)\b", re.I)
def _is_vital(text):
    t = text or ""
    return bool(_VITALS_RE.search(t)) and bool(re.search(r"\d", t))

# ---------------------------------------------------------------- note grounding ----
def grounding_text(transcript, spans):
    """Transcript + the English meaning of every verified span, so English note text can be
    checked against a Hindi-English conversation."""
    tl = transcript.lower()
    extra = [s.get("normalized_en", "") for s in spans]
    extra += [v for k, v in HINGLISH_NORMALIZE.items() if re.search(rf"(?<!{_WB}){re.escape(k)}(?!{_WB})", tl)]
    return transcript + "\n" + " ".join(x for x in extra if x)

def span_terms(s):
    terms = set()
    for t in (s.get("normalized_en"), s.get("text")):
        if t and re.search(r"[A-Za-z]", t):
            terms.add(t.lower().strip())
    n = (s.get("normalized_en") or "").lower().strip()
    if n in _FAST_NORMALIZE:
        terms.add(_FAST_NORMALIZE[n])
    sn = snomed_keyword(n) if n else None
    if sn:
        terms.add(sn["t"].lower())
    return [t for t in terms if len(t) >= 3]

def is_mentioned(s, note_lower):
    """Tool: is this K-span reflected in the note (term, most of its words, or all of its numbers)?"""
    for t in span_terms(s):
        if t in note_lower:
            return True
        words = re.findall(r"[a-z]{4,}", t)
        if words and sum(w in note_lower or w.rstrip("s") in note_lower for w in words) >= max(1, (len(words) + 1) // 2):
            return True
    nums = re.findall(r"\d+(?:\.\d+)?", (s.get("normalized_en") or "") + " " + (s.get("text") or ""))
    return bool(nums) and all(n in note_lower for n in nums)

print("[tools] span alignment, cue-rule voter, knowledge lookup, SNOMED/ICD-10/CPT coding, grounding: ready")

In [ ]:
# ============================ 7. NOTE + METRIC HELPERS (carried over from the straight implementation) ============================
# Section parsing, cleaning, plan grounding, speculation stripping, combined-A/P splitting,
# note assembly, ROUGE/BLEU/BERTScore, the anchored judge prompt and clinical inference.
# These are the pieces that were already validated; the agents below call them as tools.

def _parse_ksoap_sections(raw_output):
    cleaned = raw_output
    cleaned = re.sub(r'\*\*([A-Z]):\*\*', r'\1:', cleaned)
    cleaned = re.sub(r'\*\*([A-Z])\*\*\s*:', r'\1:', cleaned)
    cleaned = re.sub(r'\*\*', '', cleaned)
    cleaned = re.sub(r'#{1,4}\s*', '', cleaned)

    sections = {"K": "", "S": "", "O": "", "A": "", "P": ""}

    section_pattern = re.compile(
        r'(?:^|\n)\s*([KSOAP])\s*:\s*',
        re.MULTILINE
    )

    matches = list(section_pattern.finditer(cleaned))

    for i, m in enumerate(matches):
        key = m.group(1)
        start = m.end()
        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(cleaned)

        content = cleaned[start:end].strip()

        if key in sections and not sections[key]:
            sections[key] = content

    return sections

_SECTION_HELP = {
    "S": "S: subjective. Write it as short labelled lines in this order: Chief complaint; HPI "
         "(onset, site, character, severity out of 10, radiation, what makes it better or worse, "
         "and any medication already tried and whether it helped); Pertinent negatives (every "
         "symptom the patient denied); Past medical history (the patient's age if stated, "
         "every chronic condition named, every current medication BY NAME, allergies, previous "
         "surgeries or procedures, and any past cardiac or major event with its timing - write "
         "'none reported' where the patient denied them); Family history (relation, condition "
         "and age or timing if stated); Social history (occupation or studies, living "
         "situation, alcohol, tobacco including pack-years or years smoked if stated, "
         "recreational drugs). Include any concern the patient "
         "expresses about their own diagnosis. If the transcript is internally inconsistent, "
         "record the clinically coherent version (for example episodes lasting seconds that "
         "recur over three days, rather than 'constant and lasting seconds').",
    "O": "O: objective - vital signs and physical-examination findings stated in the transcript. "
         "If the transcript contains none, write exactly: No objective findings stated.",
    "A": "A: assessment - a neutral one-or-two-line summary of the confirmed problems AS NAMED "
         "in the transcript. Do NOT add your own diagnosis or differential diagnoses. If the "
         "doctor stated no assessment, write exactly: No assessment given.",
    "P": "P: plan - what the doctor said WILL or MAY be done: investigations, tests, referrals, "
         "medications, follow-up and conditional plans. Phrases such as 'we will do some "
         "investigations', 'we may need to start you on X' or 'we will get you an ECG' ARE the "
         "plan and must be recorded (state them as planned or conditional). Include any working "
         "diagnosis the doctor said they are investigating. Do NOT invent orders the doctor never "
         "mentioned. Only if the doctor said nothing at all about next steps, write exactly: "
         "No plan given.",
}

_AI_SUGGESTION_PROMPT = (
    "You are a clinical decision-support assistant. You are given a doctor-patient transcript "
    "and the SOAP note documenting it.\n"
    "Suggest the standard-of-care next steps that were NOT already ordered by the doctor: "
    "relevant tests, referrals, counselling or safety advice for this presentation.\n"
    "Write 2 to 4 short lines, plain text, no markdown. Do not repeat what the doctor already "
    "ordered. This is advisory only and is clearly labelled as such."
)

# ---- stray-heading + label cleanup -------------------------------------------------
# matches "3. Objective Section", "B. Objective", "C. Assessment and Plan", "**Plan**", etc.
_HEAD_RE = re.compile(r'^\s*(\*\*)?\s*(?:[0-9]+|[A-Ea-e])?\s*[\.\)]?\s*'
                      r'(subjective|objective|assessment|plan)'
                      r'(\s+and\s+plan)?\s*(sections?)?\s*(\(\s*[SOAP]\s*\))?'
                      r'\s*(\*\*)?\s*:?\s*$', re.I)

# sentences that are the model's own speculation rather than the transcript's content
_SPECULATION_RE = re.compile(
    r'(differential\s+diagnos|may\s+(?:include|represent)|could\s+(?:be|include|indicate)|'
    r'concern(?:ing)?\s+for|suspicious\s+for|rul(?:e|ed|ing)\s+out|'
    r'cannot\s+be\s+(?:ruled\s+out|excluded)|consider(?:ation)?\s+of|'
    r'likely\s+(?:represents|due\s+to)|work-?up\s+for)', re.I)

_STRICT_A_PROMPT = (
    "You are a clinical scribe. Write ONLY the Assessment line of a SOAP note.\n"
    "State the clinician's impression or working diagnosis AS GIVEN in the transcript - for "
    "example a likely or suspected diagnosis the doctor named, and the cause if the doctor "
    "stated one. Use ONLY what the transcript contains.\n"
    "Do NOT add differential diagnoses, possibilities or causes the doctor did not mention.\n"
    "If the doctor stated no assessment at all, reply with exactly: No assessment given.\n"
    "Reply with the assessment text only, no 'A:' label, plain text."
)

_SPEC_STOPWORDS = {"differential","differentials","diagnosis","diagnoses","include","includes",
    "including","likely","possible","possibly","may","could","would","consider","consideration",
    "suspicious","suspected","suggestive","concern","concerning","rule","ruled","out","cannot",
    "excluded","represents","represent","etiology","presentation","versus","given","this","that",
    "with","without","and","the","for","most","some","other","related"}

def _strip_speculation(txt, transcript=""):
    """Keep the Assessment grounded: a sentence that speculates is kept ONLY if the doctor
    actually said its content (its medical terms appear in the transcript). Invented
    diagnoses/differentials the doctor never mentioned are dropped."""
    tl = (transcript or "").lower()
    keep = []
    for p in re.split(r'(?<=[.!?])\s+', txt):
        p = p.strip()
        if not p:
            continue
        if not _SPECULATION_RE.search(p):
            keep.append(p); continue                       # no speculation marker -> keep
        words = [w for w in re.findall(r"[a-z][a-z'\-]{3,}", p.lower())
                 if w not in _SPEC_STOPWORDS and w not in _PLAN_STOPWORDS]
        if not words:
            keep.append(p); continue
        grounded = sum(1 for w in words if w in tl or w.rstrip('s') in tl)
        if grounded >= (len(words) + 1) // 2:              # majority of terms are in transcript
            keep.append(p)                                 # the doctor said this -> keep
    return " ".join(keep).strip()

def _clean_section(txt):
    lines = txt.splitlines()
    while lines and _HEAD_RE.match(lines[0]):
        lines.pop(0)
    while lines and (not lines[-1].strip() or _HEAD_RE.match(lines[-1])):
        lines.pop()
    out = "\n".join(lines).strip()
    # keep S/O/A/P purely extractive: any advisory tail belongs in the AI Suggestion part
    out = re.sub(r'\[?\s*Disclaimer\s*\]?.*', '', out, flags=re.S | re.I).strip()
    out = re.sub(r'\bAI Suggestion\s*:?.*', '', out, flags=re.S | re.I).strip()
    return out

# ---- plan verification: every ordered item must actually appear in the transcript ----
# The fine-tuned model was trained on notes whose Plan always has content, so when the
# doctor orders nothing it confabulates one ("order abdominal ultrasound", "NPO status").
# Neither the drug regex nor cosine grounding catches that. So we verify each plan item
# against the transcript by its distinctive words and drop the ones with no support.
_PLAN_STOPWORDS = {
    "order","ordered","orders","initiate","initiated","start","started","begin","consider",
    "considered","advise","advised","advice","counsel","counselled","counseled","recommend",
    "recommended","schedule","scheduled","arrange","patient","doctor","given","pending",
    "status","further","potential","need","regarding","management","evaluation","review",
    "results","result","plan","planned","follow","followup","follow-up","appointment",
    "discuss","discussed","provide","provided","ensure","monitor","monitoring","continue",
    "instruct","instructed","perform","performed","obtain","obtained","refer","referral",
    "assess","assessment","treatment","treat","care","clinical","plans","plan's","plan."}

def _verify_plan_grounded(plan_text, transcript):
    """Drop plan items whose distinctive words never appear in the transcript."""
    tl = transcript.lower()
    items = [p.strip(" -*\u2022\t") for p in re.split(r'(?<=[.!?])\s+|\n+', plan_text)
             if p.strip(" -*\u2022\t")]
    kept, dropped = [], []
    for it in items:
        words = {w for w in re.findall(r"[a-z][a-z'\-]{3,}", it.lower())
                 if w not in _PLAN_STOPWORDS}
        if not words:                       # nothing distinctive -> keep (e.g. a canonical line)
            kept.append(it); continue
        if any(w in tl or w[:-1] in tl or w.rstrip('s') in tl for w in words):
            kept.append(it)
        else:
            dropped.append(it)
    if dropped:
        print(f"  [SOAP] dropped {len(dropped)} unsupported plan item(s): "
              f"{'; '.join(d[:44] for d in dropped)}")
    return " ".join(kept).strip(), dropped

# the model sometimes writes its own "nothing here" wording instead of the canonical line
_EMPTY_SECTION_RE = re.compile(
    r'^\W*(no|none|not)\b[^.]{0,80}\b(finding|findings|exam|examination|data|documented|'
    r'provided|recorded|stated|available|performed|result|results|assessment|plan|orders?)\b',
    re.I)

# ---- combined "Assessment and Plan" handling -------------------------------------
# The fine-tuned model was trained on gold notes that use ONE "Assessment and Plan"
# section, so it writes headers like "4. Assessment and Plan" and "A/P:". The parser
# only matches a single letter + colon ("A:"), so those lines are not recognised and
# everything after O -- including the real plan -- gets absorbed into the O section.
# Fix: normalise those headers to "A:" before parsing, then split the plan back out.
# NOTE ON ORDER: the colon-bearing form ("A/P:") must be converted FIRST. If the
# heading rule ran first with an optional colon it would delete that line outright,
# taking the whole plan with it.
_AP_INLINE_RE = re.compile(                      # "A/P:" / "Assessment and Plan:" + content
    r'(?im)^[ \t]*(?:\d+[\.\)]\s*)?\**\s*(?:assessment\s*(?:and|&|/)\s*plan|a\s*/\s*p)'
    r'\s*\**\s*:[ \t]*')
_AP_HEADING_RE = re.compile(                     # bare title line, NO colon
    r'(?im)^[ \t]*(?:\d+[\.\)]\s*)?\**\s*(?:assessment\s*(?:and|&|/)\s*plan|a\s*/\s*p)'
    r'\s*\**[ \t]*$')

def _normalize_note_labels(raw):
    """Turn combined Assessment-and-Plan headers into a plain 'A:' the parser can read."""
    out = _AP_INLINE_RE.sub("A: ", raw)      # "A/P: ..." -> "A: ..."   (must run first)
    out = _AP_HEADING_RE.sub("", out)        # drop a bare "4. Assessment and Plan" title
    return out

_PLAN_LABEL_RE = re.compile(r'(?im)^[\s\-*#>]*\**\s*plan\**\s*:\**.*$')

def _split_ap(a_text, p_text):
    """If the Plan ended up inside A (combined section), move it into P."""
    if len(re.findall(r'[A-Za-z]+', p_text)) >= 2 and not _EMPTY_SECTION_RE.match(p_text.strip()):
        return a_text, p_text                # P already has real content
    ms = list(_PLAN_LABEL_RE.finditer(a_text))
    if not ms:
        return a_text, p_text
    m = ms[-1]
    a_part = a_text[:m.start()].rstrip()
    tail   = re.sub(r'(?im)^[\s\-*#>]*\**\s*plan\**\s*:\**\s*', '', m.group(0))
    p_part = (tail + "\n" + a_text[m.end():]).strip()
    # a numbered problem list may continue after the plan -> that part belongs back in A
    nxt = re.search(r'(?im)^\s*\d+[\.\)]\s+\S', p_part)
    if nxt and nxt.start() > 0:
        a_part = (a_part + "\n" + p_part[nxt.start():]).strip()
        p_part = p_part[:nxt.start()].strip()
    if len(re.findall(r'[A-Za-z]+', p_part)) < 2 or len(re.findall(r'[A-Za-z]+', a_part)) < 2:
        return a_text, p_text
    print("  [SOAP] plan was inside the Assessment block -> moved into P")
    return a_part, p_part

_PLACEHOLDER = {"O": "No objective findings stated.",
                "A": "No assessment given.",
                "P": "No plan given."}

AI_SUGGESTION_HEADER = "AI Suggestion (not stated in the transcript; advisory only)"

def k_section(spans):
    """K-SOAP 'K' line: every verified K-span as ("span as spoken", English concept) [Entity type, Assertion]."""
    parts = []
    for s in spans or []:
        if s.get("_inferred"):
            continue                                   # rule-inferred syndromes are not spans from the conversation
        concept = (s.get("normalized_en") or s["text"]).strip()
        assertion = s["existence"] + ("; family" if s.get("experiencer") == "family" else "")
        parts.append(f'("{s["text"]}", {concept[:1].upper() + concept[1:]}) [{s["type"]}, {assertion}]')
    return ", ".join(parts) if parts else "No clinical keywords extracted."

def _assemble(sec, ai_suggestion="", spans=None):
    """Display note in K-SOAP format: K (verified spans, when given), S/O/A/P, then the labelled AI Suggestion part.
    The K line and the AI Suggestion are dropped before scoring (extractive_soap_only)."""
    out = f"K: {k_section(spans)}\n\n" if spans is not None else ""
    out += "".join(f"{k}: {sec[k].strip()}\n\n" for k in "SOAP" if sec.get(k, "").strip())
    if ai_suggestion.strip():
        out += f"[Disclaimer]\n{AI_SUGGESTION_HEADER}:\n{ai_suggestion.strip()}\n"
    return out.strip()

def extractive_soap_only(note_text):
    """Everything the transcript supports: drop the K line and the AI Suggestion part.
    This is what the evaluation sees, so labelled advice never lowers the scores."""
    t = re.sub(r'(?ms)^\s*K\s*:.*?(?=^\s*[SOAP]\s*:|\Z)', '', note_text)   # drop K block
    t = re.split(r'\[?\s*Disclaimer\s*\]?|\bAI Suggestion\b', t, maxsplit=1, flags=re.I)[0]
    return t.strip()

def _tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

def rouge_n(candidate, reference, n=1):
    cand_tok = _tokenize(candidate)
    ref_tok = _tokenize(reference)
    if not cand_tok or not ref_tok:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    cand_ng = Counter(tuple(cand_tok[i:i+n]) for i in range(len(cand_tok)-n+1))
    ref_ng = Counter(tuple(ref_tok[i:i+n]) for i in range(len(ref_tok)-n+1))

    overlap = sum((cand_ng & ref_ng).values())
    p = overlap / max(sum(cand_ng.values()), 1)
    r = overlap / max(sum(ref_ng.values()), 1)
    f1 = 2 * p * r / max(p + r, 1e-8)
    return {"precision": round(p, 4), "recall": round(r, 4), "f1": round(f1, 4)}

def rouge_l(candidate, reference):
    cand_tok = _tokenize(candidate)
    ref_tok = _tokenize(reference)
    if not cand_tok or not ref_tok:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    m, n = len(cand_tok), len(ref_tok)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if cand_tok[i-1] == ref_tok[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]
    p = lcs / max(m, 1)
    r = lcs / max(n, 1)
    f1 = 2 * p * r / max(p + r, 1e-8)
    return {"precision": round(p, 4), "recall": round(r, 4), "f1": round(f1, 4)}

def bleu_score(candidate, reference, max_n=4):
    cand_tok = _tokenize(candidate)
    ref_tok = _tokenize(reference)
    if not cand_tok or not ref_tok:
        return {"bleu": 0.0, "bleu1": 0.0, "bleu2": 0.0, "bleu4": 0.0, "brevity_penalty": 0.0}

    import math

    c_len = len(cand_tok)
    r_len = len(ref_tok)
    if c_len == 0:
        bp = 0.0
    elif c_len >= r_len:
        bp = 1.0
    else:
        bp = math.exp(1 - r_len / c_len)

    precisions = []
    individual = {}
    for n in range(1, max_n + 1):
        cand_ng = Counter(tuple(cand_tok[i:i+n]) for i in range(max(len(cand_tok)-n+1, 0)))
        ref_ng = Counter(tuple(ref_tok[i:i+n]) for i in range(max(len(ref_tok)-n+1, 0)))
        clipped = sum(min(cand_ng[ng], ref_ng[ng]) for ng in cand_ng)
        total = max(sum(cand_ng.values()), 1)
        p_n = clipped / total
        precisions.append(p_n)
        individual[f"bleu{n}"] = round(p_n, 4)

    log_avg = 0.0
    weight = 1.0 / max_n
    for p_n in precisions:
        if p_n == 0:
            log_avg = float('-inf')
            break
        log_avg += weight * math.log(p_n)

    if log_avg == float('-inf'):
        bleu = 0.0
    else:
        bleu = bp * math.exp(log_avg)

    return {
        "bleu": round(bleu, 4),
        "bleu1": individual.get("bleu1", 0.0),
        "bleu2": individual.get("bleu2", 0.0),
        "bleu4": individual.get("bleu4", 0.0),
        "brevity_penalty": round(bp, 4),
    }

def bertscore_document(candidate, reference):
    if not candidate.strip() or not reference.strip():
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    c_emb = embedder.encode(candidate, convert_to_tensor=True)
    r_emb = embedder.encode(reference, convert_to_tensor=True)
    sim = float(util.cos_sim(c_emb, r_emb)[0][0])
    return {"precision": round(sim, 4), "recall": round(sim, 4), "f1": round(sim, 4)}

def bertscore_sentence(candidate, reference):
    c_sents = [s.strip() for s in re.split(r'[.!?]+', candidate) if len(s.strip()) > 10]
    r_sents = [s.strip() for s in re.split(r'[.!?]+', reference) if len(s.strip()) > 10]
    if not c_sents or not r_sents:
        return bertscore_document(candidate, reference)

    c_embs = embedder.encode(c_sents, convert_to_tensor=True)
    r_embs = embedder.encode(r_sents, convert_to_tensor=True)
    sim_matrix = util.cos_sim(c_embs, r_embs)

    p = float(sim_matrix.max(dim=1).values.mean())
    r = float(sim_matrix.max(dim=0).values.mean())
    f1 = 2 * p * r / max(p + r, 1e-8)
    return {"precision": round(p, 4), "recall": round(r, 4), "f1": round(f1, 4)}

_FAIR_JUDGE_PROMPT = """You are an experienced clinical documentation reviewer grading a SOAP note written from a doctor-patient TRANSCRIPT. Judge the note against the TRANSCRIPT only, not against any ideal of exhaustive length: a concise note that correctly captures the encounter is a GOOD note.

First write a brief 2-3 sentence ANALYSIS, then score FOUR axes as integers 1-5.
Use this professional scale (4 = a solid, clinically usable note; reserve 5 for excellent; 1-2 only for genuinely poor notes):

quality_of_soap (structure and clinical correctness)
  5 excellent: clean S/O/A/P, precise terminology, no errors
  4 good, clinically usable: correct structure, sound terminology, trivial nits at most
  3 acceptable: right idea, some structural or wording issues
  2 weak: notable structural or clinical problems
  1 poor: wrong structure or clinically incorrect

adequacy_wrt_transcript (captures the clinically relevant information that IS in the transcript)
  5 captures essentially all relevant findings
  4 captures the chief complaint and the main findings; minor omissions only
  3 captures the main complaint but misses some relevant detail
  2 captures little of the relevant content
  1 misses most of it

faithfulness (contains ONLY what the transcript supports)
  5 fully supported, nothing invented
  4 supported overall, at most one minor unsupported detail
  3 mostly supported, a couple of unsupported details
  2 several unsupported statements
  1 substantially invented content

alignment_score (overall, does the note faithfully reflect this encounter)
  5 excellent overall reflection
  4 good overall reflection, minor gaps
  3 generally aligned with some gaps
  2 weak alignment
  1 poor alignment

How to apply the scale fairly:
- A section that correctly states "No objective findings stated", "No assessment given" or
  "No plan given" when the transcript genuinely contains none is CORRECT documentation.
  Do not treat such a section as an omission or as poor quality.
- The transcript comes from automatic speech recognition and may contain recognition errors
  and mislabelled speakers. Judge the note on the information it could faithfully extract,
  not on the transcript's own defects.
- A concise, well-organised note that captures the clinically relevant content deserves 4.
  Reserve 5 for a note you would sign without edits, and 1-2 for genuinely poor work.

TRANSCRIPT:
{transcript}

SOAP NOTE TO SCORE:
{soap}

Write your ANALYSIS (2-3 sentences), then on the final line output ONLY this JSON object:
{{"quality_of_soap": <1-5>, "adequacy_wrt_transcript": <1-5>, "faithfulness": <1-5>, "alignment_score": <1-5>}}"""

def _get_confirmed_data(coded_entities):
    ids, texts = set(), set()
    for e in coded_entities:
        if e["status"] == "Confirmed":
            texts.add(e["text"].lower())
            sn = e.get("snomed")
            if sn:
                ids.add(sn["concept_id"])
    return ids, texts

def _has(sn_ids, texts, target_ids, target_kw):
    return (any(t in sn_ids for t in target_ids) or
            any(kw in txt for txt in texts for kw in target_kw))

def _make_inferred(text, sn_id, sn_term, icd_code, icd_desc):
    return {
        "text": text, "label": "Disease", "status": "Confirmed",
        "reasoning": "Deterministic clinical inference",
        "first_offset": None, "_inferred": True,
        "snomed": {"concept_id": sn_id, "term": sn_term,
                   "entity_text": text, "method": "clinical_inference",
                   "crosswalk_icd10": icd_code, "crosswalk_icd10_desc": icd_desc},
        "icd10_cm": {"code": icd_code, "description": icd_desc,
                     "cosine_score": 1.0, "rerank_score": 10.0,
                     "method": "clinical_inference"},
    }

def _run_clinical_inference(coded_entities):
    sn_ids, texts = _get_confirmed_data(coded_entities)
    inferred = []
    if (_has(sn_ids, texts, ["14094001"], ["missed period", "late period", "amenorrhea"]) and
        _has(sn_ids, texts, ["422587007", "422400008"], ["nausea", "vomiting", "nauseated"]) and
        _has(sn_ids, texts, ["162116003"], ["urinary frequency"])):
        inferred.append(_make_inferred("suspected pregnancy (inferred: amenorrhea + nausea + urinary frequency)",
            "77386006", "Pregnancy", "Z32.00", "Encounter for pregnancy test, result unknown"))
    if (_has(sn_ids, texts, ["267036007"], ["shortness of breath", "difficulty breathing"]) and
        _has(sn_ids, texts, ["267038008"], ["swelling", "edema", "swollen"]) and
        _has(sn_ids, texts, [], ["orthopnea", "lying flat", "pillows", "crackles", "heart failure"])):
        inferred.append(_make_inferred("CHF exacerbation (inferred: dyspnea + edema + orthopnea)",
            "42343007", "Congestive heart failure", "I50.9", "Heart failure, unspecified"))
    if (_has(sn_ids, texts, ["29857009"], ["chest pain", "chest pressure"]) and
        _has(sn_ids, texts, ["267036007"], ["shortness of breath", "difficulty breathing"]) and
        _has(sn_ids, texts, [], ["smoking", "smoker", "cocaine", "high blood pressure", "hypertension", "diabetes"])):
        inferred.append(_make_inferred("suspected ACS (inferred: chest pain + dyspnea + risk factors)",
            "394659003", "Acute coronary syndrome", "I24.9", "Acute ischemic heart disease, unspecified"))
    if (_has(sn_ids, texts, ["62315008"], ["diarrhea", "loose stools"]) and
        _has(sn_ids, texts, ["422587007", "422400008"], ["nausea", "vomiting", "nauseated"]) and
        _has(sn_ids, texts, ["21522001", "73063007"], ["abdominal pain", "belly pain", "cramp", "crampy"])):
        inferred.append(_make_inferred("suspected gastroenteritis (inferred: diarrhea + N/V + abdominal pain)",
            "235856003", "Gastroenteritis", "K52.9", "Noninfective gastroenteritis and colitis, unspecified"))
    return inferred

print("[helpers] note parsing / grounding / metrics / judge prompt / clinical inference: ready")

In [ ]:
# ============================ 8. SHARED AGENT STATE ============================
# Every agent reads what it needs from this state and returns only the keys it changes.
# `trace` is append-only (operator.add), so the full chain of agent decisions is kept.
from typing import TypedDict, Annotated, List, Dict, Any, Optional

class SOAPState(TypedDict, total=False):
    # inputs
    audio_path: Optional[str]
    reference_soap: Optional[str]
    # Transcript
    raw_transcript: str
    transcript: str
    language_profile: Dict[str, Any]
    # K-Spans + VerifiNER
    kspan_attempts: int
    kspan_feedback: str
    candidate_spans: List[Dict[str, Any]]
    verified_spans: List[Dict[str, Any]]
    rejected_spans: List[Dict[str, Any]]
    verifier_decision: str
    # SOAP generator + Reflection
    soap: Dict[str, str]
    reflection_round: int
    reflection_feedback: str
    reflection_decision: str
    reflection_log: List[Dict[str, Any]]
    best_soap: Dict[str, str]
    best_score: Optional[float]
    best_issues: Dict[str, Any]
    # Addition
    coded_spans: List[Dict[str, Any]]
    cpt_codes: List[Dict[str, Any]]
    ai_suggestion: str
    final_note: str
    # Evaluation + report
    eval_metrics: Dict[str, Any]
    report: str
    timings: Dict[str, float]
    trace: Annotated[List[Dict[str, Any]], operator.add]


def _step(agent, action, **detail):
    """One trace event (also printed live so you can watch the agents work)."""
    ev = {"agent": agent, "action": action, "time": datetime.now().strftime("%H:%M:%S"), **detail}
    if VERBOSE:
        extra = "  ".join(f"{k}={v}" for k, v in detail.items() if not isinstance(v, (list, dict)))
        print(f"  [{agent}] {action}" + (f"   ({extra})" if extra else ""))
    return ev

def _timed(state, name, t0):
    d = dict(state.get("timings") or {})
    d[name] = round(d.get(name, 0.0) + time.time() - t0, 2)
    return d

print("[state] SOAPState defined")

---
## Whiteboard box 1: **Transcript** → `asr_agent` + `diarization_agent`

* **ASR agent**: loads the audio at 16 kHz (8 kHz dataset audio is resampled), runs Whisper, and profiles the
  language (Hindi / English / code-mixed, Devanagari vs romanised), so later agents know what they are reading.
* **Diarization agent**: labels Doctor / Patient / Attendant turns *without translating anything*. It checks its
  own output with the `word_preservation` tool. If the LLM dropped or translated words (< 85% kept), it retries
  with that feedback and keeps the best attempt. If the transcript already has speaker labels, it skips the LLM.

In [ ]:
# ============================ 9. TRANSCRIPT AGENTS: ASR + DIARIZATION ============================
def asr_agent(state):
    t0 = time.time()
    import librosa
    y, _ = librosa.load(state["audio_path"], sr=16000, mono=True)    # 8 kHz dataset audio -> 16 kHz
    gk = {"task": "transcribe"}                                         # never translate to English
    if ASR_LANGUAGE:
        gk["language"] = ASR_LANGUAGE
    out = asr_pipe({"raw": y, "sampling_rate": 16000}, return_timestamps=True, generate_kwargs=gk)
    text = (out.get("text", "") if isinstance(out, dict) else str(out)).strip()
    if not text:
        raise ValueError("ASR produced an empty transcript")
    prof = language_profile(text)
    return {"raw_transcript": text, "language_profile": prof, "timings": _timed(state, "asr", t0),
            "trace": [_step("ASR", "transcribed audio", seconds=round(len(y) / 16000, 1),
                            chars=len(text), language=prof["label"])]}


DIARIZE_SYSTEM = """You are the Diarization agent in a clinical documentation pipeline.
You receive an automatic speech-recognition transcript of a medical consultation in Hindi, English, or
Hindi-English code-mixed speech (romanised and/or Devanagari). It has no speaker labels.
Split it into turns and label each turn Doctor, Patient or Attendant (a relative speaking for the patient).
  Doctor   : asks diagnostic questions, examines, states vitals/findings, diagnoses, prescribes, advises.
  Patient  : describes symptoms, answers questions, gives personal and medical history.
  Attendant: a family member / companion speaking about the patient.
STRICT RULES
- Copy the words EXACTLY. Do NOT translate, transliterate, correct spelling, summarise, add or drop words.
  Keep Devanagari as Devanagari and romanised Hindi as romanised Hindi.
- Only decide where each turn starts and who is speaking.
Return ONLY a JSON object: {"turns": [{"speaker": "Doctor|Patient|Attendant", "text": "..."}]}"""

_DIARIZE_EX_IN = "Bataiye kya problem hai? Doctor sahab do din se bukhar hai aur khansi bhi. Khansi mein balgam aata hai? Nahi, sukhi khansi hai."
_DIARIZE_EX_OUT = json.dumps({"turns": [
    {"speaker": "Doctor", "text": "Bataiye kya problem hai?"},
    {"speaker": "Patient", "text": "Doctor sahab do din se bukhar hai aur khansi bhi."},
    {"speaker": "Doctor", "text": "Khansi mein balgam aata hai?"},
    {"speaker": "Patient", "text": "Nahi, sukhi khansi hai."}]}, ensure_ascii=False)

def diarize_messages(raw, feedback=""):
    """The exact prompt the Diarization agent sends (also used to build its LoRA training data)."""
    user = f"Transcript:\n{raw}\n\nJSON object:"
    if feedback:
        user = f"{feedback}\n\n{user}"
    return [{"role": "system", "content": DIARIZE_SYSTEM},
            {"role": "user", "content": f"Transcript:\n{_DIARIZE_EX_IN}\n\nJSON object:"},
            {"role": "assistant", "content": _DIARIZE_EX_OUT},
            {"role": "user", "content": user}]

def _normalise_labels(text):
    text = re.sub(r"(?im)^\s*(?:dr\.?|doctor|डॉक्टर)\s*:", "Doctor:", text)
    text = re.sub(r"(?im)^\s*(?:patient|pt\.?|मरीज|रोगी)\s*:", "Patient:", text)
    return re.sub(r"(?im)^\s*(?:attendant|relative)\s*:", "Attendant:", text).strip()

def diarization_agent(state):
    t0 = time.time()
    raw = state["raw_transcript"].strip()
    prof = state.get("language_profile") or language_profile(raw)
    labelled = len(re.findall(r"(?im)^\s*(?:dr\.?|doctor|patient|pt\.?|attendant|डॉक्टर|मरीज)\s*:", raw))
    if labelled >= 2:                                  # agent decision: nothing to do
        return {"transcript": _normalise_labels(raw), "language_profile": prof,
                "timings": _timed(state, "diarization", t0),
                "trace": [_step("Diarization", "input already has speaker labels -> kept as is",
                                language=prof["label"])]}
    feedback, best, events = "", ("", 0.0), []
    for attempt in (1, 2):
        d = LLM.json(diarize_messages(raw, feedback), role="diarization",
                     max_tokens=min(16000, int(len(raw) / 2.5) + 800))
        turns = [t for t in (d.get("turns") or []) if isinstance(t, dict) and str(t.get("text", "")).strip()]
        text = "\n".join(f"{str(t.get('speaker', 'Patient')).strip().title()}: {str(t['text']).strip()}" for t in turns)
        kept = word_preservation(raw, text) if text else 0.0
        events.append(_step("Diarization", f"attempt {attempt}: {len(turns)} turns", words_kept=f"{kept:.0%}"))
        if kept > best[1]:
            best = (text, kept)
        if kept >= 0.85:
            break
        feedback = (f"FEEDBACK: your previous attempt kept only {kept:.0%} of the original words. Do NOT drop, "
                    "translate or paraphrase anything - only split into turns and add speaker labels.")
    text, kept = best
    if kept < 0.5:                                      # unusable -> keep the raw ASR text, unlabelled
        text = raw
        events.append(_step("Diarization", "labelling failed the preservation check -> using raw ASR text"))
    return {"transcript": _normalise_labels(text), "language_profile": prof,
            "timings": _timed(state, "diarization", t0), "trace": events}

---
## Whiteboard box 2: **K-Spans + info** (10 entities, 5 assertion classes) → `kspan_agent`

Extracts every clinically relevant **key span** exactly as it was spoken. Medical terms are kept in their
original form (Hindi, Devanagari or English), as the project's *Medical Entity Preservation* step requires.
Each span carries its **info**:

| field | values |
|---|---|
| `type` | one of the 10 `ENTITY_TYPES` |
| `existence` | one of the 5 `EXISTENCE_CLASSES` → Present · Absent · Uncertain · Hypothetical · Not specified |
| `normalized_en` | English clinical term (`sir mein dard` → headache) |
| `severity`, `temporality`, `experiencer`, `speaker`, `cue` | nature, current/historical, patient/family, who said it, the deciding cue word |

The output is the whiteboard format, e.g. `(Disease, Existence: Present) "sugar" → diabetes mellitus`.
Long conversations are extracted chunk by chunk. If VerifiNER sends the agent back, the verifier's feedback
is included in the retry.

In [ ]:
# ============================ 10. K-SPAN AGENT (K-Spans + info) ============================
_KSPAN_TEMPLATE = """You are the K-Span agent. From a Hindi-English code-mixed doctor-patient conversation, extract every
clinically relevant KEY SPAN (K-span) together with its information.

ENTITY TYPES (exactly one of these <<NT>>):
<<TYPES>>

EXISTENCE / ASSERTION CLASS (exactly one of these <<NE>>):
<<EXIST>>

For every span also give:
  normalized_en : the standard English clinical term ("sir mein dard" -> "headache", "bukhar" -> "fever")
  severity      : one of <<SEV>>
  temporality   : one of <<TEMP>>
  experiencer   : one of <<EXP>>  (family = a relative has it, e.g. "papa ko sugar hai")
  speaker       : Doctor | Patient | Attendant (who said the span)
  cue           : the word(s) that decided the existence class ("nahi", "ho sakta hai", "agar"), or ""

RULES
1. "text" MUST be copied character-for-character from the transcript, in its ORIGINAL language and script.
   Never translate or re-spell the span; the English meaning goes in normalized_en.
2. Extract pertinent negatives: "khansi nahi hai" -> text "khansi", existence Absent.
3. A drug and its dose are two spans: "Paracetamol" (Drug) and "650 mg din mein teen baar" (Dosage).
4. Vital signs keep their value: "BP 118 by 76" (Vital).
5. If a concept is repeated, extract its clearest mention once, unless its existence changes.
6. Do NOT extract greetings, filler words, bare time phrases ("teen din se") or speaker labels.
7. Be thorough: a full consultation usually has 15-40 spans.

EXAMPLE
Transcript:
Doctor: Bukhar kab se hai? Khansi bhi hai?
Patient: Do din se bukhar hai, khansi nahi hai. Mere papa ko sugar hai.
Doctor: Shayad viral hai. Agar rash aaye toh turant aana. CBC karwa lijiye.
Output:
<<EXAMPLE>>

Return ONLY the JSON object {"entities": [...]}."""

_KSPAN_EXAMPLE = {"entities": [
    {"text": "bukhar", "type": "Symptom", "existence": "Present", "normalized_en": "fever", "severity": "unspecified",
     "temporality": "current", "experiencer": "patient", "speaker": "Patient", "cue": "hai"},
    {"text": "khansi", "type": "Symptom", "existence": "Absent", "normalized_en": "cough", "severity": "unspecified",
     "temporality": "current", "experiencer": "patient", "speaker": "Patient", "cue": "nahi"},
    {"text": "sugar", "type": "Disease", "existence": "Present", "normalized_en": "diabetes mellitus",
     "severity": "unspecified", "temporality": "current", "experiencer": "family", "speaker": "Patient", "cue": "hai"},
    {"text": "viral", "type": "Disease", "existence": "Uncertain", "normalized_en": "viral infection",
     "severity": "unspecified", "temporality": "current", "experiencer": "patient", "speaker": "Doctor", "cue": "Shayad"},
    {"text": "rash", "type": "Symptom", "existence": "Hypothetical", "normalized_en": "rash", "severity": "unspecified",
     "temporality": "unspecified", "experiencer": "patient", "speaker": "Doctor", "cue": "Agar"},
    {"text": "CBC", "type": "Test", "existence": "Present", "normalized_en": "complete blood count",
     "severity": "unspecified", "temporality": "current", "experiencer": "patient", "speaker": "Doctor", "cue": "karwa lijiye"},
]}

def _kspan_system():
    return (_KSPAN_TEMPLATE
            .replace("<<NT>>", str(len(ENTITY_TYPES)))
            .replace("<<NE>>", str(len(EXISTENCE_CLASSES)))
            .replace("<<TYPES>>", "\n".join(f"  - {k}: {v}" for k, v in ENTITY_TYPES.items()))
            .replace("<<EXIST>>", "\n".join(f"  - {k}: {v}" for k, v in EXISTENCE_CLASSES.items()))
            .replace("<<SEV>>", ", ".join(SEVERITY_VALUES))
            .replace("<<TEMP>>", ", ".join(TEMPORALITY_VALUES))
            .replace("<<EXP>>", ", ".join(EXPERIENCER_VALUES))
            .replace("<<EXAMPLE>>", json.dumps(_KSPAN_EXAMPLE, ensure_ascii=False, indent=1)))

# label normalisation (LLMs drift: "medication", "negated", "possible", ...)
_TYPE_SYN = {"medication": "Drug", "medicine": "Drug", "drugs": "Drug", "dose": "Dosage", "frequency": "Dosage",
             "condition": "Disease", "diagnosis": "Disease", "disorder": "Disease", "problem": "Disease",
             "sign": "Symptom", "finding": "Symptom", "complaint": "Symptom", "lab": "Test", "investigation": "Test",
             "imaging": "Test", "lab test": "Test", "vital sign": "Vital", "vitals": "Vital", "body part": "Anatomy",
             "body site": "Anatomy", "site": "Anatomy", "social": "Lifestyle", "social history": "Lifestyle",
             "habit": "Lifestyle", "surgery": "Procedure", "treatment": "Procedure", "referral": "Procedure"}
_EXIST_SYN = {"confirmed": "Present", "positive": "Present", "yes": "Present", "present": "Present",
              "negated": "Absent", "negative": "Absent", "denied": "Absent", "no": "Absent", "absent": "Absent",
              "possible": "Uncertain", "probable": "Uncertain", "suspected": "Uncertain", "uncertain": "Uncertain",
              "conditional": "Hypothetical", "hypothetical": "Hypothetical", "future": "Hypothetical",
              "unknown": "Not specified", "not specified": "Not specified", "not_specified": "Not specified",
              "unspecified": "Not specified", "not stated": "Not specified"}

def norm_type(t):
    t = str(t or "").strip()
    for k in ENTITY_TYPES:
        if t.lower() == k.lower():
            return k
    mapped = _TYPE_SYN.get(t.lower())
    return mapped if mapped in ENTITY_TYPES else None

def norm_existence(x):
    x = str(x or "").strip()
    for k in EXISTENCE_CLASSES:
        if x.lower() == k.lower():
            return k
    mapped = _EXIST_SYN.get(x.lower().replace("-", " "))
    return mapped if mapped in EXISTENCE_CLASSES else None

def _norm_choice(v, allowed, default):
    v = str(v or "").strip().lower()
    return v if v in allowed else default

def _chunks(text, limit):
    out, cur = [], ""
    for ln in text.split("\n"):
        if cur and len(cur) + len(ln) + 1 > limit:
            out.append(cur)
            cur = ln
        else:
            cur = f"{cur}\n{ln}" if cur else ln
    if cur:
        out.append(cur)
    final = []
    for c in out:                        # an unlabelled single-line transcript -> split at sentence ends
        while len(c) > limit * 1.5:
            cut = max(c.rfind(". ", 0, limit), c.rfind("? ", 0, limit), c.rfind("। ", 0, limit))
            cut = cut + 1 if cut > limit // 2 else limit
            final.append(c[:cut])
            c = c[cut:].lstrip()
        final.append(c)
    return [c for c in final if c.strip()]

def kspan_messages(chunk, feedback=""):
    """The exact prompt the K-Span agent sends for one transcript chunk (also used for LoRA training data)."""
    user = f"Transcript:\n{chunk}\n\nExtract all K-spans. JSON object:"
    if feedback:
        user = f"FEEDBACK FROM VerifiNER ON YOUR PREVIOUS ATTEMPT:\n{feedback}\n\n{user}"
    return [{"role": "system", "content": _kspan_system()}, {"role": "user", "content": user}]

def kspan_agent(state):
    t0 = time.time()
    text = state["transcript"]
    attempt = int(state.get("kspan_attempts") or 0) + 1
    fb = state.get("kspan_feedback") or ""
    cands, seen = [], set()
    for chunk in _chunks(text, KSPAN_CHUNK_CHARS):
        d = LLM.json(kspan_messages(chunk, fb), role="kspan", max_tokens=4000, temperature=0.0)
        for e in d.get("entities") or []:
            if not isinstance(e, dict) or not str(e.get("text", "")).strip():
                continue
            c = {"text": str(e["text"]).strip(),
                 "type": norm_type(e.get("type")) or str(e.get("type", "")).strip(),
                 "existence": norm_existence(e.get("existence")) or "Not specified",
                 "normalized_en": str(e.get("normalized_en", "") or "").strip(),
                 "severity": _norm_choice(e.get("severity"), SEVERITY_VALUES, "unspecified"),
                 "temporality": _norm_choice(e.get("temporality"), TEMPORALITY_VALUES, "unspecified"),
                 "experiencer": _norm_choice(e.get("experiencer"), EXPERIENCER_VALUES, "patient"),
                 "speaker": str(e.get("speaker", "") or "").strip().title(),
                 "cue": str(e.get("cue", "") or "").strip()}
            if not c["normalized_en"]:
                c["normalized_en"] = HINGLISH_NORMALIZE.get(c["text"].lower(), c["text"])
            key = (c["text"].lower(), c["type"], c["existence"])
            if key not in seen:
                seen.add(key)
                cands.append(c)
    return {"candidate_spans": cands, "kspan_attempts": attempt, "timings": _timed(state, "kspan", t0),
            "trace": [_step("K-Span", f"attempt {attempt}: extracted {len(cands)} candidate spans",
                            types=dict(Counter(c["type"] for c in cands)),
                            existence=dict(Counter(c["existence"] for c in cands)))]}

---
## Whiteboard box 3: **K-Spans refinement — VerifiNER** (span → entity) → `verifiner_agent`

This follows the VerifiNER idea (*verification-augmented NER via knowledge-grounded reasoning with LLMs*):
verify the **span** first, then the **entity** (type and assertion) using retrieved **knowledge**, and settle
disagreements by **consistency voting**.

1. **Span verification** (tool `align_span`): exact → case-insensitive → token → fuzzy match against the transcript.
   Spans that are not in the conversation are rejected as hallucinated. Near-matches (ASR spellings such as
   *bukhaar/bukhar*) are corrected to the exact transcript text.
2. **Knowledge retrieval** (tool `kb_lookup_batch`): SNOMED CT concept + nearest ICD-10-CM descriptions for each span.
3. **Entity verification** by the LLM, sampled `VERIFY_VOTES` times: is it a real entity? correct type? correct
   existence? correct boundaries?
4. **Consistency voting**: votes from the extractor + every verifier sample + a **Hindi-English cue-rule voter**
   (`nahi` → Absent, `shayad / ho sakta hai` → Uncertain, `agar … toh` → Hypothetical, a doctor's question
   answered by the patient).

**Agent decision:** if too few spans survive (< `VERIFY_MIN_SURVIVAL`), VerifiNER sends the K-Span agent back
with the rejection reasons as feedback (at most `KSPAN_MAX_ATTEMPTS` times).

In [ ]:
# ============================ 11. VerifiNER AGENT (K-span refinement) ============================
_VERIFIER_TEMPLATE = """You are VerifiNER, a knowledge-grounded verifier for clinical named-entity recognition on
Hindi-English code-mixed doctor-patient conversations.
Each candidate has: the span (marked [[like this]] inside its context), the proposed type and existence, and
KNOWLEDGE retrieved from SNOMED CT / ICD-10-CM for its English meaning.

For EVERY candidate decide:
  is_entity      : true if it is a genuine clinically relevant entity of one of the types below; false for
                   filler, greetings, bare time phrases or non-clinical words.
  span_ok        : true if the span boundaries are right; false if it should be shorter or longer.
  corrected_span : when span_ok is false, the corrected span copied EXACTLY from the context (same script); else "".
  type           : the correct type, one of: <<TYPES_INLINE>>
  existence      : the correct existence class, one of: <<EXIST_INLINE>>
                   Read Hindi cues carefully: nahi / nahin / mat = negation; shayad / ho sakta hai / lagta hai =
                   uncertain; agar ... toh = hypothetical. A doctor's question answered by the patient decides
                   existence ("Khansi?" -> "Nahi" = Absent).
  experiencer    : patient | family | other
  normalized_en  : the standard English clinical term
  reason         : a few words

TYPE DEFINITIONS
<<TYPES>>
EXISTENCE DEFINITIONS
<<EXIST>>

Use the KNOWLEDGE to check the TYPE (a SNOMED disorder is a Disease, a drug concept is a Drug, ...).
The CONTEXT decides EXISTENCE.
Return ONLY a JSON object:
{"verdicts": [{"id": 0, "is_entity": true, "span_ok": true, "corrected_span": "", "type": "...",
               "existence": "...", "experiencer": "patient", "normalized_en": "...", "reason": "..."}]}"""

def _verifier_system():
    return (_VERIFIER_TEMPLATE
            .replace("<<TYPES_INLINE>>", ", ".join(ENTITY_TYPES))
            .replace("<<EXIST_INLINE>>", ", ".join(EXISTENCE_CLASSES))
            .replace("<<TYPES>>", "\n".join(f"  - {k}: {v}" for k, v in ENTITY_TYPES.items()))
            .replace("<<EXIST>>", "\n".join(f"  - {k}: {v}" for k, v in EXISTENCE_CLASSES.items())))

def _vote(values, tie_break):
    vals = [v for v in values if v]
    if not vals:
        return tie_break
    ranked = Counter(vals).most_common()
    tied = [v for v, n in ranked if n == ranked[0][1]]
    return tie_break if tie_break in tied else tied[0]

def verifiner_agent(state):
    t0 = time.time()
    text, cands = state["transcript"], state.get("candidate_spans") or []
    events, rejected, stage = [], [], []

    # ---- step 1: span verification (tool) ----
    for c in cands:
        al = align_span(c["text"], text)
        if not al:
            rejected.append({**c, "_reason": "span not found in the transcript (hallucinated span)"})
            continue
        c2 = {**c, "text": al["matched"], "offset": al}
        if al["method"] not in ("exact", "case_insensitive"):
            c2["_span_fix"] = f'"{c["text"]}" -> "{al["matched"]}" ({al["method"]} {al["score"]})'
        stage.append(c2)
    events.append(_step("VerifiNER", "step 1 span verification",
                        aligned=len(stage), hallucinated=len(rejected),
                        corrected=sum(1 for c in stage if c.get("_span_fix"))))

    # ---- step 2: knowledge retrieval (tool) ----
    knowledge = kb_lookup_batch([c.get("normalized_en") or c["text"] for c in stage])
    for c, kn in zip(stage, knowledge):
        c["knowledge"] = kn

    # ---- step 3: LLM entity verification, sampled VERIFY_VOTES times ----
    verdicts = {i: [] for i in range(len(stage))}
    system = _verifier_system()
    for b in range(0, len(stage), VERIFY_BATCH):
        batch = stage[b:b + VERIFY_BATCH]
        payload = [{"id": i, "span": c["text"], "proposed_type": c["type"], "proposed_existence": c["existence"],
                    "context": context_window(text, c["offset"]["start"], c["offset"]["end"]),
                    "knowledge": c["knowledge"]} for i, c in enumerate(batch)]
        user = "CANDIDATES:\n" + json.dumps(payload, ensure_ascii=False, indent=1) + "\n\nJSON object:"
        for v in range(VERIFY_VOTES):
            d = LLM.json([{"role": "system", "content": system}, {"role": "user", "content": user}],
                         role="verifier", max_tokens=min(8000, 180 * len(batch) + 400),
                         temperature=0.0 if v == 0 else 0.7)
            for item in d.get("verdicts") or []:
                if isinstance(item, dict) and isinstance(item.get("id"), int) and 0 <= item["id"] < len(batch):
                    verdicts[b + item["id"]].append(item)
    events.append(_step("VerifiNER", f"step 3 entity verification ({VERIFY_VOTES} samples per span)",
                        verdicts=sum(len(v) for v in verdicts.values())))

    # ---- step 4: consistency voting (extractor + verifier samples + cue-rule voter) ----
    verified, seen_off, seen_concept = [], set(), set()
    n_type_fix = n_exist_fix = 0
    for i, c in enumerate(stage):
        vs = verdicts[i]
        first = vs[0] if vs else {}
        if vs and sum(bool(v.get("is_entity", True)) for v in vs) * 2 <= len(vs):
            reasons = Counter(str(v.get("reason", "")).strip() for v in vs if not v.get("is_entity", True))
            rejected.append({**c, "_reason": "not a clinical entity: " + (reasons.most_common(1)[0][0] or "verifier vote")})
            continue
        # span boundary correction (majority of samples say the span is wrong)
        fixes = [str(v.get("corrected_span", "")).strip() for v in vs if v.get("span_ok") is False]
        if fixes and len(fixes) * 2 > len(vs):
            new = Counter(f for f in fixes if f).most_common(1)
            al = align_span(new[0][0], text) if new else None
            if al and al["matched"] != c["text"]:
                c["_span_fix"] = f'boundary "{c["text"]}" -> "{al["matched"]}"'
                c["text"], c["offset"] = al["matched"], al
        # type vote
        t_votes = [norm_type(c["type"])] + [norm_type(v.get("type")) for v in vs]
        new_type = _vote(t_votes, norm_type(first.get("type")) or norm_type(c["type"]))
        # existence vote (+ rule voter)
        rv = rule_assertion(text, c["offset"]["start"], c["offset"]["end"])
        e_votes = [c["existence"]] + [norm_existence(v.get("existence")) for v in vs] + ([rv] if rv else [])
        new_exist = _vote(e_votes, norm_existence(first.get("existence")) or c["existence"])
        # experiencer / temporality (LLM majority, cue rules as extra voters)
        x_votes = [c["experiencer"]] + [_norm_choice(v.get("experiencer"), EXPERIENCER_VALUES, None) for v in vs]
        rx = rule_experiencer(text, c["offset"]["start"], c["offset"]["end"])
        new_exp = _vote(x_votes + ([rx] if rx else []), c["experiencer"])
        rt = rule_temporality(text, c["offset"]["start"], c["offset"]["end"])
        if rt and c["temporality"] != "historical":
            c["temporality"] = rt
        if not new_type:
            rejected.append({**c, "_reason": f'type "{c["type"]}" is not one of the {len(ENTITY_TYPES)} entity types'})
            continue
        n_type_fix += new_type != c["type"]
        n_exist_fix += new_exist != c["existence"]
        c["verification"] = {
            "type_votes": f"{sum(t == new_type for t in t_votes)}/{len(t_votes)}",
            "existence_votes": f"{sum(e == new_exist for e in e_votes)}/{len(e_votes)}",
            "rule_vote": rv, "was": f'{c["type"]}/{c["existence"]}' if (new_type, new_exist) != (c["type"], c["existence"]) else ""}
        c["type"], c["existence"], c["experiencer"] = new_type, new_exist, new_exp
        if first.get("normalized_en"):
            c["normalized_en"] = str(first["normalized_en"]).strip()
        c.pop("knowledge", None)
        off_key = (c["offset"]["start"], c["offset"]["end"], c["type"])
        concept_key = (c["text"].lower(), c["type"], c["existence"], c["experiencer"])   # same mention twice
        if off_key in seen_off or concept_key in seen_concept:
            rejected.append({**c, "_reason": "duplicate of an already verified span"})
            continue
        seen_off.add(off_key)
        seen_concept.add(concept_key)
        verified.append(c)
    verified.sort(key=lambda s: s["offset"]["start"])
    events.append(_step("VerifiNER", "step 4 consistency voting", verified=len(verified),
                        rejected=len(rejected), type_changed=n_type_fix, existence_changed=n_exist_fix))

    # ---- agent decision: accept, or send the K-Span agent back with feedback ----
    survival = len(verified) / max(1, len(cands))
    retry = ((not verified) or survival < VERIFY_MIN_SURVIVAL) and int(state.get("kspan_attempts") or 1) < KSPAN_MAX_ATTEMPTS
    feedback = ""
    if retry:
        top = Counter(r["_reason"].split(":")[0] for r in rejected).most_common(4)
        examples = "; ".join(f'"{r["text"]}" ({r["_reason"][:60]})' for r in rejected[:8])
        feedback = (f"Only {len(verified)} of {len(cands)} spans survived verification. Main problems: "
                    + ", ".join(f"{k} x{n}" for k, n in top) + f". Examples: {examples}. Copy spans EXACTLY "
                    "from the transcript, use only the listed types, and extract clinical content only.")
    events.append(_step("VerifiNER", "decision: " + ("RETRY K-Span agent" if retry else "accept"),
                        survival=f"{survival:.0%}"))
    return {"verified_spans": verified, "rejected_spans": rejected, "kspan_feedback": feedback,
            "verifier_decision": "retry" if retry else "accept",
            "timings": _timed(state, "verifiner", t0), "trace": events}

---
## Whiteboard box 4: **Fine-tuning** → `soap_generator_agent`

The **fine-tuned gpt-4o-mini** (trained on noisy transcript → SOAP pairs) writes the note. Unlike the straight
implementation, it now receives the **verified K-spans as a checklist** (the whiteboard arrow K-spans → SOAP),
grouped by existence class:

* Present → stated.
* Absent → written as pertinent negatives.
* Uncertain → only as the doctor's suspected impression.
* Hypothetical → only as conditional advice in P.

The note is always written in English, even when the conversation is Hindi-English.

On a **revision round**, the agent gets its previous draft plus the Reflection agent's feedback and fixes only
those points. A revision that loses the Subjective section is discarded, and the previous draft is kept.

In [ ]:
# ============================ 12. SOAP GENERATOR AGENT (fine-tuned model) ============================
def kspan_checklist(spans):
    """Verified K-spans grouped by existence class, in the whiteboard's (Type, Existence) form."""
    if not spans:
        return "(no verified key spans)"
    lines = []
    for ex in EXISTENCE_CLASSES:
        grp = [s for s in spans if s["existence"] == ex]
        if not grp:
            continue
        lines.append(f"{ex}:")
        for s in grp:
            info = []
            if s.get("severity") not in (None, "", "unspecified"):
                info.append(s["severity"])
            if s.get("temporality") == "historical":
                info.append("historical")
            if s.get("experiencer") not in (None, "", "patient"):
                info.append(f"experiencer={s['experiencer']}")
            extra = f" ({', '.join(info)})" if info else ""
            lines.append(f'  - ({s["type"]}, Existence: {ex}) "{s["text"]}" = {s.get("normalized_en") or s["text"]}{extra}')
    return "\n".join(lines)

def _soap_gen_system(sections=("S", "O", "A", "P"), max_lines=8):
    body = "\n".join(_SECTION_HELP[k] for k in sections)
    return ("You are the SOAP generator, a fine-tuned clinical scribe. Convert the doctor-patient conversation "
            "into a SOAP note.\n"
            "The conversation may be Hindi, English or Hindi-English code-mixed. ALWAYS write the note in clear "
            "English clinical language, translating the meaning faithfully.\n"
            "Use ONLY information stated in the conversation. The VERIFIED KEY SPANS are your checklist: every "
            "Present item must appear; every Absent item must appear as a pertinent negative; Uncertain items only "
            "as the doctor's suspected / possible impression; Hypothetical items only as conditional advice in P "
            "('if X, then Y'); items with experiencer=family only under Family history.\n\n"
            f"Output these sections, each starting on its own line with the single-letter label:\n{body}\n\n"
            "Rules:\n"
            f"- You MUST output every one of these labels: {', '.join(sections)}, each as 'X:' at the start of a line "
            "(not 'Subjective', not numbered headings).\n"
            f"- Keep each section concise: at most {max_lines} short lines.\n"
            "- Plain text only. No markdown, no bold, no numbered headings.\n"
            "- Do NOT write any disclaimer or 'AI Suggestion'; that is produced separately.")

_SOAP_LABEL_RE = re.compile(r"(?:^|\n)[ \t]*([SOAP])[ \t]*:[ \t]*")

def _parse_soap_ordered(raw):
    """Split a note at S:/O:/A:/P: labels that appear IN ORDER. A stray label inside a section
    (e.g. a vitals line "P: 88 bpm" in O, or "A: ..." inside S) no longer steals the next section:
    we keep the longest in-order chain of labels (ties -> the later occurrence ends the chain)."""
    txt = re.sub(r"\*\*([SOAP])\*\*\s*:", r"\1:", raw or "")
    txt = re.sub(r"\*\*([SOAP]):\*\*", r"\1:", txt)
    txt = re.sub(r"\*\*|#{1,4}[ \t]*", "", txt)
    ms = list(_SOAP_LABEL_RE.finditer(txt))
    if not ms:
        return {k: "" for k in "SOAP"}
    rank = [ "SOAP".index(m.group(1)) for m in ms]
    best_len, prev = [1] * len(ms), [-1] * len(ms)
    for i in range(len(ms)):
        for j in range(i):
            if rank[j] < rank[i] and best_len[j] + 1 > best_len[i]:
                best_len[i], prev[i] = best_len[j] + 1, j
    end = max(range(len(ms)), key=lambda i: (best_len[i], i))
    chain = []
    while end != -1:
        chain.append(ms[end])
        end = prev[end]
    chain.reverse()
    out = {k: "" for k in "SOAP"}
    for n, m in enumerate(chain):
        stop = chain[n + 1].start() if n + 1 < len(chain) else len(txt)
        out[m.group(1)] = txt[m.end():stop].strip()
    return out

# The fine-tuned model often writes full-word headings ("1. Subjective", "**Objective:**", "Assessment and Plan",
# "## Plan") instead of "S:". Convert heading LINES (or a heading word directly followed by ':') into single-letter
# labels. A sentence that merely starts with "Assessment of ..." is left alone.
_HEAD_WORDS = r"(subjective|objective|assessment\s*(?:and|&|/)\s*plan|assessment|impression|plan)"
_HEAD_PREFIX = r"^[ \t]*(?:#{1,6}[ \t]*)?(?:\*\*)?[ \t]*(?:\d+[.)]|[A-Ea-e][.)]|[ivx]+[.)])?[ \t]*(?:\*\*)?[ \t]*"
_HEAD_SUFFIX = r"(?:[ \t]+(?:section|note|findings))?(?:[ \t]*\([ \t]*[SOAP][ \t]*\))?[ \t]*(?:\*\*)?"
_HEAD_LINE_RE = re.compile(_HEAD_PREFIX + _HEAD_WORDS + _HEAD_SUFFIX + r"[ \t]*:?[ \t]*(?:\*\*)?[ \t]*$", re.I | re.M)
_HEAD_INLINE_RE = re.compile(_HEAD_PREFIX + _HEAD_WORDS + _HEAD_SUFFIX + r"[ \t]*:[ \t]*(?:\*\*)?[ \t]*(?=\S)", re.I | re.M)

def _heading_letter(word):
    w = re.sub(r"\s+", " ", word.lower())
    return "A" if w.startswith(("assessment", "impression")) else w[0].upper()

def _normalize_headings(raw):
    raw = _HEAD_LINE_RE.sub(lambda m: f"{_heading_letter(m.group(1))}:", raw)
    return _HEAD_INLINE_RE.sub(lambda m: f"{_heading_letter(m.group(1))}: ", raw)

def _parse_note(raw):
    # headings first: the inherited _normalize_note_labels deletes a bare "Assessment and Plan" title line
    got = _parse_soap_ordered(_normalize_note_labels(_normalize_headings(raw or "")))
    return {k: _clean_section(got.get(k, "") or "") for k in "SOAP"}

_REFORMAT_SYSTEM = ("Reformat the clinical note below into exactly four sections, each starting on its own line with "
                    "the label 'S:', 'O:', 'A:' or 'P:'. Move every sentence to the section it belongs to (patient-"
                    "reported history -> S; vitals and examination -> O; clinician's impression -> A; tests, medicines, "
                    "advice, follow-up -> P). Do NOT add, remove or reword content. Plain text only.")

def _flagged_sections(entry):
    """Sections the last reflection round complained about."""
    flagged = set(entry.get("empty_sections", []))
    for item in entry.get("hallucinations", []) + entry.get("missing", []):
        flagged |= set(re.findall(r"[SOAP]", str(item.get("section", ""))))
    for item in entry.get("misplaced", []):
        flagged |= set(re.findall(r"[SOAP]", f'{item.get("from", "")} {item.get("to", "")}'))
    return flagged

def soap_lines(transcript):
    return 8 if len(transcript) < 5000 else 14                # long consultations need longer sections

def soap_gen_user(transcript, spans, prev=None, feedback=""):
    user = f"CONVERSATION:\n{transcript}\n\nVERIFIED KEY SPANS (K):\n{kspan_checklist(spans)}"
    if prev:
        draft = "\n".join(f"{k}: {prev.get(k, '')}" for k in "SOAP")
        user += (f"\n\nYOUR PREVIOUS DRAFT:\n{draft}\n\n"
                 "REFLECTION FEEDBACK - fix every point below, keep everything else that is correct, "
                 f"and output the COMPLETE revised note:\n{feedback}")
    return user

def soap_gen_messages(transcript, spans, prev=None, feedback=""):
    """The exact prompt for a first draft (prev=None) or a revision (also used for LoRA training data)."""
    return [{"role": "system", "content": _soap_gen_system(max_lines=soap_lines(transcript))},
            {"role": "user", "content": soap_gen_user(transcript, spans, prev, feedback)}]

def soap_generator_agent(state):
    t0 = time.time()
    rnd = int(state.get("reflection_round") or 0)
    prev = state.get("soap") if rnd > 0 else None
    user = soap_gen_user(state["transcript"], state.get("verified_spans") or [], prev, state.get("reflection_feedback", ""))
    lines = soap_lines(state["transcript"])
    # first draft = "generator" (fine-tuned model / generator adapter); revisions = REVISION_ROLE ("reviser":
    # gpt-4.1-mini or the reviser adapter, which follow edit feedback far more reliably)
    role = REVISION_ROLE if prev else "generator"
    raw = LLM.chat(soap_gen_messages(state["transcript"], state.get("verified_spans") or [], prev,
                                     state.get("reflection_feedback", "")),
                   role=role, max_tokens=SOAP_MAX_TOKENS, temperature=0.2)
    sec = _parse_note(raw)
    fmt = ""
    if sum(bool(sec[k].strip()) for k in "SOAP") <= 1 and len(raw.split()) > 40:
        # the note has content but no usable section labels -> let the base model re-label it (content unchanged)
        fixed = _parse_note(LLM.chat([{"role": "system", "content": _REFORMAT_SYSTEM},
                                      {"role": "user", "content": raw}], role="agent", max_tokens=SOAP_MAX_TOKENS,
                                     temperature=0.0))
        if sum(bool(fixed[k].strip()) for k in "SOAP") > sum(bool(sec[k].strip()) for k in "SOAP"):
            sec, fmt = fixed, " [re-labelled unlabelled output]"
    words = lambda d: sum(len(d[k].split()) for k in "SOAP")
    if not sec["P"].strip() and len(sec["A"].split()) > 25:
        # plan written inside a combined "Assessment and Plan" block -> re-label (content unchanged)
        fixed = _parse_note(LLM.chat([{"role": "system", "content": _REFORMAT_SYSTEM},
                                      {"role": "user", "content": "\n".join(f"{k}: {sec[k]}" for k in "SOAP")}],
                                     role="agent", max_tokens=SOAP_MAX_TOKENS, temperature=0.0))
        if fixed["P"].strip() and fixed["S"].strip() and abs(words(fixed) - words(sec)) <= 0.2 * max(1, words(sec)):
            sec, fmt = fixed, fmt + " [plan split out of A]"
    missing = [k for k in "SOAP" if not sec[k].strip()]
    if missing:                                       # targeted regeneration of the sections still missing
        got = _parse_note(LLM.chat([{"role": "system", "content": _soap_gen_system(tuple(missing), lines)},
                                    {"role": "user", "content": user}], role=role, max_tokens=1500))
        for k in missing:
            if got[k].strip():
                sec[k] = got[k]
    action = ("revised draft" if prev else "first draft") + fmt
    if prev:
        # targeted revision: only sections the reflection flagged may change; the rest stay as they were
        flagged = _flagged_sections((state.get("reflection_log") or [{}])[-1])
        if flagged:
            for k in "SOAP":
                if k not in flagged and prev.get(k, "").strip():
                    sec[k] = prev[k]
            action += f" (changed only {''.join(k for k in 'SOAP' if k in flagged)})"
    if not sec["S"].strip():
        if prev and prev.get("S", "").strip():
            sec, action = dict(prev), "revision had no Subjective -> kept previous draft"
        else:
            sec["S"] = _clean_section(re.sub(r"^\s*S\s*:\s*", "", (raw or "").strip()))[:3000]
            action += " (unlabelled output used as S)"
    filled = [k for k in "SOAP" if sec[k].strip()]
    return {"soap": sec, "timings": _timed(state, "soap_generator", t0),
            "trace": [_step("SOAP-Generator", f"round {rnd}: {action}", model=LLM.models[role],
                            sections="".join(filled), regenerated="".join(missing) or "-")]}

---
## Whiteboard box 5: **Reflection** → `reflection_agent`

Each round reviews the current draft in two parts:

**Auto-repair (tools, applied directly):** split a combined *Assessment and Plan* block, drop plan items the
conversation never mentions (`_verify_plan_grounded`), remove ungrounded differentials from A
(`_strip_speculation`), and normalise empty sections. Grounding is checked against the transcript **plus the
English meaning of every verified span**, so an English note can be checked against Hindi speech.

**① Hallucination check:** drug names that were never mentioned; **assertion conflicts** (a span that is
*Absent* in the K-spans but written as present); unsupported claims found by an LLM critic.

**② S, O, A, P check:** every section present; every verified K-span (Present / Absent / Uncertain) covered;
vitals in O, orders in P; misplaced content found by the critic.

**Agent decision:** if unresolved issues remain and `reflection_round < MAX_REFLECTION_ROUNDS`, the feedback goes
back to the SOAP generator. Otherwise the agent accepts. Each draft gets a score
(2 × hallucinations + missing + misplaced), and the **best-scoring draft is kept**, so a revision can never make
the final note worse.

**③ Addition** is handled by the next agent.

In [ ]:
# ============================ 13. REFLECTION AGENT (hallucination check + S,O,A,P check) ============================
REFLECTION_CRITIC_SYSTEM = """You are the Reflection agent reviewing a DRAFT SOAP note (English) written from a doctor-patient
CONVERSATION that may be Hindi-English code-mixed. You also get the VERIFIED KEY SPANS extracted from the conversation.
Check the draft and return ONLY a JSON object:
{"unsupported": [{"section": "S|O|A|P", "claim": "<text from the draft>", "reason": "..."}],
 "missing":     [{"section": "S|O|A|P", "fact": "<fact from the conversation>"}],
 "misplaced":   [{"claim": "<text from the draft>", "from": "S|O|A|P", "to": "S|O|A|P"}]}
unsupported = anything the conversation does not support: invented findings, diagnoses or differentials the
              doctor did not give, orders the doctor did not give, or a WRONG ASSERTION (the note says the patient has
              something the patient denied, or states as certain what the doctor only suspected).
missing     = clinically relevant facts in the conversation / key spans that the note omits (symptoms, pertinent
              negatives, history, medications, allergies, vitals, the doctor's impression, orders, follow-up).
misplaced   = content in the wrong section (patient-reported history -> S; vitals and examination -> O;
              doctor's impression -> A; tests, medicines, advice, follow-up -> P).
Use empty lists when there is nothing to report. Judge meaning, not wording or language; do not nitpick."""

_NEG_IN_NOTE = re.compile(r"\b(no|not|denies|denied|denying|without|negative|negatives|absent|none|never|nil)\b", re.I)

def _is_empty_section(txt):
    """Only a SHORT 'nothing here' line counts as empty. The inherited regex alone also matched real content that
    merely starts with 'No ...' (e.g. 'No change to metformin; start amlodipine 5 mg ...') and wiped the plan."""
    t = (txt or "").strip()
    return not t or (len(t.split()) <= 12 and bool(_EMPTY_SECTION_RE.match(t)))

def _note_sentences(text):
    return [p.strip() for p in re.split(r"(?<=[.!?;])\s+|\n+", text) if p.strip()]

def _format_feedback(hall, missing, misplaced, empty):
    out = []
    if empty:
        out.append("0. EMPTY SECTIONS - write these sections: " + ", ".join(empty))
    if hall:
        out.append("1. HALLUCINATION CHECK - remove or correct these unsupported statements:")
        out += [f'   - [{h.get("section", "?")}] "{str(h.get("claim", ""))[:160]}" -> {h.get("reason", "")}' for h in hall[:12]]
    if missing:
        out.append("2. S,O,A,P CHECK - add these facts from the conversation to the stated section:")
        out += [f'   - [{m.get("section", "S")}] {str(m.get("fact", ""))[:160]}' for m in missing[:15]]
    if misplaced:
        out.append("   Move these to the correct section:")
        out += [f'   - "{str(m.get("claim", ""))[:120]}": {m.get("from", "?")} -> {m.get("to", "?")}' for m in misplaced[:10]]
    return "\n".join(out)

def critic_messages(src, spans, sec):
    """The exact prompt the Reflection critic sends (also used for LoRA training data)."""
    note_txt = "\n".join(f"{k}: {sec[k]}" for k in "SOAP")
    return [{"role": "system", "content": REFLECTION_CRITIC_SYSTEM},
            {"role": "user", "content": f"CONVERSATION:\n{src}\n\nVERIFIED KEY SPANS:\n{kspan_checklist(spans)}"
                                        f"\n\nDRAFT SOAP NOTE:\n{note_txt}\n\nJSON object:"}]

def reflection_agent(state):
    t0 = time.time()
    rnd = int(state.get("reflection_round") or 0)
    src, spans = state["transcript"], state.get("verified_spans") or []
    ground = grounding_text(src, spans)
    sec = {k: (state.get("soap") or {}).get(k, "") or "" for k in "SOAP"}
    auto_fixed, hall, missing, misplaced = [], [], [], []

    # ---------------- auto-repair (deterministic tools) ----------------
    a, p = _split_ap(sec["A"], sec["P"])
    if (a, p) != (sec["A"], sec["P"]):
        sec["A"], sec["P"] = a, p
        auto_fixed.append("moved the plan out of a combined Assessment-and-Plan block")
    if not _is_empty_section(sec["P"]):
        p2, dropped = _verify_plan_grounded(sec["P"], ground)
        if dropped:
            sec["P"] = p2
            auto_fixed += [f'dropped plan item not in the conversation: "{d[:90]}"' for d in dropped]
    if sec["A"].strip():
        a2 = _strip_speculation(sec["A"], ground)
        if a2 != sec["A"]:
            auto_fixed.append(f'removed ungrounded speculation from A: "{sec["A"][:90]}"')
            sec["A"] = a2
    for k in ("O", "A", "P"):
        if _is_empty_section(sec[k]):
            sec[k] = _PLACEHOLDER[k]

    note_all = "\n".join(sec[k] for k in "SOAP")
    nl, gl = note_all.lower(), ground.lower()

    # ---------------- 1. hallucination check ----------------
    for drug in sorted({m.group().lower() for m in _KNOWN_DRUGS.finditer(note_all)}):
        if drug not in gl:
            hall.append({"section": "P", "claim": drug, "reason": "drug never mentioned in the conversation", "source": "tool"})
    sents = _note_sentences(note_all)
    for s in spans:
        if s["existence"] != "Absent" or s["type"] not in ("Symptom", "Disease", "Allergy"):
            continue
        terms = span_terms(s)
        hits = [x for x in sents if any(t in x.lower() for t in terms)]
        if hits and not any(_NEG_IN_NOTE.search(x) for x in hits):
            hall.append({"section": "S", "claim": hits[0][:160], "source": "tool",
                         "reason": f'assertion conflict: "{s["text"]}" was DENIED in the conversation (Absent) but the note states it as present'})

    # ---------------- 2. S,O,A,P check ----------------
    empty = [k for k in "SOAP" if not sec[k].strip()]
    for s in spans:
        if s["type"] in COVERAGE_TYPES and s["existence"] in ("Present", "Absent", "Uncertain") and not is_mentioned(s, nl):
            where = ("O" if s["type"] == "Vital" else
                     "P" if s["type"] in ("Test", "Procedure") and s.get("speaker", "").lower() == "doctor" else
                     "A" if s["existence"] == "Uncertain" else "S")
            label = {"Absent": "pertinent negative", "Uncertain": "suspected"}.get(s["existence"], "present")
            fam = ", family history" if s.get("experiencer") == "family" else ""
            missing.append({"section": where, "source": "tool",
                            "fact": f'{s.get("normalized_en") or s["text"]} ({label}{fam}; said as "{s["text"]}")'})
    o_l = sec["O"].lower()
    for s in spans:
        if s["type"] == "Vital" and s["existence"] == "Present" and is_mentioned(s, nl) and not is_mentioned(s, o_l):
            misplaced.append({"claim": s.get("normalized_en") or s["text"], "from": "S/A/P", "to": "O", "source": "tool"})

    # ---------------- LLM critic (covers both checks) ----------------
    try:
        d = LLM.json(critic_messages(src, spans, sec), role="critic", max_tokens=1500, temperature=0.0)
    except Exception as e:
        d = {}
        print(f"  [Reflection] critic unavailable: {str(e)[:70]}")
    have = {str(m.get("fact", "")).lower()[:40] for m in missing}
    for x in d.get("unsupported") or []:
        if isinstance(x, dict) and str(x.get("claim", "")).strip():
            hall.append({**x, "source": "critic"})
    for x in d.get("missing") or []:
        if isinstance(x, dict) and str(x.get("fact", "")).strip() and str(x["fact"]).lower()[:40] not in have:
            missing.append({**x, "source": "critic"})
    for x in d.get("misplaced") or []:
        if isinstance(x, dict) and str(x.get("claim", "")).strip() and x.get("from") != x.get("to"):
            misplaced.append({**x, "source": "critic"})

    # ---------------- score, keep best draft, decide ----------------
    score = 2 * len(hall) + len(missing) + len(misplaced) + 5 * len(empty)
    best_score = state.get("best_score")
    improved = best_score is None or score <= best_score
    issues = {"hallucinations": hall, "missing": missing, "misplaced": misplaced, "empty_sections": empty,
              "auto_fixed": auto_fixed, "round": rnd, "score": score}
    # revise only for real problems: any hallucination / empty section, or more than REFLECTION_ACCEPT_SCORE
    # minor issues; and stop as soon as a revision no longer improves the score (critics always find nits)
    prev = (state.get("reflection_log") or [{}])[-1].get("score")
    real_problem = bool(hall) or bool(empty) or score > REFLECTION_ACCEPT_SCORE
    stalled = prev is not None and score >= prev
    decision = "revise" if real_problem and not stalled and rnd < MAX_REFLECTION_ROUNDS else "accept"
    entry = {**issues, "decision": decision, "kept_as_best": improved}
    ev = _step("Reflection", f"round {rnd}: score {score} -> {decision.upper()}"
               + ("" if improved else " (earlier draft stays best)"),
               hallucinations=len(hall), missing=len(missing), misplaced=len(misplaced), auto_fixed=len(auto_fixed))
    return {"soap": sec,
            "best_soap": sec if improved else state.get("best_soap"),
            "best_score": score if improved else best_score,
            "best_issues": issues if improved else state.get("best_issues"),
            "reflection_round": rnd + 1 if decision == "revise" else rnd,
            "reflection_feedback": _format_feedback(hall, missing, misplaced, empty),
            "reflection_decision": decision,
            "reflection_log": list(state.get("reflection_log") or []) + [entry],
            "timings": _timed(state, "reflection", t0), "trace": [ev]}

---
## Whiteboard box 5-③: **Addition → standard codes** → `addition_agent`, then `evaluation_agent` and `report_agent`

* **Standard codes** (tools): SNOMED CT for every non-denied span; ICD-10-CM for diagnoses only (Present
  symptoms / diseases / allergies of the patient, plus *family history of …* Z-codes); CPT for tests and
  procedures that were ordered. Deterministic clinical inference adds suspected syndromes (e.g. gastroenteritis
  from diarrhoea + vomiting + abdominal pain), marked `[inferred]`.
* **AI Suggestion**: standard-of-care steps the doctor did *not* mention, placed after a `[Disclaimer]`,
  clearly labelled, and **never scored**.
* **Evaluation agent**:
  * independent LLM judge with an anchored 4-axis rubric at temperature 0;
  * K-span coverage and the hallucinations left in the final note;
  * cross-lingual grounding using a multilingual embedder;
  * ROUGE / BLEU / BERTScore when a reference SOAP is given.
* **Report agent**: one report (transcript, note, K-spans, VerifiNER rejections, reflection log, codes, scores,
  agent trace), also saved as JSON.

In [ ]:
# ============================ 14. ADDITION AGENT (standard codes + AI suggestion) ============================
def addition_agent(state):
    t0 = time.time()
    sec = dict(state.get("best_soap") or state.get("soap") or {})
    src, spans = state["transcript"], state.get("verified_spans") or []
    coded = []
    for s in spans:
        e = {**s, "snomed": None, "icd10_cm": None}
        term = (e.get("normalized_en") or e["text"]).strip()
        term_n = _FAST_NORMALIZE.get(term.lower(), term)
        if e["existence"] != "Absent" and e["type"] not in ("Dosage", "Anatomy"):
            e["snomed"] = get_snomed(term_n, e["type"]) or (get_snomed(term, e["type"]) if term_n != term else None)
        if e["type"] in ("Symptom", "Disease", "Allergy") and e["existence"] == "Present" and not _is_vital(term):
            if e.get("experiencer") == "family":
                fh = f"family history of {term_n}"
                sn = get_snomed(fh)
                if sn and "family" in sn["term"].lower():
                    e["snomed"] = sn
                e["icd10_cm"] = get_icd10(fh, "Disease", sn if sn and "family" in sn["term"].lower() else None)
            elif e.get("experiencer", "patient") == "patient":
                e["icd10_cm"] = get_icd10(term_n, e["type"], e["snomed"])
        coded.append(e)

    # deterministic clinical inference (reused rules) -> suspected syndromes, clearly marked
    adapted = [{"status": "Confirmed" if c["existence"] == "Present" and c.get("experiencer") == "patient" else "Other",
                "text": (c.get("normalized_en") or c["text"]).lower(), "snomed": c.get("snomed")} for c in coded]
    inferred = _run_clinical_inference(adapted)
    for inf in inferred:
        coded.append({"text": inf["text"], "type": "Disease", "existence": "Uncertain",
                      "normalized_en": inf["snomed"]["term"], "severity": "unspecified", "temporality": "current",
                      "experiencer": "patient", "speaker": "(inferred)", "offset": None, "_inferred": True,
                      "snomed": inf["snomed"], "icd10_cm": inf["icd10_cm"]})

    cpt, seen = [], set()
    for e in coded:
        if e["type"] in ("Test", "Procedure") and e["existence"] == "Present":
            term = e.get("normalized_en") or e["text"]
            c = get_cpt(term) or (get_cpt(e["text"]) if term != e["text"] else None)
            if c and c["code"] not in seen:
                seen.add(c["code"])
                cpt.append({**c, "source_span": e["text"]})

    note_so_far = "".join(f"{k}: {sec.get(k, '')}\n" for k in "SOAP")
    try:
        ai = LLM.chat([{"role": "system", "content": _AI_SUGGESTION_PROMPT},
                       {"role": "user", "content": f"TRANSCRIPT:\n{src}\n\nSOAP NOTE:\n{note_so_far}"}],
                      role="agent", max_tokens=400, temperature=0.2).strip()
        ai = re.sub(r"^\s*(\[?\s*Disclaimer\s*\]?|AI Suggestion)\s*:?\s*", "", ai, flags=re.I).strip()
    except Exception as e:
        ai = ""
        print(f"  [Addition] AI suggestion skipped: {str(e)[:70]}")
    n_sn = sum(1 for e in coded if e.get("snomed"))
    n_icd = sum(1 for e in coded if (e.get("icd10_cm") or {}).get("code"))
    return {"coded_spans": coded, "cpt_codes": cpt, "ai_suggestion": ai, "final_note": _assemble(sec, ai, state.get("verified_spans") or []),
            "timings": _timed(state, "addition", t0),
            "trace": [_step("Addition", "standard codes + AI suggestion", snomed=n_sn, icd10=n_icd,
                            cpt=len(cpt), inferred=len(inferred))]}


# ============================ 15. EVALUATION AGENT ============================
def kspan_coverage(note, spans):
    rel = [s for s in spans if s["type"] in COVERAGE_TYPES and s["existence"] in ("Present", "Absent", "Uncertain")]
    nl = note.lower()
    miss = [s["text"] for s in rel if not is_mentioned(s, nl)]
    return {"coverage": round((len(rel) - len(miss)) / max(1, len(rel)), 4),
            "found": len(rel) - len(miss), "total": len(rel), "missing": miss}

def cross_lingual_grounding(note, transcript, thresh=0.5):
    """Share of note sentences whose closest conversation turn is semantically similar (multilingual encoder)."""
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", note)
             if len(s.strip()) > 12 and not re.fullmatch(r"[SOAP]\s*:", s.strip())]
    turns = [t.strip() for t in re.split(r"\n+|(?<=[.!?।])\s+", transcript) if len(t.strip()) > 5]
    if not sents or not turns:
        return {"grounded_ratio": 1.0, "n_sentences": len(sents), "note_transcript_cosine": None}
    se = ml_embedder.encode(sents, convert_to_tensor=True)
    te = ml_embedder.encode(turns, convert_to_tensor=True)
    sims = util.cos_sim(se, te)
    grounded = sum(1 for i in range(len(sents)) if float(sims[i].max()) >= thresh)
    doc = float(util.cos_sim(ml_embedder.encode([note], convert_to_tensor=True),
                             ml_embedder.encode([transcript], convert_to_tensor=True)).max())
    return {"grounded_ratio": round(grounded / len(sents), 3), "n_sentences": len(sents),
            "note_transcript_cosine": round(doc, 4)}

_BERT_SCORER = {}
def true_bertscore(cand, ref):
    """Real BERTScore (Zhang et al. 2020): roberta-large token matching, rescaled with the official baseline."""
    if "s" not in _BERT_SCORER:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score"])
        from bert_score import BERTScorer
        _BERT_SCORER["s"] = BERTScorer(lang="en", rescale_with_baseline=True)
    p, r, f = _BERT_SCORER["s"].score([cand], [ref])
    return {"precision": round(float(p[0]), 4), "recall": round(float(r[0]), 4), "f1": round(float(f[0]), 4)}

_JUDGE_AXES = ["quality_of_soap", "adequacy_wrt_transcript", "faithfulness", "alignment_score"]

def llm_judge(transcript, soap_text):
    """Anchored rubric, analysis-then-score, temperature 0, independent model (judge role)."""
    prompt = _FAIR_JUDGE_PROMPT.format(transcript=transcript[:40000], soap=soap_text[:8000])   # full dialogue
    msg = [{"role": "system", "content": "You are a fair, consistent clinical documentation reviewer. The "
            "conversation may be Hindi-English code-mixed while the note is in English: judge meaning, not language."},
           {"role": "user", "content": prompt}]
    d = _parse_json(LLM.chat(msg, role="judge", max_tokens=600, temperature=0.0), expect="dict") or {}
    scores = {k: int(d[k]) if isinstance(d.get(k), (int, float)) and 1 <= d[k] <= 5 else 3 for k in _JUDGE_AXES}
    scores["composite"] = round(sum(scores[k] for k in _JUDGE_AXES) / len(_JUDGE_AXES), 2)
    return scores

def evaluation_agent(state):
    t0 = time.time()
    note = extractive_soap_only(state.get("final_note", ""))      # AI Suggestion is never scored
    t, spans = state["transcript"], state.get("verified_spans") or []
    best = state.get("best_issues") or {}
    em = {"kspan_stats": {"candidates": len(state.get("candidate_spans") or []), "verified": len(spans),
                          "rejected": len(state.get("rejected_spans") or []),
                          "by_type": dict(Counter(s["type"] for s in spans)),
                          "by_existence": dict(Counter(s["existence"] for s in spans))},
          "keyword_coverage": kspan_coverage(note, spans),
          "hallucination": {"count": len(best.get("hallucinations", [])),
                            "items": [str(h.get("claim", ""))[:80] for h in best.get("hallucinations", [])]},
          "reflection": {"rounds": len(state.get("reflection_log") or []),
                         "auto_fixed": sum(len(e.get("auto_fixed", [])) for e in state.get("reflection_log") or []),
                         "hallucinations_caught": sum(len(e.get("hallucinations", [])) for e in state.get("reflection_log") or []),
                         "final_score": state.get("best_score")}}
    try:
        em["grounding"] = cross_lingual_grounding(note, t)
    except Exception as e:
        print(f"  [Evaluation] grounding skipped: {str(e)[:60]}")
    ref = state.get("reference_soap")
    if ref:
        em["rouge1"] = rouge_n(note, ref, n=1)
        em["rougeL"] = rouge_l(note, ref)
        em["bleu"] = bleu_score(note, ref)
        em["semsim_sent"] = bertscore_sentence(note, ref)   # sentence-embedding similarity (NOT BERTScore)
        if TRUE_BERTSCORE:
            try:
                em["bertscore"] = true_bertscore(note, ref)
            except Exception as e:
                print(f"  [Evaluation] BERTScore skipped: {str(e)[:60]}")
    try:
        em["llm_judge"] = llm_judge(t, note)
    except Exception as e:
        print(f"  [Evaluation] judge skipped: {str(e)[:60]}")
    j = em.get("llm_judge", {})
    return {"eval_metrics": em, "timings": _timed(state, "evaluation", t0),
            "trace": [_step("Evaluation", "scored the extractive S/O/A/P", judge=j.get("composite", "-"),
                            coverage=f'{em["keyword_coverage"]["coverage"]:.0%}',
                            hallucinations=em["hallucination"]["count"])]}


# ============================ 16. REPORT AGENT ============================
_RULE, _THIN = "=" * 90, "-" * 90

def _table(headers, rows, widths):
    out = ["".join(str(h).ljust(w) for h, w in zip(headers, widths)), _THIN]
    for r in rows:
        out.append("".join(str(c)[:w - 1].ljust(w) for c, w in zip(r, widths)))
    return out

def format_eval_metrics(em):
    j, g = em.get("llm_judge", {}), em.get("grounding", {})
    kc, h, ks, rf = em.get("keyword_coverage", {}), em.get("hallucination", {}), em.get("kspan_stats", {}), em.get("reflection", {})
    out = [_RULE, "EVALUATION  (extractive S/O/A/P only; the AI Suggestion is excluded)", _RULE,
           "LLM-as-judge (independent model, anchored rubric, temperature 0, 1-5):",
           f"   Quality of SOAP          : {j.get('quality_of_soap', '?')}",
           f"   Adequacy wrt transcript  : {j.get('adequacy_wrt_transcript', '?')}",
           f"   Faithfulness             : {j.get('faithfulness', '?')}",
           f"   Alignment score          : {j.get('alignment_score', '?')}",
           f"   Composite                : {j.get('composite', '?')} / 5",
           "Agentic pipeline checks:",
           f"   K-spans (cand/verified/rej): {ks.get('candidates', '?')} / {ks.get('verified', '?')} / {ks.get('rejected', '?')}",
           f"   K-span coverage in note  : {kc.get('coverage', 0):.0%}  ({kc.get('found', 0)}/{kc.get('total', 0)})"
           + (f"  missing: {kc.get('missing', [])[:6]}" if kc.get("missing") else ""),
           f"   Hallucinations left      : {h.get('count', 0)}  {h.get('items', [])[:4]}",
           f"   Reflection rounds        : {rf.get('rounds', '?')}  (hallucinations caught {rf.get('hallucinations_caught', 0)}, "
           f"auto-fixes {rf.get('auto_fixed', 0)}, final score {rf.get('final_score', '?')})",
           f"   Cross-lingual grounding  : {g.get('grounded_ratio', '?')}  ({g.get('n_sentences', '?')} sentences)"
           f"   note~transcript cosine {g.get('note_transcript_cosine', '?')}"]
    if em.get("rouge1"):
        out += [f"   ROUGE-1 F1 / ROUGE-L F1  : {em['rouge1']['f1']:.3f} / {em['rougeL']['f1']:.3f}",
                f"   BLEU                     : {em['bleu']['bleu']:.3f}",
                f"   BERTScore F1 (rescaled)  : {em.get('bertscore', {}).get('f1', 'n/a')}",
                f"   Sentence-emb. similarity : {em['semsim_sent']['f1']:.3f}  (not BERTScore)"]
    return "\n".join(out + [_RULE])

def build_report(r):
    spans, coded = r.get("verified_spans") or [], r.get("coded_spans") or []
    lp = r.get("language_profile") or {}
    L = ["AGENTIC SOAP PIPELINE - REPORT", f"Generated: {datetime.now():%Y-%m-%d %H:%M:%S}",
         "DISCLAIMER: AI-generated. Requires clinician review.",
         f"Language: {lp.get('label', '?')}  |  models: " + ", ".join(f"{k}={v}" for k, v in LLM.models.items()), ""]
    L += [_RULE, "TRANSCRIPT (diarized)", _RULE, r.get("transcript", ""), ""]
    L += [_RULE, "SOAP NOTE", _RULE, r.get("final_note", ""), ""]
    L += [_RULE, f"K-SPANS  (verified by VerifiNER: {len(spans)})   format: (Type, Existence: ...) \"span\" -> English", _RULE]
    rows = []
    for i, s in enumerate(spans, 1):
        v = s.get("verification", {})
        rows.append([f"{i}.", f'({s["type"]}, Existence: {s["existence"]})', s["text"], s.get("normalized_en", ""),
                     s.get("severity", ""), s.get("experiencer", ""),
                     f'{v.get("type_votes", "")} {v.get("existence_votes", "")}',
                     (s.get("_span_fix") or v.get("was") or "")])
    L += _table(["#", "(Type, Existence)", "Span (as spoken)", "English", "Severity", "Who", "Votes t/e", "Fix"],
                rows, [4, 34, 24, 24, 12, 9, 12, 26]) if rows else ["   none"]
    rej = r.get("rejected_spans") or []
    L += ["", _RULE, f"VerifiNER REJECTIONS ({len(rej)})", _RULE]
    L += [f'   "{x.get("text", "")}" [{x.get("type", "")}] -> {x.get("_reason", "")}' for x in rej] or ["   none"]
    L += ["", _RULE, "REFLECTION LOG", _RULE]
    for e in r.get("reflection_log") or []:
        L.append(f"   round {e['round']}: score {e['score']} -> {e['decision']}"
                 f"{'  (best so far)' if e.get('kept_as_best') else ''}")
        for a in e.get("auto_fixed", []):
            L.append(f"      auto-fix     : {a}")
        for h in e.get("hallucinations", []):
            L.append(f"      hallucination: [{h.get('section', '?')}] {str(h.get('claim', ''))[:80]} ({h.get('reason', '')[:70]})")
        for m in e.get("missing", []):
            L.append(f"      missing      : [{m.get('section', '?')}] {str(m.get('fact', ''))[:110]}")
        for m in e.get("misplaced", []):
            L.append(f"      misplaced    : {str(m.get('claim', ''))[:80]} ({m.get('from', '?')} -> {m.get('to', '?')})")
    srows = [[(e.get("text") or "")[:30] + (" [inferred]" if e.get("_inferred") else ""), e["type"], e["existence"],
              e["snomed"]["concept_id"], e["snomed"]["term"]] for e in coded if e.get("snomed")]
    L += ["", _RULE, "SNOMED CT CONCEPTS", _RULE]
    L += _table(["Finding", "Type", "Existence", "SNOMED CT", "Concept"], srows, [34, 11, 14, 13, 30]) if srows else ["   none mapped"]
    irows = [[(e.get("text") or "")[:30] + (" [inferred]" if e.get("_inferred") else ""),
              "family hx" if e.get("experiencer") == "family" else e["existence"],
              e["icd10_cm"]["code"], (e["icd10_cm"].get("description") or "")[:40]]
             for e in coded if (e.get("icd10_cm") or {}).get("code")]
    L += ["", _RULE, "ICD-10-CM DIAGNOSIS CODES (diagnoses only; no vitals, drugs or procedures)", _RULE]
    L += _table(["Finding", "Assertion", "ICD-10", "Description"], irows, [34, 14, 10, 42]) if irows else ["   none"]
    crows = [[c["code"], c.get("description", "")[:58], c.get("source_span", "")] for c in r.get("cpt_codes") or []]
    L += ["", _RULE, "CPT PROCEDURE CODES", _RULE]
    L += _table(["CPT", "Description", "From span"], crows, [9, 60, 21]) if crows else ["   none"]
    L += ["", format_eval_metrics(r.get("eval_metrics") or {}), "", _RULE, "AGENT TRACE", _RULE]
    L += [f"   {i:>2}. {e['time']}  {e['agent']:<15} {e['action']}" for i, e in enumerate(r.get("trace") or [], 1)]
    tm = r.get("timings") or {}
    L += ["   time per agent (s): " + ", ".join(f"{k}={v}" for k, v in tm.items()),
          "   LLM calls by role : " + ", ".join(f"{k}={v}" for k, v in LLM.calls.items()), _RULE]
    return "\n".join(L)

def report_agent(state):
    t0 = time.time()
    rep = build_report(state)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = os.path.join(OUTPUT_DIR, f"run_{stamp}.json")
    keep = ["audio_path", "language_profile", "transcript", "final_note", "ai_suggestion", "verified_spans",
            "rejected_spans", "reflection_log", "coded_spans", "cpt_codes", "eval_metrics", "timings", "trace"]
    with open(path, "w", encoding="utf-8") as f:
        json.dump({k: state.get(k) for k in keep}, f, ensure_ascii=False, indent=1, default=str)
    with open(path.replace(".json", "_report.txt"), "w", encoding="utf-8") as f:
        f.write(rep)
    return {"report": rep, "timings": _timed(state, "report", t0),
            "trace": [_step("Report", f"saved {os.path.basename(path)}")]}

In [ ]:
# ============================ 17. BUILD THE AGENT GRAPH (LangGraph) ============================
from langgraph.graph import StateGraph, START, END

def route_start(state):
    """Audio in -> ASR first; a pasted transcript goes straight to diarization."""
    return "diarization_agent" if (state.get("raw_transcript") or "").strip() else "asr_agent"

def route_after_verifiner(state):
    return "kspan_agent" if state.get("verifier_decision") == "retry" else "soap_generator_agent"

def route_after_reflection(state):
    return "soap_generator_agent" if state.get("reflection_decision") == "revise" else "addition_agent"

graph = StateGraph(SOAPState)
graph.add_node("asr_agent", asr_agent)                        # Transcript
graph.add_node("diarization_agent", diarization_agent)        # Transcript
graph.add_node("kspan_agent", kspan_agent)                    # K-Spans + info
graph.add_node("verifiner_agent", verifiner_agent)            # K-Spans refinement (VerifiNER)
graph.add_node("soap_generator_agent", soap_generator_agent)  # Fine-tuned generator
graph.add_node("reflection_agent", reflection_agent)          # Reflection 1 + 2
graph.add_node("addition_agent", addition_agent)              # Reflection 3 (Addition -> codes)
graph.add_node("evaluation_agent", evaluation_agent)
graph.add_node("report_agent", report_agent)

graph.add_conditional_edges(START, route_start, ["asr_agent", "diarization_agent"])
graph.add_edge("asr_agent", "diarization_agent")
graph.add_edge("diarization_agent", "kspan_agent")
graph.add_edge("kspan_agent", "verifiner_agent")
graph.add_conditional_edges("verifiner_agent", route_after_verifiner, ["kspan_agent", "soap_generator_agent"])
graph.add_edge("soap_generator_agent", "reflection_agent")
graph.add_conditional_edges("reflection_agent", route_after_reflection, ["soap_generator_agent", "addition_agent"])
graph.add_edge("addition_agent", "evaluation_agent")
graph.add_edge("evaluation_agent", "report_agent")
graph.add_edge("report_agent", END)
soap_agent_app = graph.compile()


def run_soap_agent(audio_path=None, transcript=None, reference_soap=None):
    """Run the full agent graph on an audio file OR a transcript string. Returns the final state."""
    if not audio_path and not (transcript or "").strip():
        raise ValueError("Give an audio_path or a transcript.")
    LLM.calls.clear()
    t0 = time.time()
    print("=" * 90 + f"\nAGENTIC SOAP RUN  <- {audio_path or 'transcript text'}\n" + "=" * 90)
    init = {"audio_path": audio_path, "raw_transcript": (transcript or "").strip() if not audio_path else "",
            "reference_soap": reference_soap, "kspan_attempts": 0, "reflection_round": 0,
            "reflection_log": [], "timings": {}, "trace": []}
    final = soap_agent_app.invoke(init, config={"recursion_limit": 60})
    print(f"=== done in {time.time() - t0:.1f}s | LLM calls: {dict(LLM.calls)} ===")
    return final

# ---- show the graph ----
print(soap_agent_app.get_graph().draw_mermaid())
try:
    from IPython.display import Image, display
    display(Image(soap_agent_app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"(graph image needs internet for mermaid.ink: {str(e)[:60]}) - the mermaid text above shows the same graph")

---
## Run it

1. **Demo (no audio needed)**: an unlabelled Hindi-English consultation, like raw Whisper output. It exercises
   every agent: diarization, Present / Absent / Uncertain / Hypothetical spans, family history, an allergy,
   vitals, tests, a drug + dosage, reflection and coding.
2. **One audio file**: set `AUDIO_PATH`, or leave it `None` to use the first audio file found under `/kaggle/input`.
3. **Batch**: a whole folder of audio (`.wav/.mp3/...`) or transcripts (`.txt`), with optional reference SOAP notes
   (`<same name>.txt` in `reference_dir`). Produces a CSV summary for the dataset.
4. **BeTraC 2026 benchmark**: downloads the public dev split from Hugging Face and scores every note against the
   gold SOAP note (plus Whisper WER in audio mode).

In [ ]:
# ============================ 18. DEMO: Hindi-English transcript (no audio needed) ============================
DEMO_TRANSCRIPT = (
    "Namaste doctor sahab. Namaste, bataiye kya problem hai? Doctor mujhe teen din se bukhar hai aur sir mein "
    "bahut dard ho raha hai. Bukhar kitna tha, aapne check kiya? Haan kal raat 102 tha. Khansi ya saans lene mein "
    "koi problem? Nahi, khansi nahi hai, saans theek hai. Ulti ya dast? Kal ek baar ulti hui thi. Dast nahi hai. "
    "Body pain? Haan, poore body mein dard hai aur bahut weakness lag rahi hai. Koi dawai li? Maine Crocin li thi, "
    "thoda aaram mila. Aapko sugar ya BP ki problem hai? Nahi, but mere papa ko diabetes hai. Koi allergy? "
    "Penicillin se allergy hai. Theek hai, main check karta hoon. Temperature 101.4 hai, BP 118 by 76, pulse 96. "
    "Gala thoda red hai. Mujhe lagta hai viral fever hai, dengue bhi ho sakta hai. Main CBC aur dengue NS1 test "
    "likh raha hoon. Paracetamol 650 mg din mein teen baar lijiye, aur paani zyada piyo. Agar platelets kam aaye "
    "ya bleeding ho toh turant aana. Do din baad report ke saath follow up kariye. Theek hai doctor, thank you."
)

if RUN_DEMO:
    demo = run_soap_agent(transcript=DEMO_TRANSCRIPT)
    print(demo["report"])
else:
    print("[demo] skipped (RUN_DEMO = False)")

In [ ]:
# ============================ 19. RUN ON ONE AUDIO FILE ============================
AUDIO_PATH = None          # e.g. "/kaggle/input/<your-dataset>/audio/conv_01.wav"; None = first audio found
REFERENCE_SOAP = None      # optional gold SOAP text for ROUGE / BLEU / BERTScore

if AUDIO_PATH is None and os.path.isdir("/kaggle/input"):
    for _root, _, _files in sorted(os.walk("/kaggle/input")):
        _hits = sorted(f for f in _files if f.lower().endswith(AUDIO_EXTS))
        if _hits:
            AUDIO_PATH = os.path.join(_root, _hits[0])
            break
if AUDIO_PATH:
    audio_result = run_soap_agent(audio_path=AUDIO_PATH, reference_soap=REFERENCE_SOAP)
    print(audio_result["report"])
else:
    print("No audio found under /kaggle/input - attach a dataset or set AUDIO_PATH (the demo above needs no audio).")

In [ ]:
# ============================ 20. BATCH RUN (whole dataset folder) ============================
def run_batch(input_dir, reference_dir=None, limit=None):
    """Run the agent graph on every audio file / .txt transcript in `input_dir`.
    reference_dir (optional): gold SOAP notes named <same stem>.txt. Saves notes + a CSV summary."""
    files = sorted(os.path.join(r, f) for r, _, fs in os.walk(input_dir) for f in fs
                   if f.lower().endswith(AUDIO_EXTS + (".txt",)))[:limit]
    notes_dir = os.path.join(OUTPUT_DIR, "notes")
    os.makedirs(notes_dir, exist_ok=True)
    rows = []
    for i, path in enumerate(files, 1):
        stem = os.path.splitext(os.path.basename(path))[0]
        ref = None
        if reference_dir and os.path.exists(os.path.join(reference_dir, stem + ".txt")):
            ref = open(os.path.join(reference_dir, stem + ".txt"), encoding="utf-8").read()
        print(f"\n##### [{i}/{len(files)}] {stem}")
        try:
            if path.lower().endswith(".txt"):
                r = run_soap_agent(transcript=open(path, encoding="utf-8").read(), reference_soap=ref)
            else:
                r = run_soap_agent(audio_path=path, reference_soap=ref)
            with open(os.path.join(notes_dir, stem + "_soap.txt"), "w", encoding="utf-8") as f:
                f.write(r["final_note"])
            em = r.get("eval_metrics", {})
            j, ks = em.get("llm_judge", {}), em.get("kspan_stats", {})
            rows.append({"file": stem, "language": (r.get("language_profile") or {}).get("label"),
                         "spans_verified": ks.get("verified"), "spans_rejected": ks.get("rejected"),
                         "kspan_coverage": em.get("keyword_coverage", {}).get("coverage"),
                         "hallucinations": em.get("hallucination", {}).get("count"),
                         "reflection_rounds": em.get("reflection", {}).get("rounds"),
                         "grounded_ratio": em.get("grounding", {}).get("grounded_ratio"),
                         "judge_composite": j.get("composite"), "faithfulness": j.get("faithfulness"),
                         "rouge1_f1": em.get("rouge1", {}).get("f1"), "bleu": em.get("bleu", {}).get("bleu"),
                         "seconds": round(sum((r.get("timings") or {}).values()), 1)})
        except Exception as e:
            print(f"   FAILED: {e}")
            rows.append({"file": stem, "error": str(e)[:200]})
    df = pd.DataFrame(rows)
    out = os.path.join(OUTPUT_DIR, "batch_summary.csv")
    df.to_csv(out, index=False)
    print(f"\nsaved {out}")
    try:
        print(df.select_dtypes("number").describe().T[["mean", "min", "max"]].round(3))
    except Exception:
        pass
    return df

# Example (uncomment and point at your dataset):
# batch_df = run_batch("/kaggle/input/<your-dataset>/audio", reference_dir="/kaggle/input/<your-dataset>/soap", limit=None)
# batch_df

---
## Dataset test: **BeTraC 2026** (Hugging Face `BeTraC/betrac-2026`)

Synthetic English doctor–patient conversations (JSALT 2025; Labrak et al., Interspeech 2026, CC-BY-4.0).
Each sample has **Opus audio (16 kHz)**, a **gold transcript**, a **gold SOAP note** and JSON metadata, stored as
WebDataset tar shards. The dataset is public, so no token or upload is needed. **Internet must be on.**

The next cell downloads only the **dev split** (400 dialogs, ~469 MB). This is the evaluation split; the train
shards (8.7 GB) are for fine-tuning and should not be used for testing. It extracts the split to
`/kaggle/working/betrac_dev/{audio,transcript,soap,meta}` and runs the agent graph on the first `BETRAC_LIMIT` dialogs.

* `BETRAC_MODE = "audio"`: the full pipeline from audio (ASR → … → SOAP), plus **Whisper WER** against the gold transcript.
* `BETRAC_MODE = "transcript"`: starts from the gold transcript (skips ASR). Cheaper; isolates the NLP agents.

Every note is scored against the **gold SOAP note** (ROUGE / BLEU / BERTScore) as well as by the judge,
faithfulness, hallucination and K-span metrics. BeTraC is English, so it tests the pipeline itself;
the Hindi-English behaviour is tested on your own dataset.

In [ ]:
# ============================ 21. DATASET TEST: BeTraC 2026 (Hugging Face) ============================
BETRAC_REPO  = "BeTraC/betrac-2026"
BETRAC_SHARD = "data/dev-00000.tar"   # dev = evaluation split (400 dialogs)
BETRAC_MODE  = "audio"                # "audio" (ASR + WER) or "transcript" (start from the gold transcript)
BETRAC_LIMIT = 10                     # start small (LLM cost / time); None = all 400
BETRAC_SEED  = 42                     # random sample across the dev split (None = first N in file order)
BETRAC_RESUME = True                  # skip dialogs already in the results CSV (continue across Kaggle sessions)
                                      # (results files carry PIPELINE_VERSION, so different versions never mix)

import tarfile, shutil
from huggingface_hub import hf_hub_download

def prepare_betrac(shard=BETRAC_SHARD, out_root=os.path.join(OUTPUT_DIR, "..", "betrac_dev")):
    """Download one WebDataset shard and unpack it into audio/ transcript/ soap/ meta/ folders (by sample key)."""
    out_root = os.path.abspath(out_root)
    dirs = {k: os.path.join(out_root, k) for k in ("audio", "transcript", "soap", "meta")}
    if os.path.isdir(dirs["soap"]) and os.listdir(dirs["soap"]):
        print(f"[betrac] already prepared -> {out_root}")
        return dirs
    for d in dirs.values():
        os.makedirs(d, exist_ok=True)
    print(f"[betrac] downloading {BETRAC_REPO}/{shard} ...")
    tar_path = hf_hub_download(repo_id=BETRAC_REPO, repo_type="dataset", filename=shard, token=HF_TOKEN or None)
    n = Counter()
    with tarfile.open(tar_path) as tf:
        for m in tf:
            if not m.isfile():
                continue
            base = os.path.basename(m.name)
            key, _, suffix = base.partition(".")          # WebDataset: key = text before the first dot
            target = {"opus": ("audio", ".opus"), "transcript.txt": ("transcript", ".txt"),
                      "soap.txt": ("soap", ".txt"), "json": ("meta", ".json")}.get(suffix.lower())
            if not target:
                continue
            with tf.extractfile(m) as src, open(os.path.join(dirs[target[0]], key + target[1]), "wb") as dst:
                shutil.copyfileobj(src, dst)
            n[target[0]] += 1
    print(f"[betrac] extracted {dict(n)} -> {out_root}")
    return dirs

def _wer_norm(t, whisper_norm=True):
    t = re.sub(r"(?im)^\s*(doctor|patient|attendant|dr|pt)\s*:", " ", t or "")
    if whisper_norm and "_asr_proc" in globals():
        try:          # Whisper's English normaliser: numbers, spellings, contractions, fillers (standard for WER)
            return _asr_proc.tokenizer.normalize(t).split()
        except Exception:
            pass
    return re.sub(r"[^a-z0-9' ]+", " ", t.lower()).split()

def word_error_rate(ref, hyp, whisper_norm=True):
    """Word-level Levenshtein distance / reference length (speaker labels ignored).
    Whisper's normaliser deletes text between brackets; an unmatched '[' or '<' in a reference can wipe out
    whole paragraphs. If normalisation removed >10% of the reference words, fall back to the basic normaliser."""
    r, h = _wer_norm(ref, whisper_norm), _wer_norm(hyp, whisper_norm)
    if whisper_norm and (len(r) < 0.9 * len(_wer_norm(ref, False)) or len(h) < 0.9 * len(_wer_norm(hyp, False))):
        r, h = _wer_norm(ref, False), _wer_norm(hyp, False)
    if not r:
        return None
    prev = list(range(len(h) + 1))
    for i, rw in enumerate(r, 1):
        cur = [i] + [0] * len(h)
        for j, hw in enumerate(h, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (rw != hw))
        prev = cur
    return round(prev[-1] / len(r), 4)

def run_betrac(dirs, mode=BETRAC_MODE, limit=BETRAC_LIMIT, seed=BETRAC_SEED, resume=BETRAC_RESUME):
    keys = sorted(os.path.splitext(f)[0] for f in os.listdir(dirs["soap"]))
    if seed is not None:
        import random
        random.Random(seed).shuffle(keys)          # spread the sample over the whole dev split
    keys = keys[:limit]
    out = os.path.join(OUTPUT_DIR, f"betrac_dev_{mode}_{PIPELINE_VERSION}_results.csv")
    done = pd.read_csv(out) if resume and os.path.exists(out) else pd.DataFrame()
    if len(done) and "error" in done:
        done = done[done["error"].isna()].drop(columns=["error"])       # failed dialogs are retried
    done_keys = set(done["key"]) if len(done) else set()
    if done_keys:
        print(f"[betrac] resuming: {len(done_keys & set(keys))} of {len(keys)} already done -> skipped")
    keys = [k for k in keys if k not in done_keys]
    audio_ok = None
    rows = []
    for i, key in enumerate(keys, 1):
        read = lambda d, ext: open(os.path.join(dirs[d], key + ext), encoding="utf-8").read() \
            if os.path.exists(os.path.join(dirs[d], key + ext)) else ""
        gold_soap, gold_tr = read("soap", ".txt"), read("transcript", ".txt")
        print(f"\n##### BeTraC [{i}/{len(keys)}] {key}")
        try:
            if mode == "audio":
                apath = os.path.join(dirs["audio"], key + ".opus")
                if audio_ok is None:                       # can librosa read .opus here? else convert to wav
                    try:
                        import librosa
                        librosa.load(apath, sr=16000, duration=1)
                        audio_ok = True
                    except Exception as e:
                        print(f"[betrac] .opus not readable directly ({str(e)[:50]}) -> converting with ffmpeg")
                        audio_ok = False
                if not audio_ok:
                    wav = os.path.join(dirs["audio"], key + ".wav")
                    if not os.path.exists(wav):
                        subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", apath, "-ar", "16000", "-ac", "1", wav])
                    apath = wav
                r = run_soap_agent(audio_path=apath, reference_soap=gold_soap)
            else:
                r = run_soap_agent(transcript=gold_tr, reference_soap=gold_soap)
            em = r.get("eval_metrics", {})
            j, ks = em.get("llm_judge", {}), em.get("kspan_stats", {})
            audio = mode == "audio"
            row = {"key": key, "mode": mode,
                   "asr_wer": word_error_rate(gold_tr, r.get("raw_transcript", "")) if audio else None,
                   "asr_wer_raw": word_error_rate(gold_tr, r.get("raw_transcript", ""), whisper_norm=False) if audio else None,
                   "spans_verified": ks.get("verified"), "kspan_coverage": em.get("keyword_coverage", {}).get("coverage"),
                   "hallucinations": em.get("hallucination", {}).get("count"),
                   "reflection_rounds": em.get("reflection", {}).get("rounds"),
                   "reflection_scores": ">".join(str(e["score"]) for e in r.get("reflection_log") or []),
                   "judge_composite": j.get("composite"), "faithfulness": j.get("faithfulness"),
                   "rouge1_f1": em.get("rouge1", {}).get("f1"), "rougeL_f1": em.get("rougeL", {}).get("f1"),
                   "bleu": em.get("bleu", {}).get("bleu"), "bertscore_f1": em.get("bertscore", {}).get("f1"),
                   "semsim_f1": em.get("semsim_sent", {}).get("f1"),
                   "note_words": len(extractive_soap_only(r.get("final_note", "")).split()),
                   "gold_words": len(gold_soap.split()),
                   "relabelled": sum("re-labelled" in e["action"] or "unlabelled" in e["action"]
                                     for e in r.get("trace") or []),
                   "empty_sections": "".join(k for k in "SOAP" if (r.get("best_soap") or {}).get(k, "").strip()
                                             in ("", _PLACEHOLDER.get(k, "-"))),
                   "seconds": round(sum((r.get("timings") or {}).values()), 1)}
            with open(os.path.join(OUTPUT_DIR, f"betrac_{key}_soap.txt"), "w", encoding="utf-8") as f:
                f.write(r["final_note"] + "\n\n==== GOLD SOAP ====\n" + gold_soap
                        + "\n\n==== TRANSCRIPT USED ====\n" + r.get("transcript", "")
                        + "\n\n==== REFLECTION LOG ====\n" + json.dumps(r.get("reflection_log"), indent=1, default=str)[:20000])
            with open(os.path.join(OUTPUT_DIR, f"betrac_{key}_run.json"), "w", encoding="utf-8") as f:
                json.dump({"note": r.get("final_note", ""), "raw_transcript": r.get("raw_transcript", ""),
                           "verified_spans": r.get("verified_spans", [])}, f, ensure_ascii=False, default=str)
            print({k: v for k, v in row.items() if k in ("asr_wer", "judge_composite", "rouge1_f1", "bertscore_f1")})
        except Exception as e:
            print(f"   FAILED: {e}")
            row = {"key": key, "mode": mode, "error": str(e)[:200]}
        rows.append(row)
        pd.concat([done, pd.DataFrame(rows)], ignore_index=True).to_csv(out, index=False)   # save after EVERY dialog
    df = pd.concat([done, pd.DataFrame(rows)], ignore_index=True)
    df.to_csv(out, index=False)
    print(f"\nsaved {out}")
    try:
        print(df.select_dtypes("number").describe().T[["mean", "min", "max"]].round(3))
    except Exception:
        pass
    return df

if RUN_BETRAC or RUN_COMPARISON or RUN_LATENCY:
    betrac_dirs = prepare_betrac()
if RUN_BETRAC:
    print("sample gold transcript:\n", open(os.path.join(betrac_dirs["transcript"], sorted(os.listdir(betrac_dirs["transcript"]))[0]), encoding="utf-8").read()[:400])
    betrac_df = run_betrac(betrac_dirs)
    from IPython.display import display
    display(betrac_df)
else:
    print("[betrac] benchmark skipped (RUN_BETRAC = False)")

---
## LoRA adapters, step 1: export training data (one base model, one adapter per agent)

In the LoRA design, **one base model** (`LORA_BASE_MODEL`, Llama 3.2 3B) is shared by every agent, and each
agent has its own small **LoRA adapter** that is hot-swapped in for its calls, as in *Agentic AI Architectures
for SOAP Note Generation* (BioNLP 2026). This cell builds one training file per adapter from the BeTraC
**train** split (never the dev split used for testing). Every example uses **exactly the prompt the agent
sends at run time**, so the adapters learn the real job.

| Adapter | Input → target | Where the target comes from |
|---|---|---|
| `diarization` | unlabelled text → labelled turns (JSON) | BeTraC gold transcripts (real labels) |
| `kspan` | transcript chunk → verified K-spans (JSON) | distilled: current K-Span + VerifiNER agents (OpenAI teacher) |
| `verifier` | candidate spans + knowledge → verdicts (JSON) | distilled: teacher verifier output |
| `generator` | transcript + K-span checklist → S/O/A/P note | BeTraC gold SOAP notes |
| `critic` | transcript + damaged note → issue list (JSON) | gold note with known damage added (drug never mentioned, denied symptom written as present, fact removed, vitals moved) |
| `reviser` | damaged note + feedback → corrected note | the same damaged notes, with the gold note as target |

Set `STAGE = "export"` in CONFIG to run it (this switches every other step off). It needs the **OpenAI backend** (the teacher) for the distilled adapters.
The output goes to `/kaggle/working/adapter_data/*.jsonl`. After the run, **Save Version** this notebook and
add its output as an input to the training notebook.

In [ ]:
# ============================ 22. EXPORT LoRA TRAINING DATA (BeTraC train split) ============================
EXPORT_SHARDS    = ["data/train-00000.tar"]   # ~800 dialogues per shard; add more shards for more data
EXPORT_N_DISTILL = 400                     # dialogues sent through the teacher K-Span + VerifiNER (API cost)
EXPORT_N_TOTAL   = None                    # dialogues used for diarization / generator examples (None = all)
EXPORT_CLEAN_SHARE = 0.25                  # share of critic examples with an undamaged note (target: no issues)
EXPORT_DIR = os.path.abspath(os.path.join(OUTPUT_DIR, "..", "adapter_data"))
EXPORT_CLEANUP = True                      # delete the unpacked train shard afterwards (keeps the Kaggle output small)

import random as _random, contextlib

_DRUG_NAMES = [w for w in re.findall(r"[a-z]{5,}", _KNOWN_DRUGS.pattern)]
_DOSES = ["10 mg once daily", "500 mg twice daily", "20 mg at night", "250 mg three times a day"]

def _gold_turns(transcript):
    """BeTraC gold transcript ('DOCTOR: ...' one turn per line) -> [{speaker, text}]."""
    turns = []
    for line in transcript.splitlines():
        m = re.match(r"^\s*\[?(doctor|patient|attendant|nurse|caregiver|relative)\]?\s*:\s*(.*)$", line, re.I)
        if m and m.group(2).strip():
            spk = m.group(1).lower()
            turns.append({"speaker": "Doctor" if spk == "doctor" else "Patient" if spk == "patient" else "Attendant",
                          "text": m.group(2).strip()})
        elif turns and line.strip():
            turns[-1]["text"] += " " + line.strip()          # continuation line
    return turns

def _note_text(sec):
    return "\n".join(f"{k}: {sec[k]}" for k in "SOAP")

def _gold_note(gold_soap):
    """Gold SOAP (Markdown headings) -> our S:/O:/A:/P: format; None if it does not parse cleanly."""
    sec = _parse_note(gold_soap)
    if not sec["S"].strip() or sum(bool(sec[k].strip()) for k in "SOAP") < 3:
        return None
    for k in ("O", "A", "P"):
        if not sec[k].strip():
            sec[k] = _PLACEHOLDER[k]
    return sec

def _sentences(text):
    return [x for x in re.split(r"(?<=[.!?])\s+|\n+", text) if x.strip()]

def damage_note(sec, spans, transcript, rng):
    """Add 1-2 known problems to a gold note. Returns (damaged note, hallucinations, missing, misplaced)."""
    bad, hall, missing, misplaced = dict(sec), [], [], []
    tl = transcript.lower()
    ops = ["add_drug", "add_denied", "drop_fact", "move_vital"]
    rng.shuffle(ops)
    for op in ops[:rng.choice([1, 1, 2])]:
        if op == "add_drug":
            cands = [d for d in _DRUG_NAMES if d not in tl]
            if cands:
                claim = f"Started {rng.choice(cands).capitalize()} {rng.choice(_DOSES)}."
                p = "" if bad["P"].strip() == _PLACEHOLDER["P"] else bad["P"].rstrip()
                bad["P"] = (p + "\n" + claim).strip()
                hall.append({"section": "P", "claim": claim, "reason": "drug never mentioned in the conversation"})
        elif op == "add_denied":
            denied = [s for s in spans if s["existence"] == "Absent" and s["type"] in ("Symptom", "Disease")]
            if denied:
                s = rng.choice(denied)
                term = s.get("normalized_en") or s["text"]
                claim = f"Patient reports {term}."
                bad["S"] = bad["S"].rstrip() + "\n" + claim
                hall.append({"section": "S", "claim": claim,
                             "reason": f'assertion conflict: the patient denied "{s["text"]}" in the conversation'})
        elif op == "drop_fact":
            sents = [x for x in _sentences(bad["S"])[1:] if len(x) > 25]
            if sents:
                x = rng.choice(sents)
                bad["S"] = bad["S"].replace(x, "").strip()
                missing.append({"section": "S", "fact": x.strip(" -*")})
        elif op == "move_vital":
            vit = [x for x in bad["O"].splitlines() if re.search(r"\d", x) and _VITALS_RE.search(x)]
            if vit:
                x = rng.choice(vit)
                bad["O"] = bad["O"].replace(x, "").strip() or _PLACEHOLDER["O"]
                bad["S"] = bad["S"].rstrip() + "\n" + x.strip()
                misplaced.append({"claim": x.strip(" -*"), "from": "O", "to": "S"})
    return bad, hall, missing, misplaced

def _record_role(role):
    """Capture every temperature-0 (messages, reply) the router produces for one role while the teacher runs."""
    calls = []
    orig = LLMRouter.chat.__get__(LLM)                   # the real bound method
    def rec(messages, r="agent", max_tokens=1500, temperature=0.2, json_mode=False):
        out = orig(messages, r, max_tokens, temperature, json_mode)
        if r == role and not temperature:
            calls.append({"messages": list(messages), "reply": out})
        return out
    return calls, rec

@contextlib.contextmanager
def _export_settings(values):
    old = {k: globals()[k] for k in values}
    globals().update(values)
    try:
        yield
    finally:
        globals().update(old)

def export_adapter_data():
    if LLM.backend != "openai":
        raise RuntimeError("Export needs the OpenAI backend as teacher (set STAGE = 'export' in CONFIG).")
    os.makedirs(EXPORT_DIR, exist_ok=True)
    rng = _random.Random(1234)
    data = {k: [] for k in ("diarization", "kspan", "verifier", "generator", "critic", "reviser")}
    keys_all, shard_roots = [], []
    for shard in EXPORT_SHARDS:
        name = os.path.splitext(os.path.basename(shard))[0]
        d = prepare_betrac(shard=shard, out_root=os.path.join(OUTPUT_DIR, "..", f"betrac_{name}"))
        shard_roots.append(os.path.dirname(d["soap"]))
        keys_all += [(d, os.path.splitext(f)[0]) for f in sorted(os.listdir(d["soap"]))]
    rng.shuffle(keys_all)
    keys_all = keys_all[:EXPORT_N_TOTAL]
    skipped = Counter()
    for i, (d, key) in enumerate(keys_all, 1):
        tr = open(os.path.join(d["transcript"], key + ".txt"), encoding="utf-8").read()
        gold = _gold_note(open(os.path.join(d["soap"], key + ".txt"), encoding="utf-8").read())
        turns = _gold_turns(tr)
        if gold is None or len(turns) < 4:
            skipped["unparseable gold"] += 1
            continue
        transcript = "\n".join(f"{t['speaker']}: {t['text']}" for t in turns)
        # diarization: speaker labels removed -> gold turns
        raw = " ".join(t["text"] for t in turns)
        data["diarization"].append({"messages": diarize_messages(raw) + [
            {"role": "assistant", "content": json.dumps({"turns": turns}, ensure_ascii=False)}], "key": key})
        spans = []
        if i <= EXPORT_N_DISTILL:
            # teacher: K-Span + VerifiNER (one vote, temperature 0) -> verified spans; record verifier calls
            calls, rec = _record_role("verifier")
            LLM.chat = rec                                # instance attribute shadows the method while recording
            try:
                with _export_settings({"VERIFY_VOTES": 1, "VERBOSE": False}):
                    st = {"transcript": transcript, "kspan_attempts": 0, "timings": {}}
                    st.update(kspan_agent(st))
                    spans = verifiner_agent(st).get("verified_spans", [])
            except Exception as e:
                skipped["teacher failed"] += 1
                print(f"   teacher failed on {key}: {str(e)[:60]}")
            finally:
                del LLM.chat                              # back to the class method
            for c in calls:
                data["verifier"].append({"messages": c["messages"] + [{"role": "assistant", "content": c["reply"]}], "key": key})
            # kspan: each chunk -> the verified spans that fall inside it
            pos = 0
            for chunk in _chunks(transcript, KSPAN_CHUNK_CHARS):
                start = transcript.find(chunk, pos)
                end = start + len(chunk)
                pos = max(pos, end)
                inside = [s for s in spans if s.get("offset") and start <= s["offset"]["start"] < end]
                ents = [{f: s.get(f, "") for f in ("text", "type", "existence", "normalized_en", "severity",
                                                     "temporality", "experiencer", "speaker", "cue")} for s in inside]
                data["kspan"].append({"messages": kspan_messages(chunk) + [
                    {"role": "assistant", "content": json.dumps({"entities": ents}, ensure_ascii=False)}], "key": key})
            # critic + reviser: damaged gold note
            if spans:
                if rng.random() < EXPORT_CLEAN_SHARE:
                    bad, hall, missing, misplaced = dict(gold), [], [], []
                else:
                    bad, hall, missing, misplaced = damage_note(gold, spans, transcript, rng)
                target = {"unsupported": [{k: h[k] for k in ("section", "claim", "reason")} for h in hall],
                          "missing": missing, "misplaced": misplaced}
                data["critic"].append({"messages": critic_messages(transcript, spans, bad) + [
                    {"role": "assistant", "content": json.dumps(target, ensure_ascii=False)}], "key": key})
                if hall or missing or misplaced:
                    fb = _format_feedback(hall, missing, misplaced, [])
                    data["reviser"].append({"messages": soap_gen_messages(transcript, spans, bad, fb) + [
                        {"role": "assistant", "content": _note_text(gold)}], "key": key})
        # generator: transcript (+ checklist when distilled) -> gold note
        data["generator"].append({"messages": soap_gen_messages(transcript, spans) + [
            {"role": "assistant", "content": _note_text(gold)}], "key": key})
        if i % 25 == 0:
            print(f"[export] {i}/{len(keys_all)}  " + "  ".join(f"{k}={len(v)}" for k, v in data.items()))
    manifest = {"base_model": LORA_BASE_MODEL, "shards": EXPORT_SHARDS, "dialogues": len(keys_all),
                "distilled": min(EXPORT_N_DISTILL, len(keys_all)), "skipped": dict(skipped),
                "teacher_models": dict(LLM.models), "counts": {k: len(v) for k, v in data.items()},
                "created": datetime.now().isoformat(timespec="seconds")}
    for name, rows in data.items():
        with open(os.path.join(EXPORT_DIR, f"{name}.jsonl"), "w", encoding="utf-8") as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    json.dump(manifest, open(os.path.join(EXPORT_DIR, "manifest.json"), "w"), indent=1)
    if EXPORT_CLEANUP:
        import shutil
        for root in shard_roots:
            shutil.rmtree(root, ignore_errors=True)
    print(f"[export] done -> {EXPORT_DIR}")
    print(json.dumps(manifest["counts"], indent=1))
    print("[export] NEXT: Save Version -> open lora_adapter_training.ipynb -> Add Input -> this notebook's output")
    return manifest

if RUN_EXPORT:
    export_manifest = export_adapter_data()
else:
    print("[export] skipped (set STAGE = 'export' in CONFIG to build the LoRA training files)")

---
## Baseline comparison: friend's straight pipeline vs the agentic pipeline

Runs the **original straight implementation** on the **same BeTraC dialogues** as the agentic run above, then
scores **both** systems' notes with **one identical scorer**:

* the same independent judge, against the **gold** transcript;
* the same hallucination audit, done by the judge model, not by either system's own reviewer;
* the same reference metrics (ROUGE, BLEU, real BERTScore);
* K-span coverage against **one neutral fact list**, extracted once from BeTraC's **gold** transcript by the
  K-Span and VerifiNER agents and cached, so it favours neither system.

**How the straight notebook is loaded.** Its code runs in an **isolated namespace**, so none of its functions
overwrite the agentic ones. It runs exactly as its Gradio button did: its Whisper → its diarization →
`run_full_pipeline(...)`. Only these parts are skipped:

* the NeMo / MFA alignment and word-confidence cells (not part of note generation);
* the `gensim` install;
* the hard-coded Hugging Face login;
* its own evaluation step (we score it ourselves).

**Before running:**
1. Run the BeTraC cell above, so the agentic notes exist. Notes saved by earlier versions
   (`betrac_<key>_soap.txt`) are also accepted, so a finished run does not have to be repeated.
2. Make the friend's notebook file available, in one of two ways:
   * attach it as a Kaggle input (any `.ipynb` under `/kaggle/input` that contains `run_full_pipeline` is found
     automatically), or
   * set `BASELINE_NOTEBOOK_URL` to its raw GitHub link.

In [ ]:
# ============================ 22. BASELINE: straight pipeline vs agentic (same dialogues, same scorer) ============================
BASELINE_NOTEBOOK     = None   # path to the friend's .ipynb; None = search /kaggle/input automatically
BASELINE_NOTEBOOK_URL = ""     # or a raw GitHub link, e.g. "https://raw.githubusercontent.com/<user>/<repo>/main/<file>.ipynb"
BASELINE_RESULTS      = os.path.join(OUTPUT_DIR, f"betrac_dev_{BETRAC_MODE}_{PIPELINE_VERSION}_results.csv")  # agentic run to compare with

import glob, urllib.request

# ---------- 1. find and load the straight notebook in its own namespace ----------
def _find_baseline_notebook():
    if BASELINE_NOTEBOOK:
        return BASELINE_NOTEBOOK
    if BASELINE_NOTEBOOK_URL:
        dest = os.path.join(OUTPUT_DIR, "baseline_notebook.ipynb")
        urllib.request.urlretrieve(BASELINE_NOTEBOOK_URL, dest)
        return dest
    for p in sorted(glob.glob("/kaggle/input/**/*.ipynb", recursive=True)):
        try:
            if "def run_full_pipeline" in open(p, encoding="utf-8").read():
                return p
        except Exception:
            pass
    raise FileNotFoundError("Friend's notebook not found: attach it as a Kaggle input or set BASELINE_NOTEBOOK_URL.")

# cells that make up the straight pipeline (picked by content, so cell order changes do not matter)
_BASELINE_CELLS = ["DEMO CONFIG", "def run_full_pipeline", "EVAL UPGRADE", "FINAL OVERRIDES",
                   "Clean medical-codes output", "PRESENTATION LAYER"]
_BASELINE_PATCHES = [   # (pattern, replacement): only unsafe / irrelevant side effects are removed
    (r"login\(token=['\"]hf_[A-Za-z0-9]+['\"]\)", "pass  # hard-coded token removed"),
    (r"subprocess\.run\('pip install --upgrade typing_extensions'[^\n]*\)", "pass"),
    (r"subprocess\.run\('nvidia-smi', shell=True, check=True\)", "pass"),
    (r"try:\n\s*import gensim\.downloader as _gens_dl\nexcept ImportError:\n\s*subprocess\.run\(\[[^\]]*gensim[^\]]*\]\)\n\s*import gensim\.downloader as _gens_dl",
     "_gens_dl = None  # gensim install skipped (WMD not used here)"),
]

def load_baseline_pipeline():
    path = _find_baseline_notebook()
    nb = json.load(open(path, encoding="utf-8"))
    code = ["".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code"]
    fg = {"__name__": "straight_pipeline"}
    for marker in _BASELINE_CELLS:
        src = next((c for c in code if marker in c), None)
        if src is None:
            if marker == "def run_full_pipeline":
                raise RuntimeError("This notebook has no run_full_pipeline - is it the straight implementation?")
            print(f"[baseline] cell '{marker}' not found - skipped")
            continue
        for pat, rep in _BASELINE_PATCHES:
            src = re.sub(pat, rep, src)
        print(f"[baseline] loading cell: {marker}")
        exec(compile(src, f"<straight:{marker}>", "exec"), fg)
    # its own evaluation is replaced by ours (scored identically below); keep the keys run_full_pipeline prints
    fg["evaluate_soap"] = lambda *a, **k: {"llm_judge": {"composite": None},
                                           "keyword_coverage": {"coverage": 0.0}, "hallucination": {"count": 0}}
    print(f"[baseline] straight pipeline ready from {path}")
    return fg

def run_baseline_on(fg, key, audio_path):
    """Exactly what the straight notebook's Gradio button does: its Whisper -> its diarization -> run_full_pipeline."""
    t0 = time.time()
    asr = fg["pipe"](audio_path)
    raw = (asr.get("text", "") if isinstance(asr, dict) else str(asr)).strip()
    try:
        transcript = fg["diarize_asr_text"](raw)
    except Exception as e:                      # its diarizer raises when the JSON is malformed
        print(f"   [baseline] diarization failed ({str(e)[:50]}) -> raw ASR text")
        transcript = raw
    res = fg["run_full_pipeline"](transcript, k_shot=3, strategy="query_aware")
    return {"note": res.get("soap_raw", ""), "raw_transcript": raw, "seconds": round(time.time() - t0, 1)}

# ---------- 2. one scorer for both systems ----------
_AUDIT_SYSTEM = """You are auditing a SOAP note (English) against the doctor-patient CONVERSATION it was written from.
List every statement in the note that the conversation does NOT support: invented findings, symptoms, history,
diagnoses or differentials the doctor did not give, orders the doctor did not give, or a wrong assertion (the note
says the patient has something the patient denied). Judge meaning, not wording. Ignore lines that only say a section
has no content. Return ONLY a JSON object: {"unsupported": [{"claim": "...", "reason": "..."}]}"""

def audit_note(note, transcript):
    d = LLM.json([{"role": "system", "content": _AUDIT_SYSTEM},
                  {"role": "user", "content": f"CONVERSATION:\n{transcript[:40000]}\n\nSOAP NOTE:\n{note}\n\nJSON object:"}],
                 role="judge", max_tokens=1200, temperature=0.0)
    items = [x for x in d.get("unsupported") or [] if isinstance(x, dict) and str(x.get("claim", "")).strip()]
    tl = transcript.lower()
    drugs = sorted({m.group().lower() for m in _KNOWN_DRUGS.finditer(note)} - {w for w in _KNOWN_DRUGS.findall(tl)})
    return len(items) + len(drugs), [str(x["claim"])[:80] for x in items] + [f"drug: {d}" for d in drugs]

def score_note(note, gold_tr, gold_soap, spans):
    ext = extractive_soap_only(note)
    sec = _parse_note(ext)
    j = llm_judge(gold_tr, ext)
    n_hall, hall_items = audit_note(ext, gold_tr)
    row = {"judge_composite": j.get("composite"), "quality": j.get("quality_of_soap"),
           "adequacy": j.get("adequacy_wrt_transcript"), "faithfulness": j.get("faithfulness"),
           "hallucinations": n_hall, "hallucination_items": " | ".join(hall_items)[:300],
           "kspan_coverage": kspan_coverage(ext, spans)["coverage"] if spans else None,
           "rouge1_f1": rouge_n(ext, gold_soap, n=1)["f1"], "rougeL_f1": rouge_l(ext, gold_soap)["f1"],
           "bleu": bleu_score(ext, gold_soap)["bleu"], "note_words": len(ext.split()),
           "empty_sections": "".join(k for k in "SOAP" if sec[k].strip() in ("", _PLACEHOLDER.get(k, "-")))}
    if TRUE_BERTSCORE:
        try:
            row["bertscore_f1"] = true_bertscore(ext, gold_soap)["f1"]
        except Exception as e:
            print(f"   BERTScore skipped: {str(e)[:50]}")
    return row

# ---------- 3. inputs: agentic note + one neutral fact list ----------
def _agentic_output(key):
    """The agentic note: from betrac_<key>_run.json (current version) or, for runs made with an earlier
    version, from the note part of betrac_<key>_soap.txt."""
    run_file = os.path.join(OUTPUT_DIR, f"betrac_{key}_run.json")
    if os.path.exists(run_file):
        return json.load(open(run_file, encoding="utf-8"))
    soap_file = os.path.join(OUTPUT_DIR, f"betrac_{key}_soap.txt")
    if os.path.exists(soap_file):
        note = open(soap_file, encoding="utf-8").read().split("\n\n==== GOLD SOAP ====")[0]
        return {"note": note, "raw_transcript": ""}
    return None

def gold_fact_list(key, gold_tr):
    """Verified K-spans extracted from the GOLD transcript (K-Span + VerifiNER agents), cached.
    Both systems are measured against this same list, so coverage favours neither."""
    cache = os.path.join(OUTPUT_DIR, f"betrac_{key}_goldspans.json")
    if os.path.exists(cache):
        return json.load(open(cache, encoding="utf-8"))
    st = {"transcript": _normalise_labels(gold_tr), "kspan_attempts": 0, "timings": {}}
    st.update(kspan_agent(st))
    spans = verifiner_agent(st).get("verified_spans", [])
    json.dump(spans, open(cache, "w", encoding="utf-8"), ensure_ascii=False, default=str)
    return spans

# ---------- 4. run ----------
def compare_with_baseline(dirs, results_csv=BASELINE_RESULTS):
    if not os.path.exists(results_csv):
        raise FileNotFoundError(f"{results_csv} not found - run the BeTraC cell (agentic) first.")
    ours = pd.read_csv(results_csv)
    if "error" in ours:
        ours = ours[ours["error"].isna()]
    keys = list(ours["key"])
    fg = globals().get("_BASELINE_FG") or load_baseline_pipeline()   # loaded once, shared with the latency cell
    globals()["_BASELINE_FG"] = fg
    rows = []
    for i, key in enumerate(keys, 1):
        print(f"\n##### COMPARE [{i}/{len(keys)}] {key}")
        read = lambda d, ext: open(os.path.join(dirs[d], key + ext), encoding="utf-8").read()
        gold_tr, gold_soap = read("transcript", ".txt"), read("soap", ".txt")
        agentic = _agentic_output(key)
        if agentic is None:
            print("   no agentic note for this dialogue (run the BeTraC cell first) -> skipped")
            continue
        spans = gold_fact_list(key, gold_tr)          # neutral fact list, identical for both systems
        # straight pipeline (cached so a re-run does not repeat it)
        cache = os.path.join(OUTPUT_DIR, f"baseline_{key}.json")
        if os.path.exists(cache):
            base = json.load(open(cache, encoding="utf-8"))
        else:
            wav = os.path.join(dirs["audio"], key + ".wav")
            apath = wav if os.path.exists(wav) else os.path.join(dirs["audio"], key + ".opus")
            try:
                base = run_baseline_on(fg, key, apath)
            except Exception as e:
                print(f"   [baseline] FAILED: {e}")
                rows.append({"key": key, "system": "straight", "error": str(e)[:200]})
                continue
            json.dump(base, open(cache, "w", encoding="utf-8"), ensure_ascii=False)
        our_secs = ours.loc[ours["key"] == key, "seconds"]
        for system, note, secs, raw in [
                ("straight", base["note"], base.get("seconds"), base.get("raw_transcript", "")),
                ("agentic", agentic["note"], float(our_secs.iloc[0]) if len(our_secs) else None, agentic.get("raw_transcript", ""))]:
            row = {"key": key, "system": system, **score_note(note, gold_tr, gold_soap, spans), "seconds": secs}
            if raw:
                row["asr_wer"] = word_error_rate(gold_tr, raw)
            rows.append(row)
            print(f"   {system:9} judge={row['judge_composite']}  halluc={row['hallucinations']}  "
                  f"coverage={row['kspan_coverage']}  rouge1={row['rouge1_f1']}")
    df = pd.DataFrame(rows)
    out = os.path.join(OUTPUT_DIR, f"baseline_comparison_{PIPELINE_VERSION}.csv")
    df.to_csv(out, index=False)
    print(f"\nsaved {out}")

    ok = df[df["error"].isna()] if "error" in df else df
    if ok.empty or "system" not in ok:
        print("Nothing was compared - see the messages above.")
        return df, pd.DataFrame()
    metrics = [m for m in ["judge_composite", "faithfulness", "adequacy", "hallucinations", "kspan_coverage",
                           "rouge1_f1", "rougeL_f1", "bleu", "bertscore_f1", "asr_wer", "note_words", "seconds"] if m in ok]
    summary = ok.groupby("system")[metrics].mean().T.round(3)
    print("\n=== MEAN OVER THE SAME DIALOGUES ===")
    print(summary)
    paired = ok.pivot(index="key", columns="system", values=["judge_composite", "hallucinations", "kspan_coverage"]).dropna()
    if len(paired):
        print("\n=== PER-DIALOGUE: is the agentic note better? ===")
        for m, higher_is_better in [("judge_composite", True), ("kspan_coverage", True), ("hallucinations", False)]:
            a, s = paired[(m, "agentic")], paired[(m, "straight")]
            better = (a > s) if higher_is_better else (a < s)
            worse = (a < s) if higher_is_better else (a > s)
            print(f"  {m:16} agentic better {int(better.sum())}, equal {int((a == s).sum())}, worse {int(worse.sum())}  (of {len(paired)})")
    return df, summary

if RUN_COMPARISON:
    from IPython.display import display
    try:
        comparison_df, comparison_summary = compare_with_baseline(betrac_dirs)
        display(comparison_summary)
    except FileNotFoundError as e:      # friend's notebook (or the agentic results) not attached: skip, keep running
        print(f"[comparison] skipped: {e}")
else:
    print("[comparison] functions defined; comparison not re-run (RUN_COMPARISON = False)")

---
## Model sweep: Single-agent vs Multi-agent (VeriSOAP) on Llama 3B/8B, Ministral 3B/8B and GPT-5 mini

Set `STAGE = "sweep"` in CONFIG. For every model in `SWEEP_RUN` and the same BeTraC dev dialogues as the Nishat
comparison, the notebook runs **our code** in two ways:

| Architecture | What runs | LLM calls |
|---|---|---|
| **single** | one agent: ASR transcript → K-SOAP note in one call | 1 |
| **multi** | the full VeriSOAP graph: Diarization → K-Span → VerifiNER → Generator ⇄ Reflection → Addition | ~15–30 |

* **Every agent role runs on the chosen model** (no adapters; prompted). Llama loads in 4-bit, Ministral 3 from its FP8
  checkpoint de-quantised to fp16 (use **GPU T4 ×2** for Ministral 3 8B), GPT-5 mini through the API.
* **Same input for every model:** the speaker-labelled transcript saved by the agentic BeTraC run (attach that
  version), so ASR and diarization do not differ between rows (`SWEEP_INPUT = "asr"` re-runs diarization per model).
* Local models print one line per LLM call (tokens and tok/s), so a long multi-agent run shows it is still working.
* **Same scorer as the Nishat comparison:** gpt-4.1 judge, independent hallucination audit, K-span coverage against
  the gold-transcript fact list, ROUGE / BLEU / BERTScore. The VeriSOAP and Nishat rows from
  `baseline_comparison_v5.csv` are added to the table when that file is present.
* Results are saved after every run (`model_sweep_v5.csv`); attach the previous version to resume.

In [ ]:
# ============================ 22b. MODEL SWEEP: Single-agent vs Multi-agent (VeriSOAP) on each model ============================
# Every LLM role (diarization, K-Span, VerifiNER, generator, reviser, critic, addition) runs on the chosen model.
# The judge and the hallucination audit stay on OPENAI_JUDGE_MODEL, and the K-span fact list is built once from the
# GOLD transcript by the reference (OpenAI) agents, so every row is scored by the same scorer as the Nishat comparison.
import gc, glob
import pandas as pd

SINGLE_SYSTEM = (
    "You are a clinical scribe. From the doctor-patient conversation below (Hindi, English or Hindi-English "
    "code-mixed; it may have no speaker labels) write a K-SOAP note in English.\n"
    "K: every clinically relevant span, copied EXACTLY as spoken (original language and script), in the format "
    '("span as spoken", English concept) [Entity type, Assertion], separated by commas.\n'
    f"   Entity types: {', '.join(ENTITY_TYPES)}. Assertions: {', '.join(EXISTENCE_CLASSES)}.\n"
    "S: what the patient reports or denies (chief complaint first), history, medicines, allergies.\n"
    "O: vitals, examination findings and test results stated in the conversation.\n"
    "A: the doctor's impression or diagnosis as stated.\n"
    "P: tests, medicines with dose, advice, referrals and follow-up as stated by the doctor.\n"
    "Use ONLY information in the conversation; never invent. Write 'Not documented.' for an empty section.\n"
    "Output exactly five sections, each starting on its own line with 'K:', 'S:', 'O:', 'A:' or 'P:'.")


class SweepLLM:
    """Drop-in replacement for LLM: every agent role -> one model; 'judge' -> the OpenAI judge."""
    def __init__(self, name, backend, model_id):
        from openai import OpenAI
        self.name, self.backend, self.model_id = name, "sweep", model_id
        self.calls, self.call_hook, self._kind = Counter(), None, backend
        self.judge = OpenAI(api_key=OPENAI_API_KEY)
        self.models = {r: f"{name} ({model_id})" for r in AGENT_ROLES}
        self.models["judge"] = OPENAI_JUDGE_MODEL
        if backend == "openai":
            self.client, self._kw = OpenAI(api_key=OPENAI_API_KEY), {"reasoning_effort": "low"}
            return
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        if HF_TOKEN:
            from huggingface_hub import login
            login(token=HF_TOKEN)
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        if backend == "hf-ministral":                       # Ministral 3: FP8 checkpoint -> fp16 (T4 has no FP8)
            from transformers import Mistral3ForConditionalGeneration
            try:
                from transformers import MistralCommonBackend
                self.tok = MistralCommonBackend.from_pretrained(model_id)
            except Exception:
                self.tok = AutoTokenizer.from_pretrained(model_id)
            kw = dict(device_map="auto", dtype=dtype)
            try:
                from transformers import FineGrainedFP8Config
                kw["quantization_config"] = FineGrainedFP8Config(dequantize=True)
            except ImportError:
                pass
            self.model = Mistral3ForConditionalGeneration.from_pretrained(model_id, **kw)
        else:                                               # Llama: 4-bit NF4
            self.tok = AutoTokenizer.from_pretrained(model_id)
            kw = dict(device_map="auto", dtype=dtype)
            if torch.cuda.is_available():
                kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                               bnb_4bit_compute_dtype=torch.float16,
                                                               bnb_4bit_use_double_quant=True)
            self.model = AutoModelForCausalLM.from_pretrained(model_id, **kw)
        self.model.eval()
        eos = {getattr(self.tok, "eos_token_id", None)}
        g = getattr(self.model.generation_config, "eos_token_id", None)
        eos |= set(g if isinstance(g, (list, tuple)) else [g])
        for t in ("<|eot_id|>", "<|im_end|>", "</s>"):
            try:
                i = self.tok.convert_tokens_to_ids(t)
                if isinstance(i, int) and i >= 0 and i != getattr(self.tok, "unk_token_id", None):
                    eos.add(i)
            except Exception:
                pass
        self._eos = sorted(t for t in eos if isinstance(t, int) and t >= 0)
        self._pad = getattr(self.tok, "pad_token_id", None)
        self._pad = self._pad if self._pad is not None else (self._eos[0] if self._eos else 0)

    def _hf(self, messages, max_tokens, temperature):
        import torch
        try:
            enc = self.tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
        except TypeError:
            enc = self.tok.apply_chat_template(messages, return_tensors="pt", return_dict=True)
        dev = next(self.model.parameters()).device
        enc = {k: v.to(dev) for k, v in dict(enc).items() if hasattr(v, "to")}
        gen = dict(max_new_tokens=max_tokens, eos_token_id=self._eos, pad_token_id=self._pad)
        gen.update(dict(do_sample=True, temperature=temperature, top_p=0.9) if temperature else dict(do_sample=False))
        t0 = time.time()
        with torch.no_grad():
            out = self.model.generate(**enc, **gen)
        n_new = int(out.shape[1] - enc["input_ids"].shape[1])
        if VERBOSE:
            dt = time.time() - t0
            print(f"      [{self.name}] {self._role}: {enc['input_ids'].shape[1]} prompt + {n_new} new tokens "
                  f"in {dt:.0f}s ({n_new / max(dt, 1e-6):.1f} tok/s)")
        text = self.tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        return re.sub(r"(?s)<think>.*?</think>", "", text).strip()

    def _api(self, client, model, messages, max_tokens, temperature, json_mode, gpt5):
        kw = dict(model=model, messages=messages)
        if gpt5:                                             # GPT-5 family: no temperature, reasoning tokens count
            kw.update(max_completion_tokens=max_tokens + 3000, **self._kw)
        else:
            kw.update(max_tokens=max_tokens, temperature=temperature)
        if json_mode:
            kw["response_format"] = {"type": "json_object"}
        for attempt in range(5):
            try:
                return client.chat.completions.create(**kw).choices[0].message.content or ""
            except Exception as e:
                if "reasoning_effort" in str(e) and "reasoning_effort" in kw:
                    kw.pop("reasoning_effort"); self._kw = {}
                    continue
                if "response_format" in str(e) and "response_format" in kw:
                    kw.pop("response_format")
                    continue
                if attempt == 4:
                    raise
                time.sleep(2 ** attempt)

    def chat(self, messages, role="agent", max_tokens=1500, temperature=0.2, json_mode=False):
        self.calls[role] += 1
        if role == "judge":
            return self._api(self.judge, OPENAI_JUDGE_MODEL, messages, max_tokens, temperature, json_mode, False)
        if self._kind == "openai":
            return self._api(self.client, self.model_id, messages, max_tokens, temperature, json_mode,
                             self.model_id.startswith("gpt-5"))
        self._role = role
        return self._hf(messages, max_tokens, temperature)

    json = LLMRouter.json                                    # same parse-and-retry logic as the router

    def close(self):
        if hasattr(self, "model"):
            import torch
            del self.model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


def run_single_agent(raw_transcript):
    """Single-agent baseline: ONE call writes the whole K-SOAP note from the ASR transcript."""
    t0 = time.time()
    raw = LLM.chat([{"role": "system", "content": SINGLE_SYSTEM},
                    {"role": "user", "content": f"CONVERSATION:\n{raw_transcript}\n\nK-SOAP note:"}],
                   role="generator", max_tokens=SOAP_MAX_TOKENS, temperature=0.2)
    k = re.search(r"(?ms)^\s*K\s*:\s*(.*?)(?=^\s*[SOAP]\s*:|\Z)", raw)
    sec = _parse_note(extractive_soap_only(raw))
    note = (f"K: {k.group(1).strip()}\n\n" if k else "") + _assemble(sec)
    return {"final_note": note, "seconds": round(time.time() - t0, 1)}


def _sweep_keys(dirs):
    keys = sorted(os.path.splitext(f)[0] for f in os.listdir(dirs["soap"]))
    if BETRAC_SEED is not None:
        import random
        random.Random(BETRAC_SEED).shuffle(keys)            # same sample as run_betrac / the Nishat comparison
    return keys[:SWEEP_LIMIT]

def _reference_file(name):
    for p in [os.path.join(OUTPUT_DIR, name)] + sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True)):
        if os.path.exists(p):
            return p
    return None

def _sweep_input(key, dirs):
    """The SAME input text for every model.
    SWEEP_INPUT = "diarized": the speaker-labelled transcript of the reference agentic run (betrac_<key>_soap.txt),
                  so no model spends time re-labelling speakers (the diarization agent sees labels and skips).
    SWEEP_INPUT = "asr"     : the raw Whisper text; the diarization agent then runs on each model (much slower)."""
    if SWEEP_INPUT == "diarized":
        p = _reference_file(f"betrac_{key}_soap.txt")
        if p:
            t = open(p, encoding="utf-8").read()
            if "==== TRANSCRIPT USED ====" in t:
                t = t.split("==== TRANSCRIPT USED ====", 1)[1].split("==== REFLECTION LOG ====", 1)[0].strip()
                if t:
                    return t
        print(f"   [sweep] {key}: no diarized reference transcript found -> raw Whisper text")
    p = _reference_file(f"betrac_{key}_run.json")
    if p:
        t = (json.load(open(p, encoding="utf-8")).get("raw_transcript") or "").strip()
        if t:
            return t
    cache = os.path.join(OUTPUT_DIR, f"sweep_asr_{key}.txt")
    if os.path.exists(cache):
        return open(cache, encoding="utf-8").read()
    apath = os.path.join(dirs["audio"], key + ".opus")
    wav = os.path.join(dirs["audio"], key + ".wav")
    if not os.path.exists(wav):
        subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", apath, "-ar", "16000", "-ac", "1", wav])
    t = asr_agent({"audio_path": wav, "timings": {}})["raw_transcript"]
    open(cache, "w", encoding="utf-8").write(t)
    return t

def _gold_spans_cached(key, gold_tr):
    """Neutral fact list (reference OpenAI agents on the GOLD transcript), reused from the Nishat comparison."""
    for p in [os.path.join(OUTPUT_DIR, f"betrac_{key}_goldspans.json")] + \
             sorted(glob.glob(f"/kaggle/input/**/betrac_{key}_goldspans.json", recursive=True)):
        if os.path.exists(p):
            return json.load(open(p, encoding="utf-8"))
    return gold_fact_list(key, gold_tr)                     # needs the reference LLM (built at the top)


def run_model_sweep():
    global LLM, VERIFY_VOTES
    if not OPENAI_API_KEY:
        raise RuntimeError("The sweep needs OPENAI_API_KEY for the judge, the audit and the reference fact list.")
    dirs = prepare_betrac()
    keys = _sweep_keys(dirs)
    out_csv = os.path.join(OUTPUT_DIR, f"model_sweep_{PIPELINE_VERSION}.csv")
    done = pd.read_csv(out_csv) if os.path.exists(out_csv) else pd.DataFrame()
    for p in sorted(glob.glob(f"/kaggle/input/**/model_sweep_{PIPELINE_VERSION}.csv", recursive=True)):
        done = pd.concat([done, pd.read_csv(p)], ignore_index=True)          # resume across Kaggle versions
    if len(done):
        if "error" in done:
            done = done[done["error"].isna()]
        done = done.drop_duplicates(["model", "architecture", "key"], keep="last")
    have = set(zip(done.get("model", []), done.get("architecture", []), done.get("key", [])))
    rows = done.to_dict("records")
    ref_llm = LLM
    gold = {}
    for key in keys:                                        # fact lists + ASR text first, with the reference LLM
        read = lambda d, ext: open(os.path.join(dirs[d], key + ext), encoding="utf-8").read()
        gold[key] = {"tr": read("transcript", ".txt"), "soap": read("soap", ".txt")}
        gold[key]["spans"] = _gold_spans_cached(key, gold[key]["tr"])
        gold[key]["asr"] = _sweep_input(key, dirs)
    for name in SWEEP_RUN:
        todo = [(a, k) for a in SWEEP_ARCHS for k in keys if (name, a, k) not in have]
        if not todo:
            print(f"[sweep] {name}: all done")
            continue
        backend, model_id = SWEEP_MODELS[name]
        print(f"\n######## {name} ({model_id}): {len(todo)} runs ########")
        try:
            LLM = SweepLLM(name, backend, model_id)
        except Exception as e:
            print(f"[sweep] {name} could not be loaded -> skipped: {str(e)[:300]}")
            LLM = ref_llm
            continue
        votes_main, VERIFY_VOTES = VERIFY_VOTES, SWEEP_VERIFY_VOTES
        for arch, key in todo:
            g = gold[key]
            try:
                if arch == "single":
                    r = run_single_agent(g["asr"])
                    note, secs = r["final_note"], r["seconds"]
                else:
                    t0 = time.time()
                    st = run_soap_agent(transcript=g["asr"])
                    note, secs = st.get("final_note", ""), round(time.time() - t0, 1)
                row = {"model": name, "architecture": arch, "key": key,
                       **score_note(note, g["tr"], g["soap"], g["spans"]), "seconds": secs}
                with open(os.path.join(OUTPUT_DIR, f"sweep_{name}_{arch}_{key}.txt"), "w", encoding="utf-8") as f:
                    f.write(note)
            except Exception as e:
                print(f"   [{name}/{arch}] {key} FAILED: {str(e)[:200]}")
                row = {"model": name, "architecture": arch, "key": key, "error": str(e)[:300]}
            rows.append(row)
            pd.DataFrame(rows).to_csv(out_csv, index=False)                  # saved after every run
            print(f"   {name:15} {arch:6} {key}: judge={row.get('judge_composite')} rouge1={row.get('rouge1_f1')} "
                  f"seconds={row.get('seconds')}")
        VERIFY_VOTES = votes_main
        LLM.close()
        LLM = ref_llm
    return pd.DataFrame(rows), keys


SWEEP_METRICS = ["judge_composite", "faithfulness", "adequacy", "hallucinations", "kspan_coverage",
                 "rouge1_f1", "rougeL_f1", "bleu", "bertscore_f1", "note_words", "seconds"]

def sweep_table(df, keys):
    ok = df[df["error"].isna()] if "error" in df else df
    ok = ok[ok["key"].isin(keys)]                          # every row on the SAME first-SWEEP_LIMIT dialogues
    extra = []
    cmp_csv = os.path.join(OUTPUT_DIR, f"baseline_comparison_{PIPELINE_VERSION}.csv")
    cmp_csv = cmp_csv if os.path.exists(cmp_csv) else next(iter(sorted(glob.glob(
        f"/kaggle/input/**/baseline_comparison_{PIPELINE_VERSION}.csv", recursive=True))), None)
    if cmp_csv:                                             # same scorer -> same table
        c = pd.read_csv(cmp_csv)
        c = c[c["error"].isna()] if "error" in c else c
        c = c[c["key"].isin(keys)].copy()
        c["model"] = c["system"].map({"agentic": "ft gpt-4o-mini + gpt-4.1-mini (VeriSOAP)",
                                      "straight": "ft gpt-4o-mini (Nishat)"})
        c["architecture"] = c["system"].map({"agentic": "multi", "straight": "single"})
        extra.append(c)
    allrows = pd.concat([ok] + extra, ignore_index=True)
    m = [x for x in SWEEP_METRICS if x in allrows]
    t = allrows.groupby(["model", "architecture"], sort=False)[m].mean().round(3)
    t["n"] = allrows.groupby(["model", "architecture"], sort=False).size()
    key_cols = [x for x in ("judge_composite", "rouge1_f1", "bertscore_f1") if x in t]
    wide = t[key_cols].unstack("architecture") if len(t) else t
    return t, wide


if RUN_SWEEP:
    sweep_df, sweep_keys = run_model_sweep()
    sweep_long, sweep_wide = sweep_table(sweep_df, sweep_keys)
    from IPython.display import display
    print(f"\n=== SINGLE vs MULTI-AGENT per model: means over the SAME first {len(sweep_keys)} dialogues "
          "(rows with fewer, n < limit, are still running); same scorer as the Nishat comparison ===")
    display(sweep_long)
    display(sweep_wide)
    sweep_long.to_csv(os.path.join(OUTPUT_DIR, f"model_sweep_table_{PIPELINE_VERSION}.csv"))
    with open(os.path.join(OUTPUT_DIR, f"model_sweep_table_{PIPELINE_VERSION}.md"), "w", encoding="utf-8") as f:
        f.write(sweep_long.reset_index().to_markdown(index=False))
    print(f"saved {OUTPUT_DIR}/model_sweep_table_{PIPELINE_VERSION}.csv / .md")
else:
    print("[sweep] skipped (set STAGE = 'sweep' to run every model as single-agent and multi-agent)")

---
## Latency benchmark: straight vs agentic

A fair, repeatable timing comparison of the pipelines on the same BeTraC dialogues.

**What is measured.** For every run:

* **End-to-end wall time.** Model loading is excluded, and so is quality scoring: the agentic graph is rebuilt
  without the Evaluation and Report agents, and the straight pipeline's own evaluation is switched off.
* **Time per stage**, e.g. ASR, diarization, K-Span, VerifiNER, generator, reflection, addition for the agentic
  pipeline; ASR, diarization, NER, SOAP, codes for the straight one.
* **Every LLM call**: which stage made it, the model, the latency, and the prompt and completion tokens.
  From these come calls per stage, the share of time spent waiting for the LLM, and the estimated API cost.

**Fairness rules built in:**

1. **The same dialogues, interleaved.** Each dialogue runs on every system back to back, and the starting system
   rotates. If the API gets slower during the benchmark, every system is affected equally.
2. **Warm-up.** One unrecorded run per system first, so one-off costs (CUDA kernels, caches) are not counted.
3. **Two modes.**
   * `"audio"` is the full system.
   * `"transcript"` starts every system from BeTraC's gold transcript, which removes ASR settings and ASR
     errors from the comparison and times only the pipeline design.
4. **Repeats** (`LATENCY_REPEATS`): API latency is noisy, so the report gives median and p90, not only the mean.

**Configurations.** `LATENCY_SYSTEMS` can include agentic variants with different settings, e.g. a fast
variant with `VERIFY_VOTES = 1` and `MAX_REFLECTION_ROUNDS = 1`. This shows how much time each setting costs.

**Output:**

* `latency_runs_<tag>.csv` (one row per run), `latency_calls_<tag>.csv` (one row per LLM call);
* summary tables: overall, per stage, LLM vs non-LLM time, scaling with dialogue length;
* two charts: mean time per stage, and latency vs dialogue length.

The straight pipeline is loaded exactly as in the comparison cell above, and the same loaded copy is reused.
Call-level numbers need the OpenAI backend; with the local LLaMA fallback only stage times are recorded.

In [ ]:
# ============================ 23. LATENCY BENCHMARK (straight vs agentic) ============================
_needed = ["PIPELINE_VERSION", "LLM", "SOAPState", "soap_agent_app", "load_baseline_pipeline"] + (["betrac_dirs"] if RUN_LATENCY else [])
_missing = [n for n in _needed if n not in globals()]
if _missing:
    raise RuntimeError(f"Run the earlier cells first (missing: {', '.join(_missing)}). After a kernel restart: run "
                       "from the top through the BeTraC cell (it skips finished dialogues), then the comparison cell with "
                       "RUN_COMPARISON = False (defines its functions without re-scoring). Saved results are kept.")
LATENCY_MODE    = "transcript"   # "transcript" = pipeline design only (gold transcript in), "audio" = full system
LATENCY_N       = 10             # dialogues: the same random sample as the BeTraC run (BETRAC_SEED)
LATENCY_REPEATS = 1              # API latency is noisy; 2-3 repeats give steadier medians
LATENCY_WARMUP  = True           # one unrecorded run per system first
LATENCY_SYSTEMS = [
    {"name": "straight", "kind": "straight"},
    {"name": "agentic", "kind": "agentic", "overrides": {}},
    {"name": "agentic-fast", "kind": "agentic", "overrides": {"VERIFY_VOTES": 1, "MAX_REFLECTION_ROUNDS": 1}},
]
LATENCY_TAG = f"{LATENCY_MODE}_{PIPELINE_VERSION}"
# USD per 1M tokens (input, output), matched by model-name prefix. CHECK current prices at
# https://openai.com/api/pricing before reporting cost; unknown models are left without a cost.
PRICE_PER_1M = {"ft:gpt-4o-mini": (0.30, 1.20), "gpt-4o-mini": (0.15, 0.60),
                "gpt-4.1-mini": (0.40, 1.60), "gpt-4.1": (2.00, 8.00)}

import threading, contextlib, random, functools, inspect, io

# ---------- 1. recorder: stage times + every LLM call (thread-safe; the straight pipeline uses threads) ----------
_TL = threading.local()
_ACTIVE = {"rec": None}

class _Recorder:
    def __init__(self):
        self.lock, self.stages, self.calls = threading.Lock(), Counter(), []
    def add_stage(self, stage, dt):
        with self.lock:
            self.stages[stage] += dt
    def add_call(self, rec):
        with self.lock:
            self.calls.append(rec)

def _stage_wrap(fn, stage):
    """Time `fn` as `stage`; LLM calls made inside it are attributed to that stage."""
    if getattr(fn, "_latency_stage", None):
        return fn
    def wrapped(*a, **k):
        prev, _TL.stage = getattr(_TL, "stage", None), stage
        t0 = time.perf_counter()
        try:
            return fn(*a, **k)
        finally:
            if _ACTIVE["rec"] is not None:
                _ACTIVE["rec"].add_stage(stage, time.perf_counter() - t0)
            _TL.stage = prev
    if inspect.isfunction(fn):
        wrapped = functools.wraps(fn)(wrapped)       # keep the signature (LangGraph inspects node signatures)
    wrapped._latency_stage = stage
    return wrapped

def _instrument_client(client):
    """Patch an OpenAI client so every chat completion is timed and its token usage recorded."""
    if client is None:
        return False
    comp = client.chat.completions
    if getattr(comp, "_latency_patched", False):
        return True
    orig = comp.create
    def create(*a, **k):
        t0, resp, ok = time.perf_counter(), None, False
        try:
            resp = orig(*a, **k)
            ok = True
            return resp
        finally:
            if _ACTIVE["rec"] is not None:
                u = getattr(resp, "usage", None)
                _ACTIVE["rec"].add_call({"stage": getattr(_TL, "stage", None) or "other", "model": k.get("model"),
                                         "latency_s": round(time.perf_counter() - t0, 3), "ok": ok,
                                         "prompt_tokens": getattr(u, "prompt_tokens", None),
                                         "completion_tokens": getattr(u, "completion_tokens", None)})
    comp.create = create
    comp._latency_patched = True
    return True

def _price(model, pt, ct):
    for prefix in sorted(PRICE_PER_1M, key=len, reverse=True):
        if model and str(model).startswith(prefix) and pt is not None and ct is not None:
            pin, pout = PRICE_PER_1M[prefix]
            return (pt * pin + ct * pout) / 1e6
    return None

@contextlib.contextmanager
def _overrides(values):
    old = {k: globals()[k] for k in (values or {})}
    globals().update(values or {})
    try:
        yield
    finally:
        globals().update(old)

# ---------- 2. the two systems, instrumented ----------
_AGENTIC_STAGES = [("asr_agent", "asr", asr_agent), ("diarization_agent", "diarization", diarization_agent),
                   ("kspan_agent", "kspan", kspan_agent), ("verifiner_agent", "verifiner", verifiner_agent),
                   ("soap_generator_agent", "generator", soap_generator_agent),
                   ("reflection_agent", "reflection", reflection_agent), ("addition_agent", "addition", addition_agent)]

def _build_latency_app():
    """The production graph minus the Evaluation and Report agents (scoring is not part of latency)."""
    g2 = StateGraph(SOAPState)
    for name, stage, fn in _AGENTIC_STAGES:
        g2.add_node(name, _stage_wrap(fn, stage))
    g2.add_conditional_edges(START, route_start, ["asr_agent", "diarization_agent"])
    g2.add_edge("asr_agent", "diarization_agent")
    g2.add_edge("diarization_agent", "kspan_agent")
    g2.add_edge("kspan_agent", "verifiner_agent")
    g2.add_conditional_edges("verifiner_agent", route_after_verifiner, ["kspan_agent", "soap_generator_agent"])
    g2.add_edge("soap_generator_agent", "reflection_agent")
    g2.add_conditional_edges("reflection_agent", route_after_reflection, ["soap_generator_agent", "addition_agent"])
    g2.add_edge("addition_agent", END)
    return g2.compile()

_STRAIGHT_STAGES = {"pipe": "asr", "diarize_asr_text": "diarization", "run_ner_spell": "ner",
                    "run_ksoap_fewshot": "soap", "run_soap_v1_baseline": "soap", "enrich_codes": "codes",
                    "_run_clinical_inference": "codes", "get_cpt_deterministic": "codes"}

def _straight_pipeline():
    """Load the straight notebook once (shared with the comparison cell) and instrument it."""
    fg = globals().get("_BASELINE_FG") or load_baseline_pipeline()
    globals()["_BASELINE_FG"] = fg
    for fname, stage in _STRAIGHT_STAGES.items():
        if fname in fg:
            fg[fname] = _stage_wrap(fg[fname], stage)
    _instrument_client(fg.get("openai_client"))
    return fg

def _run_agentic(app, mode, gold_tr, apath):
    init = {"audio_path": apath if mode == "audio" else None,
            "raw_transcript": gold_tr if mode == "transcript" else "",
            "kspan_attempts": 0, "reflection_round": 0, "reflection_log": [], "timings": {}, "trace": []}
    app.invoke(init, config={"recursion_limit": 60})

def _run_straight(fg, mode, gold_tr, apath):
    if mode == "audio":
        asr = fg["pipe"](apath)
        raw = (asr.get("text", "") if isinstance(asr, dict) else str(asr)).strip()
        try:
            transcript = fg["diarize_asr_text"](raw)
        except Exception:
            transcript = raw                                     # same fallback as the comparison cell
    else:
        transcript = gold_tr                                     # gold transcript already has speaker labels
    with contextlib.redirect_stdout(io.StringIO()):              # its pipeline prints a lot
        fg["run_full_pipeline"](transcript, k_shot=3, strategy="query_aware")

# ---------- 3. one measured run ----------
def _measure(system, key, rep, mode, app, fg, dirs):
    gold_tr = open(os.path.join(dirs["transcript"], key + ".txt"), encoding="utf-8").read()
    wav = os.path.join(dirs["audio"], key + ".wav")
    apath = wav if os.path.exists(wav) else os.path.join(dirs["audio"], key + ".opus")
    rec, err = _Recorder(), None
    _ACTIVE["rec"] = rec
    t0 = time.perf_counter()
    try:
        with _overrides(dict(system.get("overrides") or {}, VERBOSE=False)):
            if system["kind"] == "straight":
                _run_straight(fg, mode, gold_tr, apath)
            else:
                _run_agentic(app, mode, gold_tr, apath)
    except Exception as e:
        err = str(e)[:200]
    finally:
        wall = time.perf_counter() - t0
        _ACTIVE["rec"] = None
    calls = [dict(c, system=system["name"], key=key, rep=rep) for c in rec.calls]
    for c in calls:
        c["cost_usd"] = _price(c["model"], c["prompt_tokens"], c["completion_tokens"])
    pt = sum(c["prompt_tokens"] or 0 for c in calls)
    ct = sum(c["completion_tokens"] or 0 for c in calls)
    costs = [c["cost_usd"] for c in calls if c["cost_usd"] is not None]
    asr_s = rec.stages.get("asr", 0.0)
    row = {"system": system["name"], "key": key, "rep": rep, "mode": mode, "words": len(gold_tr.split()),
           "wall_s": round(wall, 2), "asr_s": round(asr_s, 2), "pipeline_s": round(wall - asr_s, 2),
           "llm_calls": len(calls), "llm_time_s": round(sum(c["latency_s"] for c in calls), 2),
           "prompt_tokens": pt, "completion_tokens": ct,
           "cost_usd": round(sum(costs), 5) if costs else None, "error": err}
    row.update({f"stage_{s}": round(v, 2) for s, v in rec.stages.items()})
    return row, calls

# ---------- 4. benchmark loop: interleaved, rotated, resumable ----------
def run_latency_benchmark(dirs, n=LATENCY_N, repeats=LATENCY_REPEATS, mode=LATENCY_MODE,
                          systems=LATENCY_SYSTEMS, warmup=LATENCY_WARMUP, tag=LATENCY_TAG):
    keys = sorted(os.path.splitext(f)[0] for f in os.listdir(dirs["soap"]))
    if BETRAC_SEED is not None:
        random.Random(BETRAC_SEED).shuffle(keys)                 # same sample as the BeTraC / comparison cells
    sample, spare = keys[:n], (keys[n] if len(keys) > n else keys[0])
    runs_csv = os.path.join(OUTPUT_DIR, f"latency_runs_{tag}.csv")
    calls_csv = os.path.join(OUTPUT_DIR, f"latency_calls_{tag}.csv")
    runs = pd.read_csv(runs_csv).to_dict("records") if os.path.exists(runs_csv) else []
    calls = pd.read_csv(calls_csv).to_dict("records") if os.path.exists(calls_csv) else []
    done = {(r["system"], r["key"], int(r["rep"])) for r in runs if pd.isna(r.get("error"))}
    runs = [r for r in runs if pd.isna(r.get("error"))]            # failed runs are retried

    app = _build_latency_app()
    _instrument_client(getattr(LLM, "client", None))
    if getattr(LLM, "backend", "openai") != "openai":          # local base model + LoRA adapters: time the router itself
        def _hook(c):
            if _ACTIVE["rec"] is not None:
                _ACTIVE["rec"].add_call({"stage": getattr(_TL, "stage", None) or "other", "model": c["model"],
                                         "latency_s": c["latency_s"], "ok": True,
                                         "prompt_tokens": c["prompt_tokens"], "completion_tokens": c["completion_tokens"]})
        LLM.call_hook = _hook
    fg = _straight_pipeline() if any(s["kind"] == "straight" for s in systems) else None
    if getattr(LLM, "backend", "openai") == "openai" and not getattr(LLM, "client", None):
        print("[latency] no OpenAI client: stage times only, no call/token/cost numbers")

    if warmup:
        print(f"[latency] warm-up on {spare} (not recorded) ...")
        for s in systems:
            _measure(s, spare, -1, mode, app, fg, dirs)

    total = len(sample) * repeats * len(systems)
    k = 0
    for rep in range(repeats):
        for i, key in enumerate(sample):
            rot = (i + rep) % len(systems)                        # rotate which system goes first
            for s in systems[rot:] + systems[:rot]:
                k += 1
                if (s["name"], key, rep) in done:
                    continue
                row, cl = _measure(s, key, rep, mode, app, fg, dirs)
                runs.append(row)
                calls.extend(cl)
                pd.DataFrame(runs).to_csv(runs_csv, index=False)      # saved after every run
                pd.DataFrame(calls).to_csv(calls_csv, index=False)
                print(f"[{k}/{total}] {s['name']:13} {key}  wall {row['wall_s']:7.1f}s  "
                      f"llm calls {row['llm_calls']:3d}" + (f"  ERROR {row['error']}" if row["error"] else ""))
    return pd.DataFrame(runs), pd.DataFrame(calls)

# ---------- 5. report ----------
_STAGE_ORDER = ["asr", "diarization", "kspan", "verifiner", "ner", "generator", "soap", "reflection",
                "addition", "codes", "other"]

def latency_report(runs, calls, base="straight"):
    ok = (runs[runs["error"].isna()] if "error" in runs else runs).copy()
    num = ["wall_s", "pipeline_s", "asr_s", "llm_calls", "llm_time_s", "prompt_tokens", "completion_tokens", "cost_usd", "words"]
    for c in num + [c for c in ok.columns if c.startswith("stage_")]:
        if c in ok:
            ok[c] = pd.to_numeric(ok[c], errors="coerce")
    if ok.empty:
        print("No successful runs.")
        return {}
    p90 = lambda s: s.quantile(0.9)
    overall = ok.groupby("system").agg(
        runs=("wall_s", "size"), wall_mean=("wall_s", "mean"), wall_median=("wall_s", "median"),
        wall_p90=("wall_s", p90), pipeline_mean=("pipeline_s", "mean"), asr_mean=("asr_s", "mean"),
        llm_calls=("llm_calls", "mean"), llm_time=("llm_time_s", "mean"),
        prompt_tok=("prompt_tokens", "mean"), completion_tok=("completion_tokens", "mean"),
        cost_usd=("cost_usd", "mean")).round(3)
    if base in overall.index:
        overall["x_slower_than_" + base] = (overall["wall_median"] / overall.loc[base, "wall_median"]).round(2)
    print("=== OVERALL (seconds per dialogue; mode=%s) ===" % ok["mode"].iloc[0])
    print(overall.to_string())

    stage_cols = [c for c in ok.columns if c.startswith("stage_")]
    stages = ok.groupby("system")[stage_cols].mean().fillna(0).T
    stages.index = [c.replace("stage_", "") for c in stages.index]
    stages = stages.loc[[s for s in _STAGE_ORDER if s in stages.index] +
                        [s for s in stages.index if s not in _STAGE_ORDER]]
    share = (stages / ok.groupby("system")["wall_s"].mean() * 100).round(0)
    print("\n=== MEAN SECONDS PER STAGE (share of wall time %) ===")
    print(pd.concat({"seconds": stages.round(1), "% of wall": share}, axis=1).to_string())
    print("(the straight pipeline runs SOAP and coding in parallel threads, so its stage times can add up to more than its wall time)")

    out = {"overall": overall, "stages": stages}
    if len(calls):
        n_runs = ok.groupby("system").size()
        cps = (calls[calls["key"].isin(ok["key"])].groupby(["system", "stage"]).size()
               .unstack(0).fillna(0) / n_runs).round(1)
        print("\n=== LLM CALLS PER DIALOGUE, BY STAGE ===")
        print(cps.to_string())
        llm_share = (ok.groupby("system")["llm_time_s"].mean() / ok.groupby("system")["wall_s"].mean() * 100).round(0)
        print("\n=== TIME WAITING FOR THE LLM (% of wall) ===")
        print(llm_share.to_string())
        out["calls_per_stage"] = cps

    print("\n=== SCALING WITH DIALOGUE LENGTH (linear fit: wall = a + b * words) ===")
    for sys_name, d in ok.groupby("system"):
        if len(d) >= 3 and d["words"].nunique() > 1:
            b, a = np.polyfit(d["words"], d["wall_s"], 1)
            r = np.corrcoef(d["words"], d["wall_s"])[0, 1]
            print(f"  {sys_name:13} +{b * 100:5.1f} s per 100 words   (base {a:6.1f} s, r = {r:.2f})")
    _latency_charts(ok, stages)
    return out

def _latency_charts(ok, stages):
    import matplotlib.pyplot as plt
    surface, ink, ink2, grid = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
    series = ["#2a78d6", "#eb6834", "#1baf7a"]                  # validated categorical slots 1-3 (all-pairs safe)
    systems = [s["name"] for s in LATENCY_SYSTEMS if s["name"] in stages.columns][:3]
    colors = dict(zip(systems, series))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.6), facecolor=surface)
    # (a) mean seconds per stage, one bar per system
    ax = axes[0]
    cut = 0.01 * float(stages[systems].values.max() or 1)
    names = [n for n in stages.index if (stages.loc[n, systems] > cut).any()][::-1]     # skip near-zero stages
    h = 0.8 / max(1, len(systems))
    for i, n in enumerate(names):
        present = [sy for sy in systems if stages.loc[n, sy] > cut]          # centre only the bars in this row
        for j, sy in enumerate(present):
            yy = i - (j - (len(present) - 1) / 2) * h
            v = stages.loc[n, sy]
            ax.barh(yy, v, height=h * 0.9, color=colors[sy], edgecolor=surface, linewidth=2)
            if v >= 5:
                ax.text(v + 1, yy, f"{v:.0f}", va="center", fontsize=8, color=ink2)
    ax.set_yticks(np.arange(len(names)), names)
    ax.set_xlabel("mean seconds per dialogue", color=ink2)
    ax.set_title("Where the time goes (mean per stage)", loc="left", color=ink, fontsize=12)
    ax.grid(axis="x", color=grid, linewidth=0.8)
    # (b) latency vs dialogue length
    ax = axes[1]
    for s in systems:
        d = ok[ok["system"] == s]
        ax.scatter(d["words"], d["wall_s"], s=64, color=colors[s], edgecolor=surface, linewidth=2, zorder=3)
        if len(d) >= 3 and d["words"].nunique() > 1:
            b, a = np.polyfit(d["words"], d["wall_s"], 1)
            xs = np.linspace(d["words"].min(), d["words"].max(), 20)
            ax.plot(xs, a + b * xs, color=colors[s], linewidth=2, zorder=2)
    ax.set_xlabel("dialogue length (words, gold transcript)", color=ink2)
    ax.set_ylabel("wall time per dialogue (s)", color=ink2)
    ax.set_title("Latency vs dialogue length", loc="left", color=ink, fontsize=12)
    ax.grid(color=grid, linewidth=0.8)
    for ax in axes:
        ax.set_facecolor(surface)
        ax.set_axisbelow(True)                               # grid behind the marks
        ax.tick_params(colors=ink2)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_color(grid)
    from matplotlib.patches import Patch                     # one legend for both charts (same systems, same colours)
    fig.legend([Patch(color=colors[sy]) for sy in systems], systems, loc="upper center", ncol=len(systems),
               frameon=False, labelcolor=ink2, bbox_to_anchor=(0.5, 1.0))
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    path = os.path.join(OUTPUT_DIR, f"latency_{LATENCY_TAG}.png")
    fig.savefig(path, dpi=150, facecolor=surface)
    plt.show()
    print(f"chart saved: {path}")

if RUN_LATENCY:
    latency_runs, latency_calls = run_latency_benchmark(betrac_dirs)
    latency_summary = latency_report(latency_runs, latency_calls)
else:
    print("[latency] skipped (RUN_LATENCY = False)")

In [ ]:
# ============================ 21. GRADIO UI ============================
import gradio as gr

def _gradio_run(audio, text, reference):
    audio_path = audio if isinstance(audio, str) and audio else None
    if not audio_path and not (text or "").strip():
        raise gr.Error("Upload an audio file or paste a transcript.")
    r = run_soap_agent(audio_path=audio_path, transcript=None if audio_path else text,
                       reference_soap=(reference or "").strip() or None)
    trace = "| # | agent | action |\n|---|---|---|\n" + "\n".join(
        f"| {i} | {e['agent']} | {e['action']} |" for i, e in enumerate(r.get("trace") or [], 1))
    return r.get("final_note", ""), trace, r.get("report", "")

if RUN_GRADIO:
    with gr.Blocks(title="Agentic SOAP (Hindi-English)") as ui:
        gr.Markdown("# Agentic SOAP note generation - Hindi-English doctor-patient conversations\n"
                    "Transcript -> K-Spans -> VerifiNER -> Fine-tuned generator -> Reflection loop -> Codes -> Evaluation")
        with gr.Row():
            audio_in = gr.Audio(label="Consultation audio", type="filepath")
            text_in = gr.Textbox(label="...or paste a transcript", lines=8)
        ref_in = gr.Textbox(label="Reference SOAP (optional, for ROUGE / BLEU / BERTScore)", lines=3)
        run_btn = gr.Button("Run agents", variant="primary")
        note_out = gr.Textbox(label="SOAP note", lines=16)
        trace_out = gr.Markdown(label="Agent trace")
        report_out = gr.Textbox(label="Full report", lines=30)
        run_btn.click(_gradio_run, [audio_in, text_in, ref_in], [note_out, trace_out, report_out])

    gr.close_all()
    ui.launch(share=True)
else:
    print("[gradio] skipped (RUN_GRADIO = False)")